# 🚀 ASID-CL v2: Full End-to-End Continual Learning & SOTA Comparison Suite (Google Colab T4 GPU)

File Notebook độc lập (**Self-Contained 1-Click Run All**) này thực thi trọn vẹn **cả 2 chế độ End-to-End từ ảnh JPEG / văn bản thô đến kết quả cuối cùng** trên toàn bộ tập dữ liệu gốc, tối ưu hóa riêng cho **GPU NVIDIA Tesla T4 (16 GB VRAM)** kèm cơ chế **Tự động lưu & Khôi phục từng bước (Auto-Checkpoint Resume) trên Google Drive**:

1. **Bước 0 — Kết nối Google Drive & Khởi tạo Mã nguồn Độc lập:**
   - Gắn kết Google Drive tại `/content/drive/MyDrive/ASID_CL_v2_Full_Colab` để lưu trữ an toàn mọi đặc trưng `.pt`, kết quả `.json` sau mỗi Seed/Task và hình vẽ xuất bản (`.png`, `.pdf`, `.svg`). Nếu Colab bị ngắt kết nối, chỉ cần bấm **Run All** lại là hệ thống tự động chạy tiếp từ điểm dừng gần nhất.
   - Giải nén trực tiếp toàn bộ thư viện `mm_flex_merge` (`BlockAnalyticCL`, `SessionLoRA`, `AttnState`) cùng mã nguồn chính thức của 7 tác giả SOTA (`GACL`, `ACIL`, `InfLoRA`, `SD-LoRA`, `TSV-M`, `KnOTS`, `RebQ`, `DeLo`).

2. **Chế độ End-to-End 1 — Trích xuất Trực tiếp từ Ảnh/Chữ Thô (`CLIP ViT-B/16` FP16) trên TOÀN BỘ Dataset Gốc (`60,000+` mẫu) & Đối đầu 7 Phương pháp SOTA (2022–2026):**
   - **Split CIFAR-100 Full (`60,000` ảnh gốc: `50,000` train + `10,000` test, 10 Tasks × 10 Classes):** Tải trực tiếp qua `torchvision`, chạy 1-Pass `CLIP ViT-B/16` (`FP16`, `BS=256`, mất `~4.5 phút` trên T4) và lưu cache lên Google Drive.
   - **UPMC-Food101-CMML Full (`~90,000` cặp ảnh–văn bản thực tế, 10 Tasks × 10 Classes) & MM-IMDb-CMML (`20` Genres, 5 Tasks):** Tải siêu tốc 16 luồng bằng `aria2c`, trích xuất trực tiếp qua `CLIP ViT-B/16` (`FP16`) và lưu cache lên Google Drive.
   - **Chạy kiểm định 3 Tiêu chí (`Chuẩn Mã Nguồn - Chuẩn Phương Pháp - Chuẩn Cấu Hình`) & Cùng Hệ Quy Chiếu (`2P=2048` & `2P=5000`, `Task-Free CIL` & `Oracle TIL`):** Chạy 5 seeds (`2027..2031`), lưu checkpoint ngay sau mỗi seed.

3. **Chế độ End-to-End 2 — Full 2-Stage (`Stage 1: 12-Layer Null-Space LoRA` Fine-Tuning trên Ảnh/Chữ Thô + `Stage 2: Block-Gram Pushforward Transport & ASID-CL v2`):**
   - Tải trực tiếp bộ dữ liệu thảm họa đa phương thức thực tế **CrisisMMD v2.0** (`7` trận thảm họa) và **UPMC-Food101** bằng `aria2c`.
   - Huấn luyện xuyên qua 12 tầng Vision + Text Transformer với ma trận chiếu trực giao không gian rỗng `P_perp` (`Null-Space LoRA`), vận chuyển ma trận Gram không cần lưu mẫu (`transport_gram()`) và giải nghiệm đóng Khối Schur `solve_schur()`.

4. **Xuất Bản Hình Vẽ & Báo Cáo Tự Động:**
   - Tự động vẽ biểu đồ 4-Panel chuẩn bài báo (`300 DPI` PNG, PDF, SVG) và đồng bộ toàn bộ sang Google Drive.


## 1. Kiểm tra GPU Tesla T4, Kết nối Google Drive & Cài đặt Thư viện


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, sys, time, json, pathlib
import torch

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# 1. Mount Google Drive for persistent checkpointing (Resume-Safe on Colab T4)
USE_GDRIVE = True
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    SAVE_ROOT = pathlib.Path('/content/drive/MyDrive/ASID_CL_v2_Full_Colab')
    print(f"✅ Đã kết nối Google Drive! Toàn bộ cache & checkpoint sẽ được lưu tại: {SAVE_ROOT}")
except Exception as e:
    USE_GDRIVE = False
    SAVE_ROOT = pathlib.Path('/content/ASID_CL_v2_Full_Colab')
    print(f"⚠️ Không gắn được Google Drive ({e}). Lưu cục bộ tại: {SAVE_ROOT}")

CACHE_DIR = SAVE_ROOT / "feature_cache"
CKPT_DIR = SAVE_ROOT / "checkpoints"
FIG_DIR = SAVE_ROOT / "figures"
for d in (SAVE_ROOT, CACHE_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 2. Install aria2c (for 16-connection ultra-fast dataset download) and required packages
!apt-get -qq install -y aria2 > /dev/null
!pip install -q transformers torchvision scikit-learn pandas matplotlib tqdm
print("✅ Đã cài đặt xong thư viện và aria2c!")


## 2. Giải nén Mã nguồn Dự án (`mm_flex_merge`) & Mã nguồn Chính thức của 7 Tác giả SOTA


In [ ]:
import pathlib, sys, json
WORKSPACE = pathlib.Path('/content/asid_cl_workspace')
FILES = {"mm_flex_merge/__init__.py": "\"\"\"\nMM-FLEX-MERGE: Multimodal Dynamic Orthogonal Elastic Merging Framework\n======================================================================\nM\u1ed9t b\u1ed9 khung nghi\u00ean c\u1ee9u khoa h\u1ecdc chuy\u00ean s\u00e2u v\u00e0 baseline \u0111\u1ed9c quy\u1ec1n ph\u1ee5c v\u1ee5\nContinual Model Merging, \u0110a ph\u01b0\u01a1ng th\u1ee9c (Multimodal VLM - CLIP), L\u1ecdc kh\u00f4ng gian con\ntr\u1ef1c giao (Orthogonal Null-Space) v\u00e0 c\u01a1 ch\u1ebf Zero-Latency Compliance (GDPR Data Opt-Out).\n\"\"\"\n\n# --- Models ---\nfrom .models.analytic import BlockAnalyticCL\nfrom .models.heads import DualStreamCrossGatedExpert, LinearExpert, UnimodalExpert\nfrom .models.nullspace_lora import TEXT_MASK, AttnState, SessionLoRA\nfrom .models.routers import DualCentroidRouter\n\n# --- Merging Operators ---\nfrom .merging.flex_merge import MMFlexMerger\nfrom .merging.opcm import OPCMMerger\nfrom .merging.nufilt import NUFILTMerger\nfrom .merging.arithmetic import TaskArithmeticMerger, SWAMerger\n\n# --- Evaluation Protocols ---\nfrom .protocols.metrics import ContinualMergingProtocol\nfrom .protocols.missing_modality import MissingModalityProtocol\nfrom .protocols.opt_out import DataOptOutProtocol\nfrom .protocols.statistical import StatisticalSignificanceProtocol\n\n# --- Engine ---\nfrom .engine.trainer import TaskExpertTrainer\nfrom .engine.evaluator import StreamEvaluator\n\n# --- Datasets ---\nfrom .datasets.base import BaseEmbeddingDataset\nfrom .datasets.uk_crisismmd import UKCrisisMMDDataset\nfrom .datasets.crisismmd_v2 import CrisisMMDv2Dataset\n\n# --- Utilities ---\nfrom .utils.seed import set_deterministic_seed\nfrom .utils.io import load_npz_cache, save_npz_cache\n\n__version__ = \"1.1.0\"\n__all__ = [\n    # Models\n    \"AttnState\",\n    \"BlockAnalyticCL\",\n    \"DualStreamCrossGatedExpert\",\n    \"LinearExpert\",\n    \"SessionLoRA\",\n    \"TEXT_MASK\",\n    \"UnimodalExpert\",\n    \"DualCentroidRouter\",\n    # Merging Operators\n    \"MMFlexMerger\",\n    \"OPCMMerger\",\n    \"NUFILTMerger\",\n    \"TaskArithmeticMerger\",\n    \"SWAMerger\",\n    # Protocols\n    \"ContinualMergingProtocol\",\n    \"MissingModalityProtocol\",\n    \"DataOptOutProtocol\",\n    \"StatisticalSignificanceProtocol\",\n    # Engine\n    \"TaskExpertTrainer\",\n    \"StreamEvaluator\",\n    # Datasets\n    \"BaseEmbeddingDataset\",\n    \"UKCrisisMMDDataset\",\n    \"CrisisMMDv2Dataset\",\n    # Utilities\n    \"set_deterministic_seed\",\n    \"load_npz_cache\",\n    \"save_npz_cache\",\n]\n", "mm_flex_merge/models/__init__.py": "from .analytic import BlockAnalyticCL\nfrom .heads import DualStreamCrossGatedExpert, LinearExpert, UnimodalExpert\nfrom .nullspace_lora import TEXT_MASK, AttnState, SessionLoRA\nfrom .routers import DualCentroidRouter\n\n__all__ = [\n    \"AttnState\",\n    \"BlockAnalyticCL\",\n    \"DualStreamCrossGatedExpert\",\n    \"LinearExpert\",\n    \"SessionLoRA\",\n    \"TEXT_MASK\",\n    \"UnimodalExpert\",\n    \"DualCentroidRouter\",\n]\n", "mm_flex_merge/models/analytic.py": "\"\"\"Block-structured analytic continual learner on frozen or Null-Space LoRA multimodal embeddings.\n\nEach modality m has its own frozen random projection, phi_m(x) = relu(normalize(x) @ W_m).\nFor M=2 core modalities (Image 'img' and Text 'txt'), the model maintains sufficient statistics\nover the block-concatenated features phi = [phi_img, phi_txt]:\n\n    G = [[G_II, G_IT], [G_TI, G_TT]],   S = [[S_I], [S_T]]   (+ per-class Grams/counts when beta > 0)\n\nConsequences (all exact in FP64, zero retraining):\n  * update / delete        : add or subtract statistics -> deletion == retraining without the data.\n  * order invariance       : statistics are commutative sums, so task order does not matter.\n  * missing modality       : solve ridge on the Gram sub-block of the available modalities,\n                             which equals a dedicated model trained on those modalities only.\n  * asymmetric ridge       : modality-specific Tikhonov regularization (gamma_I, gamma_T) scaled\n                             by each block's own trace Tr(G_mm) / P.\n  * Block Schur Complement : solves the 2P x 2P system via P x P inversions while cancelling\n                             cross-modal collinearity (-rho * G_IT * W_T) and reusing W_T.\n  * CMML / MIL availability: accumulates partial-modality observations and normalizes block energy\n                             by per-modality observation counts N_{c, m}.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport zlib\nfrom typing import Mapping, Sequence\n\nimport torch\nimport torch.nn.functional as F\n\n\nclass BlockAnalyticCL:\n    def __init__(\n        self,\n        modality_dims: Mapping[str, int],\n        num_classes: int,\n        *,\n        projection_dim: int = 1024,\n        ridge: float | Mapping[str, float] = 0.1,\n        class_balance_beta: float = 0.0,\n        schur_coupling_rho: float = 1.0,\n        seed: int = 2026,\n    ) -> None:\n        if not modality_dims or num_classes <= 0 or projection_dim <= 0:\n            raise ValueError(\"invalid BlockAnalyticCL configuration\")\n        if isinstance(ridge, Mapping):\n            if any(float(v) <= 0 for v in ridge.values()):\n                raise ValueError(\"ridge values must be positive\")\n            self.modality_ridges: dict[str, float] | None = {str(k): float(v) for k, v in ridge.items()}\n            self.ridge = float(next(iter(self.modality_ridges.values())))\n        else:\n            if float(ridge) <= 0:\n                raise ValueError(\"ridge must be positive\")\n            self.modality_ridges = None\n            self.ridge = float(ridge)\n        if class_balance_beta < 0:\n            raise ValueError(\"class_balance_beta must be non-negative\")\n        if not (0.0 <= float(schur_coupling_rho) <= 1.0):\n            raise ValueError(\"schur_coupling_rho must be in [0, 1]\")\n\n        self.modalities = tuple(modality_dims)\n        self.num_classes = int(num_classes)\n        self.projection_dim = int(projection_dim)\n        self.beta = float(class_balance_beta)\n        self.rho = float(schur_coupling_rho)\n\n        # Per-modality seed depends only on (seed, name) so a single-modality model\n        # shares the exact projection of the same block in a multimodal model.\n        self.projections: dict[str, torch.Tensor] = {}\n        for name, dim in modality_dims.items():\n            g = torch.Generator().manual_seed(int(seed) + zlib.crc32(name.encode()))\n            self.projections[name] = torch.randn(int(dim), self.projection_dim, generator=g, dtype=torch.float64)\n\n        D = self.projection_dim * len(self.modalities)\n        self.gram = torch.zeros(D, D, dtype=torch.float64)\n        self.class_sums = torch.zeros(D, self.num_classes, dtype=torch.float64)\n        self._use_exact_class_grams = (self.beta > 0) and (self.num_classes * D * D <= 500_000_000)\n        self.class_grams = torch.zeros(self.num_classes, D, D, dtype=torch.float64) if self._use_exact_class_grams else None\n        self.class_counts = torch.zeros(self.num_classes, dtype=torch.long)\n        self.modality_class_counts = torch.zeros(len(self.modalities), self.num_classes, dtype=torch.long)\n        self._has_partial_observations = False\n        self._cache: dict[tuple[str, ...], torch.Tensor] = {}\n\n    # ------------------------------------------------------------------ features\n    def _check(self, modalities: Sequence[str] | None) -> tuple[str, ...]:\n        if modalities is None:\n            return self.modalities\n        mods = tuple(m for m in self.modalities if m in set(modalities))\n        if not mods or len(mods) != len(set(modalities)):\n            raise ValueError(f\"unknown or empty modalities {modalities}; known {self.modalities}\")\n        return mods\n\n    def _indices(self, mods: tuple[str, ...]) -> torch.Tensor:\n        P = self.projection_dim\n        return torch.cat([torch.arange(self.modalities.index(m) * P, (self.modalities.index(m) + 1) * P) for m in mods])\n\n    def features(self, inputs: Mapping[str, torch.Tensor], modalities: Sequence[str] | None = None) -> torch.Tensor:\n        mods = self._check(modalities)\n        return torch.cat(\n            [F.relu(F.normalize(inputs[m].double(), dim=-1) @ self.projections[m]) for m in mods], dim=1\n        )\n\n    # ---------------------------------------------------------------- statistics\n    def _accumulate(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        labels: torch.Tensor,\n        sign: float,\n        availability: Mapping[str, torch.Tensor] | None = None,\n    ) -> None:\n        labels = labels.long()\n        n = int(labels.shape[0])\n        P = self.projection_dim\n\n        # Support both full-modality inputs and partial-modality (CMML / MIL) inputs\n        present_mods = [m for m in self.modalities if m in inputs and inputs[m] is not None]\n        if not present_mods:\n            raise ValueError(\"at least one known modality must be provided in inputs\")\n\n        phi_blocks = []\n        avail_masks = []\n        for m_idx, m in enumerate(self.modalities):\n            if m in inputs and inputs[m] is not None:\n                block = F.relu(F.normalize(inputs[m].double(), dim=-1) @ self.projections[m])\n                if availability is not None and m in availability and availability[m] is not None:\n                    mask = availability[m].bool().view(-1)\n                    if not bool(mask.all()):\n                        self._has_partial_observations = True\n                        block = torch.where(mask.unsqueeze(1), block, torch.zeros_like(block))\n                else:\n                    mask = torch.ones(n, dtype=torch.bool)\n            else:\n                self._has_partial_observations = True\n                block = torch.zeros(n, P, dtype=torch.float64)\n                mask = torch.zeros(n, dtype=torch.bool)\n            phi_blocks.append(block)\n            avail_masks.append(mask)\n\n        phi = torch.cat(phi_blocks, dim=1)\n        counts = torch.bincount(labels, minlength=self.num_classes)\n        if sign < 0 and bool((counts > self.class_counts).any()):\n            raise ValueError(\"cannot delete more samples of a class than were observed\")\n\n        self.gram.add_(phi.T @ phi, alpha=sign)\n        self.class_sums.add_(phi.T @ F.one_hot(labels, self.num_classes).double(), alpha=sign)\n        self.class_counts.add_(counts if sign > 0 else -counts)\n        for m_idx, mask in enumerate(avail_masks):\n            if bool(mask.any()):\n                m_counts = torch.bincount(labels[mask], minlength=self.num_classes)\n                self.modality_class_counts[m_idx].add_(m_counts if sign > 0 else -m_counts)\n\n        if self.class_grams is not None:\n            for c in labels.unique().tolist():\n                f = phi[labels == c]\n                self.class_grams[c].add_(f.T @ f, alpha=sign)\n        self._cache.clear()\n\n    def update(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        labels: torch.Tensor,\n        availability: Mapping[str, torch.Tensor] | None = None,\n    ) -> None:\n        self._accumulate(inputs, labels, 1.0, availability=availability)\n\n    def delete(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        labels: torch.Tensor,\n        availability: Mapping[str, torch.Tensor] | None = None,\n    ) -> None:\n        \"\"\"Exact unlearning: remove previously seen samples (any granularity: sample/task).\"\"\"\n        self._accumulate(inputs, labels, -1.0, availability=availability)\n\n    def class_weights(self, counts: torch.Tensor | None = None) -> torch.Tensor:\n        cnt = (self.class_counts if counts is None else counts).double()\n        weights = torch.zeros_like(cnt)\n        observed = cnt > 0\n        if observed.any():\n            rel = cnt[observed] / cnt[observed].mean()\n            weights[observed] = rel.pow(-self.beta)\n            weights[observed] /= weights[observed].mean()\n        return weights\n\n    # --------------------------------------------------------------------- solve\n    def _weighted_gram_and_sums(self, mods: tuple[str, ...]) -> tuple[torch.Tensor, torch.Tensor]:\n        P = self.projection_dim\n        is_full = mods == self.modalities\n        if is_full:\n            gram_view = self.gram\n            sums_view = self.class_sums\n        elif len(mods) == 1:\n            m_idx = self.modalities.index(mods[0])\n            s_m = slice(m_idx * P, (m_idx + 1) * P)\n            gram_view = self.gram[s_m, s_m]\n            sums_view = self.class_sums[s_m]\n        else:\n            idx = self._indices(mods)\n            gram_view = self.gram[idx][:, idx]\n            sums_view = self.class_sums[idx]\n\n        if self.class_grams is None:\n            if self.beta <= 0:\n                return gram_view.clone(), sums_view.clone()\n            w = self.class_weights()\n            return gram_view.clone(), sums_view * w.unsqueeze(0)\n\n        w = self.class_weights()\n        if not self._has_partial_observations or len(mods) == 1:\n            if len(mods) == 1 and self._has_partial_observations:\n                m_idx = self.modalities.index(mods[0])\n                w = self.class_weights(self.modality_class_counts[m_idx])\n            obs = self.class_counts > 0\n            if not self._has_partial_observations and bool(obs.any()) and float((w[obs].max() - w[obs].min()).item()) < 1e-12:\n                return gram_view.clone(), sums_view * w.unsqueeze(0)\n            if is_full:\n                D_sub = self.gram.shape[0]\n                G = torch.matmul(w, self.class_grams.view(self.num_classes, -1)).view(D_sub, D_sub)\n            elif len(mods) == 1:\n                m_idx = self.modalities.index(mods[0])\n                s_m = slice(m_idx * P, (m_idx + 1) * P)\n                sub_cg = self.class_grams[:, s_m, s_m].contiguous()\n                G = torch.matmul(w, sub_cg.view(self.num_classes, -1)).view(P, P)\n            else:\n                idx = self._indices(mods)\n                G = torch.einsum(\"c,cij->ij\", w, self.class_grams[:, idx][:, :, idx])\n            S = sums_view * w.unsqueeze(0)\n            return G, S\n\n        # Availability-normalized weighting across modalities when partial observations exist\n        scales = []\n        for m in mods:\n            m_idx = self.modalities.index(m)\n            mc = self.modality_class_counts[m_idx].double()\n            sc = torch.where(mc > 0, self.class_counts.double() / mc.clamp(min=1.0), torch.ones_like(w))\n            scales.append(sc)\n\n        K = len(mods)\n        G = torch.zeros(K * P, K * P, dtype=torch.float64)\n        S = torch.zeros(K * P, self.num_classes, dtype=torch.float64)\n        sub_cgrams = self.class_grams if is_full else self.class_grams[:, self._indices(mods)][:, :, self._indices(mods)]\n        sub_csums = sums_view\n        for a in range(K):\n            sa = slice(a * P, (a + 1) * P)\n            wa = w * scales[a]\n            G[sa, sa] = torch.einsum(\"c,cij->ij\", wa, sub_cgrams[:, sa, sa])\n            S[sa] = sub_csums[sa] * wa.unsqueeze(0)\n            for b in range(a + 1, K):\n                sb = slice(b * P, (b + 1) * P)\n                wab = w * torch.sqrt(scales[a] * scales[b])\n                G[sa, sb] = torch.einsum(\"c,cij->ij\", wab, sub_cgrams[:, sa, sb])\n                G[sb, sa] = G[sa, sb].T\n        return G, S\n\n    def solve_schur(\n        self,\n        gamma_first: float | None = None,\n        gamma_second: float | None = None,\n        rho: float | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Optimal 2x2 Block Schur Complement solver for M=2 modalities (e.g. Image and Text).\n        Returns (W_both, W_first_only, W_second_only) in O(P^3) by reusing the unimodal inverse:\n          A_2 = G_22 + lam_2 I_P\n          W_2_only = A_2^{-1} S_2\n          Schur_1 = (G_11 + lam_1 I_P) - rho^2 * G_12 A_2^{-1} G_21\n          W_1_joint = Schur_1^{-1} (S_1 - rho * G_12 W_2_only)\n          W_2_joint = W_2_only - rho * A_2^{-1} G_21 W_1_joint\n        \"\"\"\n        if len(self.modalities) != 2:\n            raise ValueError(\"solve_schur requires exactly 2 modalities\")\n        if int(self.class_counts.sum()) <= 0:\n            raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n\n        P = self.projection_dim\n        m1, m2 = self.modalities\n        g1 = float(gamma_first if gamma_first is not None else (self.modality_ridges.get(m1, self.ridge) if self.modality_ridges else self.ridge))\n        g2 = float(gamma_second if gamma_second is not None else (self.modality_ridges.get(m2, self.ridge) if self.modality_ridges else self.ridge))\n        rho_val = float(self.rho if rho is None else rho)\n\n        G, S = self._weighted_gram_and_sums(self.modalities)\n        G_11, G_22, G_12 = G[:P, :P], G[P:, P:], G[:P, P:]\n        S_1, S_2 = S[:P], S[P:]\n\n        tr_1_uni = max(float(torch.trace(G_11)) / P, 1e-12)\n        tr_2_uni = max(float(torch.trace(G_22)) / P, 1e-12)\n        use_block_trace = (self.modality_ridges is not None) or (gamma_first is not None) or (gamma_second is not None)\n        if use_block_trace:\n            tr_1, tr_2 = tr_1_uni, tr_2_uni\n        else:\n            tr_joint = max(float(torch.trace(G)) / (2 * P), 1e-12)\n            tr_1 = tr_2 = tr_joint\n\n        eye = torch.eye(P, dtype=G.dtype)\n        A_1 = G_11 + (g1 * tr_1) * eye\n        A_2 = G_22 + (g2 * tr_2) * eye\n\n        sol_2 = torch.linalg.solve(A_2, torch.cat([G_12.T, S_2], dim=1))\n        A_2_inv_G_21 = sol_2[:, :P]\n        W_2_joint_base = sol_2[:, P:]\n\n        if use_block_trace:\n            W_1_only = torch.linalg.solve(A_1, S_1)\n            W_2_only = W_2_joint_base\n        else:\n            W_1_only = torch.linalg.solve(G_11 + (g1 * tr_1_uni) * eye, S_1)\n            W_2_only = torch.linalg.solve(G_22 + (g2 * tr_2_uni) * eye, S_2)\n\n        if rho_val == 0.0:\n            W_1_joint = W_1_only if use_block_trace else torch.linalg.solve(A_1, S_1)\n            W_both = torch.cat([W_1_joint, W_2_joint_base], dim=0)\n        else:\n            Schur_1 = A_1 - (rho_val ** 2) * (G_12 @ A_2_inv_G_21)\n            rhs_1 = S_1 - rho_val * (G_12 @ W_2_joint_base)\n            W_1_joint = torch.linalg.solve(Schur_1, rhs_1)\n            W_2_joint = W_2_joint_base - rho_val * (A_2_inv_G_21 @ W_1_joint)\n            W_both = torch.cat([W_1_joint, W_2_joint], dim=0)\n\n        return W_both, W_1_only, W_2_only\n\n    def auto_spectral_ridges(self, gamma_base: float = 0.22, auto_rho: bool = True) -> dict[str, float]:\n        \"\"\"\n        Pillar 2: Self-tuning Empirical-Bayes Centered Fisher-Spectral & Sample-Load Shrinkage Law.\n        Automatically computes modality-specific regularization (gamma_1, gamma_2) by projecting out\n        the non-discriminative positive-orthant ReLU mean and combining the dimensionless centered\n        Fisher Noise-to-Signal Ratio (FNSR_m = tr(W_m) / tr(B_m)) with the spectral effective rank\n        r_eff(G_mm), bounded sample-load scaling, and per-class density Schur coupling rho*.\n        \"\"\"\n        if len(self.modalities) != 2:\n            raise ValueError(\"auto_spectral_ridges requires exactly 2 modalities\")\n        if int(self.class_counts.sum()) <= 0:\n            raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n\n        P = self.projection_dim\n        m1, m2 = self.modalities\n        G, S = self._weighted_gram_and_sums(self.modalities)\n        G_11, G_22 = G[:P, :P], G[P:, P:]\n        S_1, S_2 = S[:P], S[P:]\n\n        w = self.class_weights()\n        cnt_w = self.class_counts.double() * w\n        N_w = max(float(cnt_w.sum().item()), 1e-12)\n        mu_1 = S_1.sum(dim=1, keepdim=True) / N_w\n        mu_2 = S_2.sum(dim=1, keepdim=True) / N_w\n        G_11_c = G_11 - N_w * (mu_1 @ mu_1.T)\n        G_22_c = G_22 - N_w * (mu_2 @ mu_2.T)\n        S_1_c = S_1 - mu_1 @ cnt_w.unsqueeze(0)\n        S_2_c = S_2 - mu_2 @ cnt_w.unsqueeze(0)\n\n        inv_cnt = torch.where(cnt_w > 0, 1.0 / cnt_w.clamp(min=1e-12), torch.zeros_like(cnt_w))\n        tr_B1 = max(float(torch.sum((S_1_c ** 2) * inv_cnt.unsqueeze(0))), 1e-12)\n        tr_B2 = max(float(torch.sum((S_2_c ** 2) * inv_cnt.unsqueeze(0))), 1e-12)\n        tr_c1 = max(float(torch.trace(G_11_c)), 1e-12)\n        tr_c2 = max(float(torch.trace(G_22_c)), 1e-12)\n        tr_u1 = max(float(torch.trace(G_11)), 1e-12)\n        tr_u2 = max(float(torch.trace(G_22)), 1e-12)\n\n        fnsr_1 = max((tr_c1 - tr_B1) / tr_B1, 0.1)\n        fnsr_2 = max((tr_c2 - tr_B2) / tr_B2, 0.1)\n        fnsr_geom = (fnsr_1 * fnsr_2) ** 0.5\n\n        r_eff1 = (tr_u1 ** 2) / max(float(torch.sum(G_11 * G_11)), 1e-12)\n        r_eff2 = (tr_u2 ** 2) / max(float(torch.sum(G_22 * G_22)), 1e-12)\n\n        idx_1 = (fnsr_1 / max(r_eff1, 1.0)) * ((tr_c1 / tr_u1) ** 0.5)\n        idx_2 = (fnsr_2 / max(r_eff2, 1.0)) * ((tr_c2 / tr_u2) ** 0.5)\n        idx_geom = max((idx_1 * idx_2) ** 0.5, 1e-12)\n\n        n_1 = max(int(self.modality_class_counts[0].sum().item()), 1)\n        n_2 = max(int(self.modality_class_counts[1].sum().item()), 1)\n        noise_scale = (fnsr_geom / 6.0) ** 0.30\n        load_1 = (0.58 + 0.52 * (float(2 * P) / (n_1 + float(2 * P)))) * noise_scale\n        load_2 = (0.58 + 0.52 * (float(2 * P) / (n_2 + float(2 * P)))) * noise_scale\n\n        eb_1 = (idx_1 / idx_geom) ** 0.65\n        eb_2 = (idx_2 / idx_geom) ** 0.65\n\n        g1 = float(min(max(gamma_base * load_1 * eb_1, 0.05), 5.0))\n        g2 = float(min(max(gamma_base * load_2 * eb_2, 0.05), 5.0))\n        self.modality_ridges = {m1: g1, m2: g2}\n        if auto_rho:\n            obs_c = max(int((self.class_counts > 0).sum().item()), 1)\n            n_per_cls = float(self.class_counts.sum().item()) / obs_c\n            self.rho = float(min(1.0, max(n_per_cls / 65.0, 0.05)))\n        self._cache.clear()\n        return dict(self.modality_ridges)\n\n    def schur_innovation_decomposition(\n        self,\n        gamma_first: float | None = None,\n        gamma_second: float | None = None,\n        rho: float | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Pillar 1: Exact Schur-Innovation Decomposition (SID).\n        Decomposes the joint 2x2 Block Schur estimator into its marginal unimodal estimators\n        plus the exact cross-modal Schur innovation matrices:\n          W_1_joint = W_1_only + dW_1_schur\n          W_2_joint = W_2_only + dW_2_schur\n        Returns (W_1_only, W_2_only, dW_1_schur, dW_2_schur, W_both).\n        \"\"\"\n        if len(self.modalities) != 2:\n            raise ValueError(\"schur_innovation_decomposition requires exactly 2 modalities\")\n        if int(self.class_counts.sum()) <= 0:\n            raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n\n        is_default = (gamma_first is None) and (gamma_second is None) and (rho is None)\n        if is_default and (\"__sid__\",) in self._cache:\n            return self._cache[(\"__sid__\",)]  # type: ignore[return-value]\n\n        P = self.projection_dim\n        m1, m2 = self.modalities\n        g1 = float(gamma_first if gamma_first is not None else (self.modality_ridges.get(m1, self.ridge) if self.modality_ridges else self.ridge))\n        g2 = float(gamma_second if gamma_second is not None else (self.modality_ridges.get(m2, self.ridge) if self.modality_ridges else self.ridge))\n        rho_val = float(self.rho if rho is None else rho)\n\n        G, S = self._weighted_gram_and_sums(self.modalities)\n        G_11, G_22, G_12 = G[:P, :P], G[P:, P:], G[:P, P:]\n        S_1, S_2 = S[:P], S[P:]\n\n        use_block_trace = (self.modality_ridges is not None) or (gamma_first is not None) or (gamma_second is not None)\n        if use_block_trace:\n            tr_1 = max(float(torch.trace(G_11)) / P, 1e-12)\n            tr_2 = max(float(torch.trace(G_22)) / P, 1e-12)\n        else:\n            tr_joint = max(float(torch.trace(G)) / (2 * P), 1e-12)\n            tr_1 = tr_2 = tr_joint\n\n        eye = torch.eye(P, dtype=G.dtype)\n        A_1 = G_11 + (g1 * tr_1) * eye\n        A_2 = G_22 + (g2 * tr_2) * eye\n\n        W_1_only = torch.linalg.solve(A_1, S_1)\n        sol_2 = torch.linalg.solve(A_2, torch.cat([G_12.T, S_2], dim=1))\n        A_2_inv_G_21 = sol_2[:, :P]\n        W_2_only = sol_2[:, P:]\n\n        if rho_val == 0.0:\n            dW_1_schur = torch.zeros_like(W_1_only)\n            dW_2_schur = torch.zeros_like(W_2_only)\n        else:\n            Schur_1 = A_1 - (rho_val ** 2) * (G_12 @ A_2_inv_G_21)\n            innov_rhs_1 = (rho_val ** 2) * (G_12 @ (A_2_inv_G_21 @ W_1_only)) - rho_val * (G_12 @ W_2_only)\n            dW_1_schur = torch.linalg.solve(Schur_1, innov_rhs_1)\n            dW_2_schur = -rho_val * (A_2_inv_G_21 @ (W_1_only + dW_1_schur))\n\n        W_both = torch.cat([W_1_only + dW_1_schur, W_2_only + dW_2_schur], dim=0)\n        res = (W_1_only, W_2_only, dW_1_schur, dW_2_schur, W_both)\n        if is_default:\n            self._cache[(\"__sid__\",)] = res  # type: ignore[assignment]\n            self._cache[self.modalities] = W_both\n            self._cache[(m1,)] = W_1_only\n            self._cache[(m2,)] = W_2_only\n            w = self.class_weights()\n            cnt_w = (self.class_counts.double() * w).clamp(min=1e-12)\n            N_w = max(float(cnt_w.sum().item()), 1e-12)\n            mu_1_vec = S_1.sum(dim=1) / N_w\n            mu_2_vec = S_2.sum(dim=1) / N_w\n            mu_1_hat = F.normalize(mu_1_vec, dim=0)\n            mu_2_hat = F.normalize(mu_2_vec, dim=0)\n            tr_u1 = max(float(torch.trace(G_11)), 1e-12)\n            tr_u2 = max(float(torch.trace(G_22)), 1e-12)\n            c0_1 = float(((N_w * float(torch.sum(mu_1_vec ** 2))) / tr_u1) ** 0.5)\n            c0_2 = float(((N_w * float(torch.sum(mu_2_vec ** 2))) / tr_u2) ** 0.5)\n            S_1_c = F.normalize(S_1 - S_1.mean(dim=1, keepdim=True), dim=0)\n            S_2_c = F.normalize(S_2 - S_2.mean(dim=1, keepdim=True), dim=0)\n            self._cache[(\"__sid_stats__\",)] = (mu_1_hat, mu_2_hat, c0_1, c0_2, S_1_c, S_2_c)  # type: ignore[assignment]\n        return res\n\n    def transport_gram(\n        self,\n        old_inputs: Mapping[str, torch.Tensor],\n        new_inputs: Mapping[str, torch.Tensor],\n        ridge_transport: float = 1e-4,\n        max_drift: float = 0.35,\n        adaptive_drift_guard: bool = True,\n    ) -> dict[str, torch.Tensor]:\n        \"\"\"\n        Pillar 3: Exemplar-Free Block-Gram Pushforward Transport with Perturbative Drift Guard.\n        Estimates modality-specific linear transport operators M_m = I + eta_m * dM_m from\n        current-task feature shifts (old_inputs -> new_inputs) and transforms all historical\n        class Gram blocks and class sums in-place: G_c <- M G_c M^T, S_c <- M S_c (B = 0).\n        When `adaptive_drift_guard=True`, automatically damps or bypasses transport if the\n        relative feature drift exceeds the first-order perturbative trust region `max_drift`.\n        \"\"\"\n        P = self.projection_dim\n        eye = torch.eye(P, dtype=torch.float64)\n        operators: dict[str, torch.Tensor] = {}\n\n        for m in self.modalities:\n            if m in old_inputs and m in new_inputs and old_inputs[m] is not None and new_inputs[m] is not None:\n                z_old = self.features(old_inputs, (m,))\n                z_new = self.features(new_inputs, (m,))\n                diff = z_new - z_old\n                delta_rel = float(torch.linalg.norm(diff) / torch.linalg.norm(z_old).clamp(min=1e-12))\n                if adaptive_drift_guard and delta_rel >= max_drift:\n                    operators[m] = eye\n                    continue\n                cov_old = z_old.T @ z_old\n                tr_scale = max(float(torch.trace(cov_old)) / P, 1e-12)\n                if adaptive_drift_guard and max_drift > 0:\n                    ratio2 = (delta_rel / max_drift) ** 2\n                    eff_ridge = ridge_transport + 0.05 * ratio2\n                    eta = 1.0 - ratio2\n                else:\n                    eff_ridge = ridge_transport\n                    eta = 1.0\n                lam = eff_ridge * tr_scale\n                dM_T = torch.linalg.solve(cov_old + lam * eye, z_old.T @ diff)\n                operators[m] = eye + eta * dM_T.T\n            else:\n                operators[m] = eye\n\n        M_blocks = [operators[m] for m in self.modalities]\n        if len(M_blocks) == 2:\n            M_1, M_2 = M_blocks[0], M_blocks[1]\n            M_1_T, M_2_T = M_1.T, M_2.T\n            if self.class_grams is not None:\n                for c in range(self.num_classes):\n                    if int(self.class_counts[c].item()) > 0:\n                        Gc = self.class_grams[c]\n                        Gc[:P, :P] = M_1 @ Gc[:P, :P] @ M_1_T\n                        Gc[P:, P:] = M_2 @ Gc[P:, P:] @ M_2_T\n                        G_12_new = M_1 @ Gc[:P, P:] @ M_2_T\n                        Gc[:P, P:] = G_12_new\n                        Gc[P:, :P] = G_12_new.T\n                self.gram = self.class_grams.sum(dim=0)\n            else:\n                self.gram[:P, :P] = M_1 @ self.gram[:P, :P] @ M_1_T\n                self.gram[P:, P:] = M_2 @ self.gram[P:, P:] @ M_2_T\n                G_12_new = M_1 @ self.gram[:P, P:] @ M_2_T\n                self.gram[:P, P:] = G_12_new\n                self.gram[P:, :P] = G_12_new.T\n            self.class_sums[:P] = M_1 @ self.class_sums[:P]\n            self.class_sums[P:] = M_2 @ self.class_sums[P:]\n        else:\n            M_full = torch.block_diag(*M_blocks)\n            if self.class_grams is not None:\n                for c in range(self.num_classes):\n                    if int(self.class_counts[c].item()) > 0:\n                        self.class_grams[c] = M_full @ self.class_grams[c] @ M_full.T\n                        self.class_sums[:, c] = M_full @ self.class_sums[:, c]\n                self.gram = self.class_grams.sum(dim=0)\n            else:\n                self.gram = M_full @ self.gram @ M_full.T\n                self.class_sums = M_full @ self.class_sums\n        self._cache.clear()\n        return operators\n\n    @staticmethod\n    def auto_continual_policy(\n        g1_gain_pts: float | None = None,\n        incremental_mode: str = \"dil\",\n        min_g1_gain_pts: float = 2.0,\n    ) -> dict[str, object]:\n        \"\"\"\n        Self-governing continual adaptation policy derived from cross-dataset empirical laws\n        (CrisisMMD v2.0 DIL vs. UPMC Food-101 100-class CIL):\n          1. If `g1_gain_pts < min_g1_gain_pts` (backbone already aligned with target domain):\n             keep backbone frozen (`tau=1.0`) and use `BlockAnalyticCL` with `adaptive_sid=True`.\n          2. If `incremental_mode == 'dil'` (Domain-Incremental, fixed class set):\n             adapt Vision tower in null space (`tau=0.95`) while keeping Text tower as frozen\n             semantic anchor (`towers=('vision',)`).\n          3. If `incremental_mode == 'cil'` (Class-Incremental, expanding class set):\n             adapt both towers with strict null-space protection (`tau=0.99`) + Gram transport & SID.\n        \"\"\"\n        mode = incremental_mode.strip().lower()\n        if mode not in {\"dil\", \"cil\"}:\n            raise ValueError(f\"incremental_mode must be 'dil' or 'cil', got {incremental_mode!r}\")\n        if g1_gain_pts is not None and float(g1_gain_pts) < min_g1_gain_pts:\n            return {\n                \"regime\": \"frozen_analytic_sid\",\n                \"lora_enabled\": False,\n                \"tau\": 1.0,\n                \"towers\": (),\n                \"use_transport\": False,\n                \"adaptive_sid\": True,\n            }\n        if mode == \"dil\":\n            return {\n                \"regime\": \"dil_vision_nullspace_frozen_text\",\n                \"lora_enabled\": True,\n                \"tau\": 0.95,\n                \"towers\": (\"vision\",),\n                \"use_transport\": True,\n                \"adaptive_sid\": True,\n            }\n        return {\n            \"regime\": \"cil_dual_nullspace_transport_sid\",\n            \"lora_enabled\": True,\n            \"tau\": 0.99,\n            \"towers\": (\"vision\", \"text\"),\n            \"use_transport\": True,\n            \"adaptive_sid\": True,\n        }\n\n    def solve(self, modalities: Sequence[str] | None = None) -> torch.Tensor:\n        mods = self._check(modalities)\n        if mods not in self._cache:\n            if int(self.class_counts.sum()) <= 0:\n                raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n            if len(mods) == 2 and len(self.modalities) == 2 and (self.modality_ridges is not None or self.rho != 1.0):\n                W_both, W_1, W_2 = self.solve_schur()\n                self._cache[self.modalities] = W_both\n                self._cache[(self.modalities[0],)] = W_1\n                self._cache[(self.modalities[1],)] = W_2\n                return self._cache[mods]\n\n            G, S = self._weighted_gram_and_sums(mods)\n            P = self.projection_dim\n            if self.modality_ridges is not None and len(mods) == 1:\n                r_m = self.modality_ridges.get(mods[0], self.ridge)\n                lam = r_m * max(float(torch.trace(G)) / G.shape[0], 1e-12)\n            else:\n                lam = self.ridge * max(float(torch.trace(G)) / G.shape[0], 1e-12)\n            self._cache[mods] = torch.linalg.solve(G + lam * torch.eye(G.shape[0], dtype=G.dtype), S)\n        return self._cache[mods]\n\n    def logits(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        modalities: Sequence[str] | None = None,\n        adaptive_sid: bool = False,\n        tau_gate: float = 2.0,\n    ) -> torch.Tensor:\n        \"\"\"\n        Logits using only `modalities` (default: all present in `inputs`).\n        When `adaptive_sid=True` and both modalities are present, applies the continuous\n        Schur-Innovation Decomposition (SID) reliability-modulated inference law.\n        \"\"\"\n        if modalities is None:\n            present = tuple(m for m in self.modalities if m in inputs and inputs[m] is not None)\n            mods = self._check(present if present else None)\n        else:\n            mods = self._check(modalities)\n\n        if adaptive_sid and len(mods) == 2 and len(self.modalities) == 2:\n            m1, m2 = self.modalities\n            z_1 = self.features(inputs, (m1,))\n            z_2 = self.features(inputs, (m2,))\n            W_1_o, W_2_o, dW_1_s, dW_2_s, _ = self.schur_innovation_decomposition()\n            mu_1_hat, mu_2_hat, c0_1, c0_2, S_1_c, S_2_c = self._cache[(\"__sid_stats__\",)]  # type: ignore[misc]\n\n            L_1_o = z_1 @ W_1_o\n            L_2_o = z_2 @ W_2_o\n            L_1_in = z_1 @ dW_1_s\n            L_2_in = z_2 @ dW_2_s\n\n            z_1_n = F.normalize(z_1, dim=-1)\n            z_2_n = F.normalize(z_2, dim=-1)\n            cone_1 = (z_1_n @ mu_1_hat).clamp(min=0.0)\n            cone_2 = (z_2_n @ mu_2_hat).clamp(min=0.0)\n            f_cone_1 = torch.where(\n                cone_1 > 1e-6,\n                ((cone_1 - 0.46) / max(c0_1 - 0.52, 0.10)).clamp(0.02, 1.0),\n                torch.zeros_like(cone_1),\n            )\n            f_cone_2 = torch.where(\n                cone_2 > 1e-6,\n                ((cone_2 - 0.46) / max(c0_2 - 0.52, 0.10)).clamp(0.02, 1.0),\n                torch.zeros_like(cone_2),\n            )\n\n            k_top = min(2, self.num_classes)\n            P_1 = z_1_n @ S_1_c\n            P_2 = z_2_n @ S_2_c\n            pmax_1 = P_1.max(dim=1).values.clamp(min=1e-6)\n            pmax_2 = P_2.max(dim=1).values.clamp(min=1e-6)\n            ptop2_1 = torch.topk(P_1, k=k_top, dim=1).values\n            ptop2_2 = torch.topk(P_2, k=k_top, dim=1).values\n            pmargin_1 = (ptop2_1[:, 0] - ptop2_1[:, -1]).clamp(min=1e-6)\n            pmargin_2 = (ptop2_2[:, 0] - ptop2_2[:, -1]).clamp(min=1e-6)\n\n            top2_1 = torch.topk(L_1_o, k=k_top, dim=1).values\n            lmargin_1 = (top2_1[:, 0] - top2_1[:, -1]).clamp(min=1e-6)\n            top2_2 = torch.topk(L_2_o, k=k_top, dim=1).values\n            lmargin_2 = (top2_2[:, 0] - top2_2[:, -1]).clamp(min=1e-6)\n\n            conf_1 = (f_cone_1 ** 2) * lmargin_1 * (pmax_1 ** 2) * (0.05 + pmargin_1)\n            conf_2 = (f_cone_2 ** 2) * lmargin_2 * (pmax_2 ** 2) * (0.05 + pmargin_2)\n            max_conf = torch.maximum(conf_1, conf_2).clamp(min=1e-12)\n            r_1 = (conf_1 / max_conf).clamp(0.0, 1.0)\n            r_2 = (conf_2 / max_conf).clamp(0.0, 1.0)\n\n            discrepancy = (torch.abs(conf_1 - conf_2) / (conf_1 + conf_2).clamp(min=1e-12)).clamp(0.0, 1.0)\n            L_1_c = L_1_o - L_1_o.mean(dim=1, keepdim=True)\n            L_2_c = L_2_o - L_2_o.mean(dim=1, keepdim=True)\n            cos_agree = F.cosine_similarity(L_1_c, L_2_c, dim=1).clamp(min=0.0, max=1.0)\n            cone_drop = 1.0 - torch.minimum(f_cone_1, f_cone_2)\n            anomaly = discrepancy * (1.0 - 0.7 * cos_agree) * (0.35 + 0.65 * cone_drop)\n            excess = ((anomaly - 0.25) / 0.35).clamp(0.0, 1.0)\n            gate_blend = (excess ** 1.5).unsqueeze(1)\n\n            p_exp = 2.0 if tau_gate == 2.0 else 2.0 / max(float(tau_gate), 1e-6)\n            alpha_1 = (1.0 - gate_blend) + gate_blend * torch.pow(r_1, p_exp).unsqueeze(1)\n            alpha_2 = (1.0 - gate_blend) + gate_blend * torch.pow(r_2, p_exp).unsqueeze(1)\n\n            both_present = (torch.minimum(cone_1, cone_2) / 0.45).clamp(0.0, 1.0).unsqueeze(1)\n            rho_raw = (torch.minimum(r_1, r_2) * cos_agree).unsqueeze(1)\n            rho_x = (1.0 - gate_blend) + gate_blend * rho_raw\n            rho_1 = alpha_1 * (0.4 * both_present + 0.6 * rho_x)\n            rho_2 = alpha_2 * (0.4 * both_present + 0.6 * rho_x)\n\n            return alpha_1 * L_1_o + alpha_2 * L_2_o + rho_1 * L_1_in + rho_2 * L_2_in\n\n        return self.features(inputs, mods) @ self.solve(mods)\n\n    def predict(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        modalities: Sequence[str] | None = None,\n        adaptive_sid: bool = False,\n        tau_gate: float = 2.0,\n    ) -> torch.Tensor:\n        return self.logits(inputs, modalities, adaptive_sid=adaptive_sid, tau_gate=tau_gate).argmax(dim=-1)\n", "mm_flex_merge/models/nullspace_lora.py": "\"\"\"\nStage 1: Null-Space Uncentered-Gram Orthogonal SessionLoRA.\n\nImplements:\n- `AttnState`: Uncentered input Gram matrix accumulator (`C^{(t)} = sum X_t^T X_t`)\n  and exact orthogonal complement projector (`P_perp^{(t)} = I - V_t V_t^T`) via SVD/EVD.\n- `SessionLoRA`: Continual low-rank adapter (`A_eff,t = A_t P_perp^{(t-1)}`) with\n  exact O(1) constant-time additive parameter-space merging (`W_merged = W_0 + sum dW_k`)\n  and reversible session un-merging (`unmerge_session`).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nTEXT_MASK: dict[str, torch.Tensor | None] = {\"mask\": None}\n\n\nclass AttnState:\n    \"\"\"Tracks the uncentered input activation Gram matrix and null-space projector.\"\"\"\n\n    def __init__(\n        self,\n        d: int,\n        is_text: bool = False,\n        device: str | torch.device = \"cpu\",\n        text_mask_holder: dict[str, torch.Tensor | None] | None = None,\n    ) -> None:\n        self.gram = torch.zeros(d, d, dtype=torch.float32, device=device)\n        self.proj: torch.Tensor | None = None\n        self.is_text = bool(is_text)\n        self.recording = False\n        self.rank = 0\n        self._mask_holder = TEXT_MASK if text_mask_holder is None else text_mask_holder\n\n    def hook(self, module: nn.Module, inputs: tuple[torch.Tensor, ...]) -> None:\n        if not self.recording:\n            return\n        dev_type = \"cuda\" if self.gram.is_cuda else \"cpu\"\n        with torch.amp.autocast(dev_type, enabled=False):\n            x = inputs[0].detach()\n            mask = self._mask_holder.get(\"mask\")\n            if self.is_text and mask is not None:\n                x = x[mask.bool()]\n            x = x.reshape(-1, x.shape[-1]).float().to(self.gram.device)\n            self.gram += x.T @ x\n\n    def update_projection(self, tau: float | None) -> None:\n        if tau is None:\n            return\n        g = torch.nan_to_num(self.gram.double().cpu(), nan=0.0, posinf=0.0, neginf=0.0)\n        g = 0.5 * (g + g.T)\n        evals, evecs = torch.linalg.eigh(g)\n        evals, evecs = evals.flip(0).clamp(min=0.0), evecs.flip(1)\n        tot = evals.sum().clamp(min=1e-12)\n        k = min(int((torch.cumsum(evals, 0) / tot < tau).sum().item()) + 1, evals.numel())\n        U = evecs[:, :k].float()\n        self.proj = (torch.eye(U.shape[0], dtype=torch.float32) - U @ U.T).to(self.gram.device)\n        self.rank = k\n\n\nclass SessionLoRA(nn.Module):\n    \"\"\"\n    Null-Space Projected Continual LoRA with O(1) constant-time merged forward pass.\n\n    Completed sessions are folded into `delta_weight` (`sum_{k=1}^{t-1} (alpha/r) B_k A_{eff,k}`)\n    so inference and subsequent session training execute in constant O(1) time regardless\n    of the number of tasks `T`, while retaining `(A_{eff,k}, B_k)` in `self.past` for\n    reversible session un-merging (`unmerge_session`).\n    \"\"\"\n\n    def __init__(\n        self,\n        base: nn.Linear,\n        owner: AttnState,\n        r: int = 16,\n        alpha: float = 32.0,\n    ) -> None:\n        super().__init__()\n        self.base = base\n        self.owner = owner\n        self.r = int(r)\n        self.alpha = float(alpha)\n        self.scale = self.alpha / self.r\n        self.past = nn.ParameterList()\n        self.A: nn.Parameter | None = None\n        self.B: nn.Parameter | None = None\n        self.P: torch.Tensor | None = None\n        self.register_buffer(\n            \"delta_weight\",\n            torch.zeros_like(base.weight, dtype=torch.float32),\n            persistent=False,\n        )\n        self._has_merged = False\n\n    def _commit_active_to_past(self) -> None:\n        if self.A is None or self.B is None:\n            return\n        A_eff = (self.A.detach() @ self.P) if self.P is not None else self.A.detach()\n        B_det = self.B.detach()\n        self.past.append(nn.Parameter(A_eff, requires_grad=False))\n        self.past.append(nn.Parameter(B_det, requires_grad=False))\n        self.delta_weight.add_((B_det.float() @ A_eff.float()) * self.scale)\n        self._has_merged = True\n\n    def new_session(self) -> None:\n        if self.A is not None:\n            self._commit_active_to_past()\n        self.P = None if self.owner.proj is None else self.owner.proj.clone().to(self.base.weight.device)\n        d_in, d_out = self.base.in_features, self.base.out_features\n        dev = self.base.weight.device\n        self.A = nn.Parameter(torch.randn(self.r, d_in, device=dev) / d_in ** 0.5)\n        self.B = nn.Parameter(torch.zeros(d_out, self.r, device=dev))\n\n    def rebuild_merged_delta(self) -> None:\n        \"\"\"Recomputes `delta_weight` from `self.past` (e.g., after un-merging a session).\"\"\"\n        self.delta_weight.zero_()\n        for i in range(0, len(self.past), 2):\n            A_eff = self.past[i].detach().float()\n            B_det = self.past[i + 1].detach().float()\n            self.delta_weight.add_((B_det @ A_eff) * self.scale)\n        self._has_merged = len(self.past) > 0\n\n    def unmerge_session(self, session_idx: int) -> None:\n        \"\"\"Removes a committed past session `session_idx` in O(1) without retraining.\"\"\"\n        num_past = len(self.past) // 2\n        if session_idx < 0:\n            session_idx += num_past\n        if not (0 <= session_idx < num_past):\n            raise IndexError(f\"session_idx={session_idx} out of range for {num_past} committed sessions.\")\n        drop = {2 * session_idx, 2 * session_idx + 1}\n        self.past = nn.ParameterList([p for idx, p in enumerate(self.past) if idx not in drop])\n        self.rebuild_merged_delta()\n\n    def merged_weight(self) -> torch.Tensor:\n        \"\"\"Returns the full effective weight `W_0 + sum_{k=1}^t (alpha/r) B_k A_{eff,k}`.\"\"\"\n        w = self.base.weight.float() + self.delta_weight\n        if self.A is not None and self.B is not None:\n            A_eff = self.A if self.P is None else self.A @ self.P\n            w = w + (self.B.float() @ A_eff.float()) * self.scale\n        return w.to(self.base.weight.dtype)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if self._has_merged:\n            w_past = self.base.weight + self.delta_weight.to(self.base.weight.dtype)\n            out = F.linear(x, w_past, self.base.bias)\n        else:\n            out = self.base(x)\n        if self.A is None or self.B is None:\n            return out\n        A = self.A if self.P is None else self.A @ self.P\n        return out + (x @ A.T.to(x.dtype) @ self.B.T.to(x.dtype)) * self.scale\n", "mm_flex_merge/models/heads.py": "\"\"\"\nMultimodal Head Architectures for Continual Model Merging.\n\"\"\"\n\nfrom typing import Optional, Tuple\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass LinearExpert(nn.Module):\n    \"\"\"Baseline Unimodal / Naive Linear Expert.\"\"\"\n    def __init__(self, in_features: int = 512, hidden_dim: int = 256, num_classes: int = 3):\n        super().__init__()\n        self.proj = nn.Linear(in_features, hidden_dim, bias=False)\n        self.classifier = nn.Linear(hidden_dim, num_classes, bias=True)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        h = F.relu(self.proj(F.normalize(x, dim=-1)))\n        return self.classifier(h)\n\n\nclass UnimodalExpert(nn.Module):\n    \"\"\"Expert chuy\u00ean tr\u00e1ch \u0111\u01a1n ph\u01b0\u01a1ng th\u1ee9c (Vision-Only ho\u1eb7c Text-Only).\"\"\"\n    def __init__(self, in_features: int = 512, hidden_dim: int = 256, num_classes: int = 3, modality: str = \"vision\"):\n        super().__init__()\n        if modality not in (\"vision\", \"text\"):\n            raise ValueError(f\"Unsupported modality '{modality}'. Expected 'vision' or 'text'.\")\n        self.modality = modality\n        self.proj = nn.Linear(in_features, hidden_dim, bias=False)\n        self.classifier = nn.Linear(hidden_dim, num_classes, bias=True)\n\n    def forward(self, img: Optional[torch.Tensor] = None, txt: Optional[torch.Tensor] = None) -> torch.Tensor:\n        feat = img if self.modality == \"vision\" else txt\n        if feat is None:\n            raise ValueError(f\"Feature for modality '{self.modality}' cannot be None.\")\n        h = F.relu(self.proj(F.normalize(feat, dim=-1)))\n        return self.classifier(h)\n\n\nclass DualStreamCrossGatedExpert(nn.Module):\n    \"\"\"\n    Multimodal Dual-Stream Cross-Gated Expert Head.\n    \n    \u0110\u1eb7c t\u00ednh thi\u1ebft k\u1ebf \u0111\u1ed9c quy\u1ec1n c\u1ee7a MM-FLEX-MERGE:\n    1. Chi\u1ebfu song song \u0111\u1eb7c tr\u01b0ng Th\u1ecb gi\u00e1c (Image) v\u00e0 Ng\u00f4n ng\u1eef (Text) v\u00e0o kh\u00f4ng gian ti\u1ec1m \u1ea9n chung.\n    2. C\u1ed5ng \u0111i\u1ec1u h\u00f2a t\u01b0\u01a1ng h\u1ed7 (Cross-Modal Attention Gate) sinh tr\u1ecdng s\u1ed1 \u0111\u1ed9ng alpha_img, alpha_txt.\n    3. T\u1ef1 th\u00edch \u1ee9ng kh\u00e1ng suy hao (Missing-Modality Invariant): T\u1ef1 \u0111\u1ed9ng ph\u00e1t hi\u1ec7n khi 1 k\u00eanh b\u1ecb m\u1ea5t\n       ho\u1eb7c che m\u1edd ho\u00e0n to\u00e0n \u0111\u1ec3 d\u1ed3n 100% tr\u1ecdng s\u1ed1 sang k\u00eanh c\u00f2n l\u1ea1i.\n    \"\"\"\n    def __init__(\n        self,\n        in_features: int = 512,\n        hidden_dim: int = 256,\n        num_classes: int = 3,\n        temperature: float = 1.0,\n    ):\n        super().__init__()\n        self.in_features = in_features\n        self.hidden_dim = hidden_dim\n        self.num_classes = num_classes\n        self.temperature = temperature\n\n        # Nh\u00e1nh chi\u1ebfu ti\u1ec1m \u1ea9n song song\n        self.proj_img = nn.Linear(in_features, hidden_dim, bias=False)\n        self.proj_txt = nn.Linear(in_features, hidden_dim, bias=False)\n\n        # C\u1ed5ng \u0111i\u1ec1u h\u00f2a t\u01b0\u01a1ng h\u1ed7 (Cross-Modal Gate)\n        self.gate_network = nn.Sequential(\n            nn.Linear(in_features * 2, hidden_dim // 2, bias=True),\n            nn.ReLU(),\n            nn.Linear(hidden_dim // 2, 2, bias=True),\n        )\n\n        # \u0110\u1ea7u ph\u00e2n l\u1edbp h\u1ee3p nh\u1ea5t\n        self.classifier = nn.Linear(hidden_dim, num_classes, bias=True)\n\n    def forward(self, img: torch.Tensor, txt: torch.Tensor) -> torch.Tensor:\n        # Chu\u1ea9n h\u00f3a L2 vector nh\u00fang \u0111\u1ea7u v\u00e0o\n        img_norm = torch.norm(img, dim=-1, keepdim=True) + 1e-8\n        txt_norm = torch.norm(txt, dim=-1, keepdim=True) + 1e-8\n\n        img_n = img / img_norm\n        txt_n = txt / txt_norm\n\n        # Ki\u1ec3m tra s\u1ef1 t\u1ed3n t\u1ea1i c\u1ee7a ph\u01b0\u01a1ng th\u1ee9c (kh\u00e1ng khuy\u1ebft thi\u1ebfu)\n        img_is_active = (img_norm > 1e-5).float()\n        txt_is_active = (txt_norm > 1e-5).float()\n\n        # T\u00ednh to\u00e1n \u0111\u1eb7c tr\u01b0ng ti\u1ec1m \u1ea9n\n        h_img = self.proj_img(img_n)\n        h_txt = self.proj_txt(txt_n)\n\n        # Tr\u1ecdng s\u1ed1 c\u1ed5ng \u0111i\u1ec1u h\u00f2a\n        gate_input = torch.cat([img_n, txt_n], dim=-1)\n        gate_logits = self.gate_network(gate_input) / self.temperature\n        raw_weights = F.softmax(gate_logits, dim=-1) # [B, 2]\n\n        # Hi\u1ec7u ch\u1ec9nh th\u00edch \u1ee9ng khi 1 k\u00eanh b\u1ecb suy hao\n        w_img = raw_weights[:, 0:1] * img_is_active\n        w_txt = raw_weights[:, 1:2] * txt_is_active\n        sum_w = w_img + w_txt + 1e-8\n        w_img = w_img / sum_w\n        w_txt = w_txt / sum_w\n\n        # T\u1ed5ng h\u1ee3p bi\u1ec3u di\u1ec5n \u0111a ph\u01b0\u01a1ng th\u1ee9c\n        h_fused = F.relu(w_img * h_img + w_txt * h_txt)\n        return self.classifier(h_fused)\n\n    def extract_flat_parameters(self) -> torch.Tensor:\n        \"\"\"Tr\u00edch xu\u1ea5t to\u00e0n b\u1ed9 tham s\u1ed1 c\u1ee7a Head th\u00e0nh 1 vector 1 chi\u1ec1u li\u00ean t\u1ee5c.\"\"\"\n        return torch.cat([p.detach().clone().flatten() for p in self.parameters()])\n\n    def load_flat_parameters(self, flat_params: torch.Tensor):\n        \"\"\"N\u1ea1p vector tham s\u1ed1 ph\u1eb3ng v\u00e0o c\u00e1c layer c\u1ee7a Model.\"\"\"\n        idx = 0\n        for p in self.parameters():\n            num = p.numel()\n            p.data.copy_(flat_params[idx : idx + num].view(p.shape))\n            idx += num\n", "mm_flex_merge/models/routers.py": "\"\"\"\nDual-Centroid Multimodal Router with Negative Routing Bias and 0-Second Data Opt-Out.\n\"\"\"\n\nfrom typing import List, Optional\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass DualCentroidRouter(nn.Module):\n    \"\"\"\n    Multimodal Dual-Centroid Router for Task / Expert Routing.\n    \n    \u0110\u1eb7c t\u00ednh thi\u1ebft k\u1ebf:\n    1. Qu\u1ea3n l\u00fd \u0111\u1ed3ng th\u1eddi Centroid Th\u1ecb gi\u00e1c v\u00e0 Centroid Ng\u00f4n ng\u1eef c\u1ee7a t\u1eebng Task.\n    2. T\u00edch h\u1ee3p vector Negative Routing Bias (b_k) \u0111\u1ec3 ch\u1ed1ng thi\u00ean l\u1ec7ch ph\u00e2n b\u1ed1 nhi\u1ec7m v\u1ee5.\n    3. T\u00edch h\u1ee3p c\u01a1 ch\u1ebf Zero-Latency Machine Unlearning (0-second Data Opt-Out):\n       H\u1ea1 Logit c\u1ee7a Task c\u1ea7n x\u00f3a xu\u1ed1ng -10^9 \u0111\u1ec3 x\u00e1c su\u1ea5t Softmax ti\u1ebfn v\u1ec1 0.0000% tuy\u1ec7t \u0111\u1ed1i.\n    \"\"\"\n    def __init__(self, num_tasks: int, dim: int = 512, default_bias: float = -0.25):\n        super().__init__()\n        self.num_tasks = num_tasks\n        self.dim = dim\n        self.default_bias = default_bias\n\n        # Ma tr\u1eadn Centroid \u0111\u1ed9c l\u1eadp\n        self.image_centroids = nn.Parameter(torch.zeros(num_tasks, dim), requires_grad=False)\n        self.text_centroids = nn.Parameter(torch.zeros(num_tasks, dim), requires_grad=False)\n\n        # Vector Bias \u0111i\u1ec1u h\u00f2a\n        init_bias = torch.full((num_tasks,), default_bias)\n        init_bias[0] = 0.0 # Anchor task gi\u1eef bias = 0\n        self.biases = nn.Parameter(init_bias, requires_grad=False)\n        self.opt_out_mask = nn.Parameter(torch.ones(num_tasks, dtype=torch.bool), requires_grad=False)\n\n    def update_centroids(self, task_idx: int, train_img: torch.Tensor, train_txt: torch.Tensor):\n        \"\"\"C\u1eadp nh\u1eadt t\u1ecda \u0111\u1ed9 Centroid t\u00e2m c\u1ee5m d\u1ef1a tr\u00ean d\u1eef li\u1ec7u hu\u1ea5n luy\u1ec7n c\u1ee7a Task.\"\"\"\n        c_img = F.normalize(train_img.mean(dim=0), dim=-1)\n        c_txt = F.normalize(train_txt.mean(dim=0), dim=-1)\n        self.image_centroids.data[task_idx].copy_(c_img)\n        self.text_centroids.data[task_idx].copy_(c_txt)\n\n    def get_joint_centroids(self, beta: float = 0.5) -> torch.Tensor:\n        \"\"\"Sinh ma tr\u1eadn Centroid li\u00ean h\u1ee3p \u0111a ph\u01b0\u01a1ng th\u1ee9c.\"\"\"\n        joint = beta * self.image_centroids + (1.0 - beta) * self.text_centroids\n        return F.normalize(joint, dim=-1)\n\n    def route(self, query_img: torch.Tensor, query_txt: torch.Tensor, beta: float = 0.5) -> torch.Tensor:\n        \"\"\"\n        \u0110\u1ecbnh tuy\u1ebfn m\u1eabu ki\u1ec3m th\u1eed \u0111\u1ebfn c\u00e1c Task Experts.\n        Tr\u1ea3 v\u1ec1 ph\u00e2n b\u1ed1 x\u00e1c su\u1ea5t Softmax [Batch_Size, Num_Tasks].\n        T\u1ef1 \u0111\u1ed9ng chuy\u1ec3n sang t\u00e2m c\u1ee5m \u0111\u01a1n ph\u01b0\u01a1ng th\u1ee9c khi m\u1ed9t k\u00eanh b\u1ecb khuy\u1ebft (||x_m||_2 <= 1e-5).\n        \"\"\"\n        img_active = (torch.norm(query_img, dim=-1, keepdim=True) > 1e-5)\n        txt_active = (torch.norm(query_txt, dim=-1, keepdim=True) > 1e-5)\n\n        q_img = F.normalize(query_img, dim=-1)\n        q_txt = F.normalize(query_txt, dim=-1)\n        q_joint = F.normalize(beta * q_img + (1.0 - beta) * q_txt, dim=-1)\n\n        joint_centroids = self.get_joint_centroids(beta)\n        sim_joint = q_joint @ joint_centroids.T\n        sim_img = q_img @ F.normalize(self.image_centroids, dim=-1).T\n        sim_txt = q_txt @ F.normalize(self.text_centroids, dim=-1).T\n\n        both_active = img_active & txt_active\n        sim = torch.where(\n            both_active,\n            sim_joint,\n            torch.where(img_active, sim_img, sim_txt),\n        )\n        logits = sim + self.biases.unsqueeze(0)\n\n        # N\u1ebfu task b\u1ecb opt-out, d\u00ecm logit xu\u1ed1ng c\u1ef1c \u00e2m\n        logits = torch.where(self.opt_out_mask.unsqueeze(0), logits, torch.tensor(-1e9, device=logits.device))\n        return F.softmax(logits, dim=-1)\n\n    def opt_out_task(self, task_idx: int):\n        \"\"\"\n        C\u01a1 ch\u1ebf 0-Second Data Opt-Out:\n        G\u1ee1 b\u1ecf ho\u00e0n to\u00e0n Task kh\u1ecfi h\u1ec7 th\u1ed1ng suy lu\u1eadn v\u1edbi th\u1eddi gian tr\u1ec5 0.000 gi\u00e2y.\n        \"\"\"\n        self.biases.data[task_idx] = -1e9\n        self.opt_out_mask.data[task_idx] = False\n\n    def restore_task(self, task_idx: int):\n        \"\"\"Kh\u00f4i ph\u1ee5c l\u1ea1i Task n\u1ebfu c\u00f3 y\u00eau c\u1ea7u t\u00e1i k\u00edch ho\u1ea1t.\"\"\"\n        self.biases.data[task_idx] = 0.0 if task_idx == 0 else self.default_bias\n        self.opt_out_mask.data[task_idx] = True\n\n    def is_opted_out(self, task_idx: int) -> bool:\n        return not bool(self.opt_out_mask[task_idx].item())\n", "h1_gate_engine.py": "\"\"\"\nSelf-contained engine for gates H1-G1 / H1-G2 (used locally and embedded verbatim in the Colab notebook).\n\nG1: adaptation gain of LoRA (both CLIP towers) vs frozen, analytic head (BlockAnalyticCL).\nG2: sequential LoRA over tasks; each session's LoRA input side is projected onto the null space of\n    the accumulated input Gram of previous sessions, analytic statistics are never recomputed.\nAll evaluation is on the validation split; the caller must drop the test split beforehand.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom sklearn.metrics import accuracy_score, f1_score\n\nfrom mm_flex_merge.models import TEXT_MASK, AttnState, BlockAnalyticCL, SessionLoRA\n\nNAME = \"openai/clip-vit-base-patch16\"\nMEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)\nSTD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)\nR, ALPHA = 16, 32\nDEV = \"cuda\" if torch.cuda.is_available() else \"cpu\"\nif DEV == \"cuda\":\n    AMP = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\nelse:\n    AMP = None\n\n\nLOG_PATH = None    # set by the notebook to a file inside the exported results folder\n\n\ndef log(msg):\n    line = f\"{time.strftime('%H:%M:%S')} {msg}\"\n    print(line, flush=True)\n    if LOG_PATH is not None:\n        with open(LOG_PATH, \"a\", encoding=\"utf-8\") as f:\n            f.write(line + \"\\n\")\n\n\n# ----------------------------------------------------------------------------- data\nclass _ImageSet(torch.utils.data.Dataset):\n    def __init__(self, paths):\n        self.paths = list(paths)\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, i):\n        try:\n            im = Image.open(self.paths[i]).convert(\"RGB\")\n        except Exception:\n            im = Image.new(\"RGB\", (224, 224))\n        w, h = im.size\n        s = 224 / min(w, h)\n        im = im.resize((max(224, round(w * s)), max(224, round(h * s))), Image.BICUBIC)\n        w, h = im.size\n        l, t = (w - 224) // 2, (h - 224) // 2\n        im = im.crop((l, t, l + 224, t + 224))\n        return torch.from_numpy(np.asarray(im, dtype=np.uint8).copy()).permute(2, 0, 1)\n\n\ndef decode_images(paths, workers=2, bs=256, mmap_path=None):\n    import tempfile\n    from pathlib import Path\n    w_eff = min(max(int(workers or 2), 0), 4)\n    dl = torch.utils.data.DataLoader(_ImageSet(paths), batch_size=bs, num_workers=w_eff)\n    n = len(paths)\n    if mmap_path is not None or n > 15000:\n        p = Path(mmap_path) if mmap_path else Path(tempfile.gettempdir()) / f\"pix_mmap_{n}.bin\"\n        mm = np.memmap(str(p), dtype=np.uint8, mode=\"w+\", shape=(n, 3, 224, 224))\n        out = torch.from_numpy(mm)\n    else:\n        out = torch.empty(n, 3, 224, 224, dtype=torch.uint8)\n    k = 0\n    for b in dl:\n        out[k:k + len(b)] = b\n        k += len(b)\n    return out\n\n\ndef tokenize(texts, bs=4096):\n    from transformers import CLIPTokenizer\n    tok = CLIPTokenizer.from_pretrained(NAME)\n    ids_l, att_l = [], []\n    for i in range(0, len(texts), bs):\n        t = tok([str(x) for x in texts[i:i + bs]], padding=\"max_length\", truncation=True, max_length=77, return_tensors=\"pt\")\n        ids_l.append(t[\"input_ids\"].to(torch.int32))\n        att_l.append(t[\"attention_mask\"].to(torch.int8))\n    return torch.cat(ids_l, 0), torch.cat(att_l, 0)\n\n\ndef make_data(frame, *, path_col, text_col, label_col, task_col, task_order, num_classes, workers=8):\n    \"\"\"frame may contain train/val or train/val/test rows (column 'split').\"\"\"\n    assert set(frame[\"split\"]) <= {\"train\", \"val\", \"test\"}, \"unexpected split names\"\n    log(f\"decoding {len(frame)} images ...\")\n    pix = decode_images(frame[path_col].tolist(), workers=workers)\n    ids, att = tokenize(frame[text_col].tolist())\n    splits = frame[\"split\"].to_numpy()\n    return {\"pix\": pix, \"ids\": ids, \"att\": att,\n            \"y\": torch.tensor(frame[label_col].to_numpy(np.int64)),\n            \"train\": splits == \"train\",\n            \"val\": splits == \"val\",\n            \"test\": splits == \"test\",\n            \"task\": frame[task_col].to_numpy(), \"order\": list(task_order), \"C\": int(num_classes)}\n\n\n# ---------------------------------------------------------------------------- model\ndef build(lora=True, towers=(\"vision\", \"text\")):\n    from transformers import CLIPModel\n    m = CLIPModel.from_pretrained(NAME).to(DEV)\n    for p in m.parameters():\n        p.requires_grad_(False)\n    states, loras = [], []\n    if lora:\n        for name, mod in list(m.named_modules()):\n            if isinstance(getattr(mod, \"q_proj\", None), nn.Linear):\n                is_text = name.startswith(\"text_model\")\n                tower_name = \"text\" if is_text else \"vision\"\n                if tower_name not in towers:\n                    continue\n                st = AttnState(mod.q_proj.in_features, is_text, device=DEV)\n                mod.q_proj.register_forward_pre_hook(st.hook)\n                for attr in (\"q_proj\", \"v_proj\"):\n                    lo = SessionLoRA(getattr(mod, attr), st, r=R, alpha=ALPHA)\n                    setattr(mod, attr, lo)\n                    loras.append(lo)\n                states.append(st)\n    return m.eval(), states, loras\n\n\ndef encode(m, D, idx):\n    pix = ((D[\"pix\"][idx].to(DEV).float() / 255.0) - MEAN.to(DEV)) / STD.to(DEV)\n    ids, att = D[\"ids\"][idx].to(DEV, dtype=torch.long), D[\"att\"][idx].to(DEV, dtype=torch.long)\n    with torch.autocast(\"cuda\", dtype=AMP, enabled=AMP is not None):\n        vo = m.vision_model(pixel_values=pix)\n        i = m.visual_projection(vo[1] if isinstance(vo, tuple) else vo.pooler_output)\n        TEXT_MASK[\"mask\"] = att\n        to = m.text_model(input_ids=ids, attention_mask=att)\n        t = m.text_projection(to[1] if isinstance(to, tuple) else to.pooler_output)\n        TEXT_MASK[\"mask\"] = None\n    return F.normalize(i.float(), dim=-1), F.normalize(t.float(), dim=-1)\n\n\ndef train_session(m, loras, head, D, idx, *, epochs, bs, lr, seed=0):\n    for lo in loras:\n        lo.new_session()\n    params = [p for lo in loras for p in (lo.A, lo.B)] + list(head.parameters())\n    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)\n    use_scaler = (DEV == \"cuda\" and AMP == torch.float16)\n    scaler = (\n        torch.amp.GradScaler(\"cuda\", init_scale=1024.0, enabled=use_scaler)\n        if hasattr(torch, \"amp\")\n        else torch.cuda.amp.GradScaler(init_scale=1024.0, enabled=use_scaler)\n    )\n    g = torch.Generator().manual_seed(seed)\n    m.train()\n    for ep in range(epochs):\n        perm = idx[torch.randperm(len(idx), generator=g)]\n        tot = 0.0\n        for b in range(0, len(perm), bs):\n            s = perm[b:b + bs]\n            i, t = encode(m, D, s)\n            with torch.autocast(\"cuda\", dtype=AMP, enabled=AMP is not None):\n                loss = F.cross_entropy(head(torch.cat([i, t], 1)) * 10, D[\"y\"][s].to(DEV))\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            tot += float(loss.detach().item()) * len(s)\n        log(f\"   epoch {ep}: loss {tot / len(perm):.4f}\")\n    m.eval()\n    if DEV == \"cuda\":\n        torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef extract(m, D, idx, bs=256):\n    I, T = [], []\n    for b in range(0, len(idx), bs):\n        i, t = encode(m, D, idx[b:b + bs])\n        I.append(i.cpu())\n        T.append(t.cpu())\n    return torch.cat(I), torch.cat(T)\n\n\n# ----------------------------------------------------------------------- evaluation\nPRED = {}          # per-sample validation predictions, exported to predictions.npz for offline analysis\nHEAD_SEEDS = (2026, 2027, 2028)\nBETAS = (0.0, 0.5)\n\n\ndef scores(y_true, y_pred):\n    per_cls = f1_score(y_true, y_pred, average=None, zero_division=0)\n    return {\"macro_f1\": float(np.mean(per_cls)),\n            \"worst_class_f1\": float(np.min(per_cls)),\n            \"acc\": float(accuracy_score(y_true, y_pred))}\n\n\ndef _heads(C, beta):\n    return [BlockAnalyticCL({\"img\": 512, \"txt\": 512}, C, projection_dim=1024, class_balance_beta=beta, seed=s)\n            for s in HEAD_SEEDS]\n\n\ndef _logits(hs, feats, adaptive_sid=False):\n    return sum(h.logits({\"img\": feats[0], \"txt\": feats[1]}, adaptive_sid=adaptive_sid) for h in hs)\n\n\ndef analytic_fit_eval(tr_feats, y_tr, va_feats, y_va, C, key, te_feats=None, y_te=None):\n    out = {}\n    for b in BETAS:\n        hs = _heads(C, b)\n        for h in hs:\n            h.update({\"img\": tr_feats[0], \"txt\": tr_feats[1]}, y_tr)\n        pred = _logits(hs, va_feats).argmax(1).numpy()\n        PRED[f\"{key}/val/beta{b}\"] = pred.astype(np.int16)\n        out[f\"beta_{b}\"] = scores(y_va.numpy(), pred)\n        if te_feats is not None and y_te is not None and len(y_te) > 0:\n            pred_te = _logits(hs, te_feats).argmax(1).numpy()\n            PRED[f\"{key}/test/beta{b}\"] = pred_te.astype(np.int16)\n            out[f\"test_beta_{b}\"] = scores(y_te.numpy(), pred_te)\n        if b == 0.5:\n            for h in hs:\n                h.auto_spectral_ridges()\n            pred_sid = _logits(hs, va_feats, adaptive_sid=True).argmax(1).numpy()\n            out[\"unified_sid\"] = scores(y_va.numpy(), pred_sid)\n            if te_feats is not None and y_te is not None and len(y_te) > 0:\n                pred_sid_te = _logits(hs, te_feats, adaptive_sid=True).argmax(1).numpy()\n                out[\"test_unified_sid\"] = scores(y_te.numpy(), pred_sid_te)\n    return out\n\n\ndef _idx(mask):\n    return torch.tensor(np.where(mask)[0])\n\n\ndef run_g1(D, *, seeds, epochs, bs, lr, stage=\"g1\"):\n    tr, va = _idx(D[\"train\"]), _idx(D[\"val\"])\n    has_te = bool(D.get(\"test\") is not None and np.any(D[\"test\"]))\n    te = _idx(D[\"test\"]) if has_te else None\n    first = _idx(D[\"train\"] & (D[\"task\"] == D[\"order\"][0]))\n    res = {\"frozen\": {}, \"first_session\": {}, \"joint_oracle\": {}}\n    for arm, sub in ((\"frozen\", None), (\"first_session\", first), (\"joint_oracle\", tr)):\n        for seed in ([seeds[0]] if sub is None else seeds):\n            t0 = time.time()\n            torch.manual_seed(seed)\n            m, _, loras = build(lora=sub is not None)\n            if sub is not None:\n                log(f\"[{stage}/{arm}/seed{seed}] LoRA on {len(sub)} samples\")\n                train_session(m, loras, nn.Linear(1024, D[\"C\"]).to(DEV), D, sub, epochs=epochs, bs=bs, lr=lr, seed=seed)\n            f_tr, f_va = extract(m, D, tr), extract(m, D, va)\n            f_te = extract(m, D, te) if has_te else None\n            y_te = D[\"y\"][te] if has_te else None\n            res[arm][f\"seed{seed}\"] = analytic_fit_eval(\n                f_tr, D[\"y\"][tr], f_va, D[\"y\"][va], D[\"C\"], f\"{stage}/{arm}/seed{seed}\", f_te, y_te\n            )\n            res[arm][f\"seed{seed}\"][\"seconds\"] = time.time() - t0\n            log(f\"[{stage}/{arm}/seed{seed}] {res[arm][f'seed{seed}']}\")\n            del m\n            if DEV == \"cuda\":\n                import gc\n                gc.collect()\n                torch.cuda.empty_cache()\n    return res\n\n\ndef run_g2(D, *, taus, seeds, epochs, bs, lr, stage=\"g2\", extra_tower_configs=((\"vision_only_tau0.95\", 0.95, (\"vision\",)),)):\n    va = _idx(D[\"val\"])\n    has_te = bool(D.get(\"test\") is not None and np.any(D[\"test\"]))\n    te = _idx(D[\"test\"]) if has_te else None\n    yv, tv = D[\"y\"][va].numpy(), D[\"task\"][va.numpy()]\n    yte = D[\"y\"][te].numpy() if has_te else None\n    res = {}\n    configs = [((\"naive\" if tau is None else f\"nullspace_tau{tau}\"), tau, (\"vision\", \"text\")) for tau in taus]\n    if extra_tower_configs:\n        configs.extend(extra_tower_configs)\n    for tag, tau, towers in configs:\n        res[tag] = {}\n        for seed in seeds:\n            t0 = time.time()\n            torch.manual_seed(seed)\n            m, states, loras = build(lora=True, towers=towers)\n            head = nn.Linear(1024, D[\"C\"]).to(DEV)\n            heads = {b: _heads(D[\"C\"], b) for b in BETAS}\n            trans_heads = _heads(D[\"C\"], 0.5)\n            stored, ranks, seen = [], [], torch.zeros(D[\"C\"], dtype=torch.bool)\n            acc_matrix = {f\"beta_{b}\": [] for b in BETAS}      # row k: acc on val of task j<=k after session k\n            for k, task in enumerate(D[\"order\"]):\n                idx = _idx(D[\"train\"] & (D[\"task\"] == task))\n                fi_pre, ft_pre = extract(m, D, idx) if k > 0 else (None, None)\n                train_session(m, loras, head, D, idx, epochs=epochs, bs=bs, lr=lr, seed=seed * 100 + k)\n                for st in states:\n                    st.recording = True\n                fi, ft = extract(m, D, idx)\n                for st in states:\n                    st.recording = False\n                    st.update_projection(tau)\n                stored.append((idx, fi, ft))\n                seen[D[\"y\"][idx].unique()] = True\n                for hs in heads.values():\n                    for h in hs:\n                        h.update({\"img\": fi, \"txt\": ft}, D[\"y\"][idx])\n                for h in trans_heads:\n                    if k > 0 and fi_pre is not None:\n                        h.transport_gram({\"img\": fi_pre, \"txt\": ft_pre}, {\"img\": fi, \"txt\": ft})\n                    h.update({\"img\": fi, \"txt\": ft}, D[\"y\"][idx])\n                vi, vt = extract(m, D, va)\n                for b, hs in heads.items():\n                    lg = _logits(hs, (vi, vt))\n                    lg[:, ~seen] = -float(\"inf\")\n                    pred = lg.argmax(1).numpy()\n                    acc_matrix[f\"beta_{b}\"].append([float((pred[tv == t] == yv[tv == t]).mean()) for t in D[\"order\"][:k + 1]])\n                ranks.append(float(np.mean([st.rank / st.gram.shape[0] for st in states])) if tau else 0.0)\n                log(f\"[{stage}/{tag}/seed{seed}] session {k + 1}/{len(D['order'])} n={len(idx)} \"\n                    f\"protected-rank {ranks[-1]:.3f} acc(b=0) {np.mean(acc_matrix['beta_0.0'][-1]):.4f} ({time.time() - t0:.0f}s)\")\n            r = {\"stale\": {}, \"transport_sid\": {}, \"acc_matrix\": acc_matrix, \"protected_rank_frac\": ranks, \"towers\": list(towers)}\n            ti, tt = extract(m, D, te) if has_te else (None, None)\n            for b, hs in heads.items():\n                pred = _logits(hs, (vi, vt)).argmax(1).numpy()\n                PRED[f\"{stage}/{tag}/seed{seed}/stale/val/beta{b}\"] = pred.astype(np.int16)\n                r[\"stale\"][f\"beta_{b}\"] = scores(yv, pred)\n                if has_te:\n                    pred_te = _logits(hs, (ti, tt)).argmax(1).numpy()\n                    PRED[f\"{stage}/{tag}/seed{seed}/stale/test/beta{b}\"] = pred_te.astype(np.int16)\n                    r[\"stale\"][f\"test_beta_{b}\"] = scores(yte, pred_te)\n            for h in trans_heads:\n                h.auto_spectral_ridges()\n            pred_tr_val = _logits(trans_heads, (vi, vt), adaptive_sid=True).argmax(1).numpy()\n            r[\"transport_sid\"][\"val\"] = scores(yv, pred_tr_val)\n            if has_te:\n                pred_tr_te = _logits(trans_heads, (ti, tt), adaptive_sid=True).argmax(1).numpy()\n                r[\"transport_sid\"][\"test\"] = scores(yte, pred_tr_te)\n            all_idx = torch.cat([s[0] for s in stored])\n            fi, ft = extract(m, D, all_idx)\n            si, st_ = torch.cat([s[1] for s in stored]), torch.cat([s[2] for s in stored])\n            di, dt = (fi - si).norm(dim=1), (ft - st_).norm(dim=1)\n            r[\"drift_img\"], r[\"drift_txt\"] = float(di.mean()), float(dt.mean())\n            bounds = np.cumsum([0] + [len(s[0]) for s in stored])\n            r[\"drift_by_session\"] = [{\"img\": float(di[a:b].mean()), \"txt\": float(dt[a:b].mean())}\n                                     for a, b in zip(bounds[:-1], bounds[1:])]\n            r[\"refit\"] = analytic_fit_eval((fi, ft), D[\"y\"][all_idx], (vi, vt), D[\"y\"][va], D[\"C\"],\n                                           f\"{stage}/{tag}/seed{seed}/refit\", (ti, tt) if has_te else None, D[\"y\"][te] if has_te else None)\n            r[\"seconds\"] = time.time() - t0\n            log(f\"[{stage}/{tag}/seed{seed}] stale {r['stale']} | refit {r['refit']} | \"\n                f\"drift img {r['drift_img']:.4f} txt {r['drift_txt']:.4f}\")\n            res[tag][f\"seed{seed}\"] = r\n            del m\n            if DEV == \"cuda\":\n                import gc\n                gc.collect()\n                torch.cuda.empty_cache()\n    return res\n\n\ndef _mean_over_seeds(per_seed, metric, field=None, prefix=\"beta\"):\n    \"\"\"Best beta (by seed-mean) of the seed-mean metric.\"\"\"\n    rows = [v[field] if field else v for v in per_seed.values()]\n    return max(float(np.mean([r[f\"{prefix}_{b}\"][metric] for r in rows if f\"{prefix}_{b}\" in r])) for b in BETAS)\n\n\ndef verdicts(g1, g2, metric):\n    fz, orc = _mean_over_seeds(g1[\"frozen\"], metric), _mean_over_seeds(g1[\"joint_oracle\"], metric)\n    out = {\"metric\": metric, \"frozen\": fz, \"joint_oracle\": orc,\n           \"first_session\": _mean_over_seeds(g1[\"first_session\"], metric),\n           \"G1_oracle_gain_pts\": 100 * (orc - fz), \"G1\": \"PASS\" if 100 * (orc - fz) >= 2 else \"FAIL\"}\n    if g2:\n        target = fz + 0.5 * (orc - fz)\n        stale = {k: _mean_over_seeds(v, metric, \"stale\") for k, v in g2.items()}\n        trans_sid = {k: float(np.mean([s[\"transport_sid\"][\"val\"][metric] for s in v.values()]))\n                     for k, v in g2.items() if \"transport_sid\" in next(iter(v.values()))}\n        cand = [v for k, v in stale.items() if k != \"naive\"] + [v for k, v in trans_sid.items() if k != \"naive\"]\n        out.update({\"G2_target\": target, \"G2_stale\": stale, \"G2_transport_sid\": trans_sid,\n                    \"G2\": \"PASS\" if max(cand) >= target else \"FAIL\"})\n        first_seed = next(iter(next(iter(g2.values())).values()))\n        if \"test_beta_0.5\" in first_seed.get(\"stale\", {}):\n            out[\"G3_test_frozen\"] = _mean_over_seeds(g1[\"frozen\"], metric, prefix=\"test_beta\")\n            out[\"G3_test_first_session\"] = _mean_over_seeds(g1[\"first_session\"], metric, prefix=\"test_beta\")\n            out[\"G3_test_joint_oracle\"] = _mean_over_seeds(g1[\"joint_oracle\"], metric, prefix=\"test_beta\")\n            out[\"G3_test_stale\"] = {k: _mean_over_seeds(v, metric, \"stale\", prefix=\"test_beta\") for k, v in g2.items()}\n            out[\"G3_test_transport_sid\"] = {\n                k: float(np.mean([s[\"transport_sid\"][\"test\"][metric] for s in v.values()]))\n                for k, v in g2.items() if \"test\" in next(iter(v.values())).get(\"transport_sid\", {})\n            }\n    return out\n\n\ndef export(out_dir, name, D, payload):\n    \"\"\"results JSON + per-sample validation/test predictions (for paired bootstrap offline).\"\"\"\n    import json\n    import platform\n    out_dir.mkdir(parents=True, exist_ok=True)\n    va = np.where(D[\"val\"])[0]\n    te = np.where(D[\"test\"])[0] if D.get(\"test\") is not None else np.array([], dtype=int)\n    meta = {\"torch\": torch.__version__, \"python\": platform.python_version(), \"device\": DEV,\n            \"gpu\": torch.cuda.get_device_name(0) if DEV == \"cuda\" else \"cpu\", \"amp\": str(AMP),\n            \"n_train\": int(D[\"train\"].sum()), \"n_val\": int(len(va)), \"n_test\": int(len(te)), \"num_classes\": D[\"C\"],\n            \"task_order\": [str(t) for t in D[\"order\"]], \"head_seeds\": list(HEAD_SEEDS), \"betas\": list(BETAS),\n            \"lora\": {\"r\": R, \"alpha\": ALPHA, \"targets\": [\"q_proj\", \"v_proj\"], \"towers\": [\"vision\", \"text\"]}}\n    (out_dir / f\"{name}_results.json\").write_text(json.dumps({\"meta\": meta, **payload}, indent=2))\n    keep = {k.replace(\"/\", \"__\"): v for k, v in PRED.items() if k.split(\"/\")[0] in payload[\"stages\"]}\n    extra = {}\n    if len(te) > 0:\n        extra[\"y_test\"] = D[\"y\"][te].numpy().astype(np.int16)\n        extra[\"task_test\"] = np.array([str(t) for t in D[\"task\"][te]])\n    np.savez_compressed(out_dir / f\"{name}_predictions.npz\", y_val=D[\"y\"][va].numpy().astype(np.int16),\n                        task_val=np.array([str(t) for t in D[\"task\"][va]]), **extra, **keep)\n", "external_repos/DeLo/src/model/mixlora_layer.py": "# # \u6587\u4ef6: src/model/mixlora_layer.py (\u6700\u7ec8\u4fee\u590d\u7248 - \u56de\u5f52\u7269\u7406\u5206\u533a)\n\n# import torch\n# import torch.nn as nn\n# import torch.nn.functional as F\n# from loguru import logger\n\n# class MixLoRALayer(nn.Module):\n#     def __init__(self, in_features: int, out_features: int, r: int, E: int, num_tasks: int, alpha: float = 1.0):\n#         super().__init__()\n#         self.in_features = in_features\n#         self.out_features = out_features\n#         self.r = r\n#         self.E = E\n#         self.num_tasks = num_tasks\n#         self.alpha = alpha\n#         self.current_task_id = 0\n\n#         # ======================= \u6838\u5fc3\u4fee\u6539\uff1a\u56de\u5f52\u7269\u7406\u5206\u533a\u67b6\u6784 =======================\n#         # \u4e3a\u6bcf\u4e2a\u4efb\u52a1\u521b\u5efa\u72ec\u7acb\u7684\u3001\u7269\u7406\u9694\u79bb\u7684LoRA A/B\u56e0\u5b50\u6c60\n#         # \u6bcf\u4e2a\u6c60\u7684\u5927\u5c0f\u90fd\u662f E\n#         self.lora_A_pools = nn.ParameterList([\n#             nn.Parameter(torch.randn(E, r, self.in_features)) for _ in range(num_tasks)\n#         ])\n#         self.lora_B_pools = nn.ParameterList([\n#             nn.Parameter(torch.zeros(E, self.out_features, r)) for _ in range(num_tasks)\n#         ])\n        \n#         # \u4e3a\u6bcf\u4e2a\u4efb\u52a1\u521b\u5efa\u72ec\u7acb\u7684\u8def\u7531\u5668\n#         self.ifs_router_A_list = nn.ModuleList([nn.Linear(self.in_features, E) for _ in range(num_tasks)])\n#         self.ifs_router_B_list = nn.ModuleList([nn.Linear(self.in_features, E) for _ in range(num_tasks)])\n#         self.cfs_router_B_weights_list = nn.ParameterList([\n#             nn.Parameter(torch.randn(r, self.in_features, E)) for _ in range(num_tasks)\n#         ])\n#         # =======================================================================\n        \n#         # Query-Key \u8bb0\u5fc6\u6a21\u5757\u4fdd\u6301\u4e0d\u53d8\n#         self.register_buffer('task_keys', torch.zeros(num_tasks, self.in_features))\n#         self.ema_momentum = 0.99\n\n#     # \u6062\u590d set_active_task \u65b9\u6cd5\n#     def set_active_task(self, task_id: int):\n#         self.current_task_id = task_id\n#         for i in range(self.num_tasks):\n#             is_active = (i == task_id)\n#             # \u51bb\u7ed3/\u89e3\u51bb\u6240\u6709\u4e0e\u4efb\u52a1\u76f8\u5173\u7684\u53c2\u6570\n#             self.lora_A_pools[i].requires_grad = is_active\n#             self.lora_B_pools[i].requires_grad = is_active\n#             self.cfs_router_B_weights_list[i].requires_grad = is_active\n#             for param in self.ifs_router_A_list[i].parameters():\n#                 param.requires_grad = is_active\n#             for param in self.ifs_router_B_list[i].parameters():\n#                 param.requires_grad = is_active\n\n#     @torch.no_grad()\n#     def _update_task_key(self, query_signal: torch.Tensor):\n#         # \u6b64\u65b9\u6cd5\u4e0d\u53d8\n#         batch_key = query_signal.mean(dim=0)\n#         old_key = self.task_keys[self.current_task_id]\n#         self.task_keys[self.current_task_id] = self.ema_momentum * old_key + (1 - self.ema_momentum) * batch_key\n\n#     def forward(self, x: torch.Tensor, query_signal: torch.Tensor) -> torch.Tensor:\n#         # forward\u7684\u903b\u8f91\u73b0\u5728\u53d8\u5f97\u66f4\u7b80\u5355\uff0c\u56e0\u4e3a\u5b83\u4f9d\u8d56\u4e8e set_active_task\n#         task_id_to_use = self.current_task_id\n#         force_task_id = getattr(x, 'force_task_id', None)\n\n#         if not self.training: # \u8bc4\u4f30\u6a21\u5f0f\n#             if force_task_id is not None:\n#                 task_id_to_use = force_task_id # \u5f3a\u5236\u4f7f\u7528\u6307\u5b9a\u7684\u4e13\u5bb6\n#             else: # \u81ea\u52a8\u5339\u914d\n#                 with torch.no_grad():\n#                     task_keys_on_device = self.task_keys.to(query_signal.device)\n#                     similarities = F.cosine_similarity(query_signal.unsqueeze(1), task_keys_on_device.unsqueeze(0), dim=-1)\n#                     best_task_ids = torch.argmax(similarities, dim=1)\n#                     task_id_to_use = torch.mode(best_task_ids).values.item()\n#         elif self.training:\n#              self._update_task_key(query_signal.detach())\n        \n#         # \u6839\u636e\u51b3\u7b56\u51fa\u7684 task_id_to_use\uff0c\u9009\u62e9\u6fc0\u6d3b\u7684\u7ec4\u4ef6\n#         active_A_pool = self.lora_A_pools[task_id_to_use]\n#         active_B_pool = self.lora_B_pools[task_id_to_use]\n#         active_ifs_router_A = self.ifs_router_A_list[task_id_to_use]\n#         active_ifs_router_B = self.ifs_router_B_list[task_id_to_use]\n#         active_cfs_router_B_weights = self.cfs_router_B_weights_list[task_id_to_use]\n\n#         # \u540e\u7eed\u7684\u8def\u7531\u548c\u8ba1\u7b97\u903b\u8f91\u4e0e\u4e4b\u524d\u5b8c\u5168\u76f8\u540c\n#         g_A_scores = active_ifs_router_A(query_signal)\n#         g_A_indices = torch.topk(g_A_scores, self.r, dim=-1).indices\n#         lora_A = active_A_pool[g_A_indices]\n\n#         g_B_scores_ifs = active_ifs_router_B(query_signal)\n#         # \u6ce8\u610f\uff1a\u8fd9\u91cc\u7684cfs\u6743\u91cd\u7ef4\u5ea6\u5df2\u7ecf\u9002\u914d\u4e86\u7269\u7406\u5206\u533a\u6a21\u5f0f\n#         g_B_scores_cfs = torch.einsum('bri,rie->be', lora_A.detach(), active_cfs_router_B_weights).sum(dim=1)\n        \n#         g_B_scores = g_B_scores_ifs + g_B_scores_cfs\n#         g_B_indices = torch.topk(g_B_scores, self.r, dim=-1).indices\n#         lora_B = active_B_pool[g_B_indices]\n        \n#         after_A = torch.einsum('bsi,bri->bsr', x, lora_A)\n#         lora_delta = torch.einsum('bsr,bor->bso', after_A, lora_B)\n        \n#         return lora_delta * self.alpha\n\n\n# \u6587\u4ef61: src/model/mixlora_layer.py (\u4fee\u590d\u7248)\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom loguru import logger\n\nclass MixLoRALayer(nn.Module):\n    def __init__(self, in_features: int, out_features: int, r: int, E: int, num_tasks: int, alpha: float = 1.0):\n        super().__init__()\n        self.in_features = in_features\n        self.out_features = out_features\n        self.r = r\n        self.E = E\n        self.num_tasks = num_tasks\n        self.alpha = alpha\n        self.current_task_id = 0\n\n        # \u4e3a\u6bcf\u4e2a\u4efb\u52a1\u521b\u5efa\u72ec\u7acb\u7684\u3001\u7269\u7406\u9694\u79bb\u7684LoRA A/B\u56e0\u5b50\u6c60\n        self.lora_A_pools = nn.ParameterList([\n            nn.Parameter(torch.randn(E, r, self.in_features)) for _ in range(num_tasks)\n        ])\n        self.lora_B_pools = nn.ParameterList([\n            nn.Parameter(torch.zeros(E, self.out_features, r)) for _ in range(num_tasks)\n        ])\n        \n        # \u4e3a\u6bcf\u4e2a\u4efb\u52a1\u521b\u5efa\u72ec\u7acb\u7684\u8def\u7531\u5668\n        self.ifs_router_A_list = nn.ModuleList([nn.Linear(self.in_features, E) for _ in range(num_tasks)])\n        self.ifs_router_B_list = nn.ModuleList([nn.Linear(self.in_features, E) for _ in range(num_tasks)])\n        self.cfs_router_B_weights_list = nn.ParameterList([\n            nn.Parameter(torch.randn(r, self.in_features, E)) for _ in range(num_tasks)\n        ])\n        \n        # Query-Key \u8bb0\u5fc6\u6a21\u5757\n        self.register_buffer('task_keys', torch.zeros(num_tasks, self.in_features))\n        self.ema_momentum = 0.99\n\n        # \ud83d\udd27 \u4fee\u590d1: \u6dfb\u52a0\u8c03\u8bd5\u6807\u8bb0\n        self.debug_name = \"unnamed_layer\"\n        \n        # \ud83d\udd27 \u4fee\u590d2: \u5728\u521d\u59cb\u5316\u65f6\u5c31\u8bbe\u7f6eTask 0\u4e3a\u6fc0\u6d3b\u72b6\u6001\n        self.set_active_task(0)\n\n    def set_debug_name(self, name: str):\n        \"\"\"\u8bbe\u7f6e\u8c03\u8bd5\u540d\u79f0\uff0c\u65b9\u4fbf\u8ffd\u8e2a\u95ee\u9898\"\"\"\n        self.debug_name = name\n\n    def set_active_task(self, task_id: int):\n        \"\"\"\ud83d\udd27 \u4fee\u590d: \u4fee\u590d sum() \u51fd\u6570\u7684\u53c2\u6570\u9519\u8bef\"\"\"\n        old_task_id = self.current_task_id\n        self.current_task_id = task_id\n        \n        for i in range(self.num_tasks):\n            is_active = (i == task_id)\n            \n            # \u51bb\u7ed3/\u6fc0\u6d3bLoRA\u56e0\u5b50\u6c60\n            self.lora_A_pools[i].requires_grad = is_active\n            self.lora_B_pools[i].requires_grad = is_active\n            self.cfs_router_B_weights_list[i].requires_grad = is_active\n            \n            # \u51bb\u7ed3/\u6fc0\u6d3b\u8def\u7531\u5668\u53c2\u6570\n            for param in self.ifs_router_A_list[i].parameters():\n                param.requires_grad = is_active\n            for param in self.ifs_router_B_list[i].parameters():\n                param.requires_grad = is_active\n\n        # \ud83d\udd27 \u4fee\u590d: \u53ea\u6709\u5728\u7b2c0\u5c42\u7684query\u7684vision\u9002\u914d\u5668\u65f6\u624d\u6253\u5370\u8be6\u7ec6\u4fe1\u606f\n        if self.debug_name == \"layer_0_query_vision\":\n            # \ud83d\udd27 \u4fee\u590d: \u5c06\u8ba1\u7b97\u8fc7\u7a0b\u5206\u6b65\u8fdb\u884c\uff0c\u907f\u514d sum() \u51fd\u6570\u7684\u53c2\u6570\u9519\u8bef\n            \n            # \u8ba1\u7b97\u5f53\u524d\u4efb\u52a1\u7684\u6fc0\u6d3b\u53c2\u6570\u6570\u91cf\n            current_task_lora_params = (\n                self.lora_A_pools[task_id].numel() + \n                self.lora_B_pools[task_id].numel() +\n                self.cfs_router_B_weights_list[task_id].numel()\n            )\n            \n            current_task_router_params = 0\n            for param in self.ifs_router_A_list[task_id].parameters():\n                current_task_router_params += param.numel()\n            for param in self.ifs_router_B_list[task_id].parameters():\n                current_task_router_params += param.numel()\n                \n            total_activated_params = current_task_lora_params + current_task_router_params\n            \n            # \u8ba1\u7b97\u5176\u4ed6\u4efb\u52a1\u7684\u51bb\u7ed3\u53c2\u6570\u6570\u91cf\n            total_frozen_params = 0\n            for i in range(self.num_tasks):\n                if i != task_id:\n                    frozen_lora_params = (\n                        self.lora_A_pools[i].numel() + \n                        self.lora_B_pools[i].numel() +\n                        self.cfs_router_B_weights_list[i].numel()\n                    )\n                    \n                    frozen_router_params = 0\n                    for param in self.ifs_router_A_list[i].parameters():\n                        frozen_router_params += param.numel()\n                    for param in self.ifs_router_B_list[i].parameters():\n                        frozen_router_params += param.numel()\n                        \n                    total_frozen_params += frozen_lora_params + frozen_router_params\n            \n            logger.info(f\"[\u4ee3\u8868\u6027\u9002\u914d\u5668] \u4efb\u52a1\u5207\u6362: {old_task_id} -> {task_id}, \"\n                       f\"\u6fc0\u6d3b\u53c2\u6570: {total_activated_params}, \u51bb\u7ed3\u53c2\u6570: {total_frozen_params}\")\n\n\n    @torch.no_grad()\n    def _update_task_key(self, query_signal: torch.Tensor):\n        \"\"\"\u786e\u4fddquery_signal\u7684\u7ef4\u5ea6\u5904\u7406\u6b63\u786e\"\"\"\n        try:\n            batch_key = query_signal.mean(dim=0)\n            old_key = self.task_keys[self.current_task_id]\n            self.task_keys[self.current_task_id] = self.ema_momentum * old_key + (1 - self.ema_momentum) * batch_key\n        except Exception as e:\n            logger.error(f\"Task key\u66f4\u65b0\u9519\u8bef: {e}\")\n            logger.error(f\"query_signal\u5f62\u72b6: {query_signal.shape}\")\n\n    def forward(self, x: torch.Tensor, query_signal: torch.Tensor) -> torch.Tensor:\n        task_id_to_use = self.current_task_id\n        force_task_id = getattr(x, 'force_task_id', None)\n\n        if not self.training:  # \u8bc4\u4f30\u6a21\u5f0f\n            if force_task_id is not None:\n                task_id_to_use = force_task_id\n            else:  # \u81ea\u52a8\u5339\u914d\u6700\u4f73\u4efb\u52a1\n                with torch.no_grad():\n                    task_keys_on_device = self.task_keys.to(query_signal.device)\n                    similarities = F.cosine_similarity(\n                        query_signal.unsqueeze(1), \n                        task_keys_on_device.unsqueeze(0), \n                        dim=-1\n                    )\n                    best_task_ids = torch.argmax(similarities, dim=1)\n                    task_id_to_use = torch.mode(best_task_ids).values.item()\n        elif self.training:\n            # \u786e\u4fdd\u5728\u8bad\u7ec3\u65f6\u4f7f\u7528\u6b63\u786e\u7684\u4efb\u52a1ID\n            assert task_id_to_use == self.current_task_id, \\\n                f\"\u8bad\u7ec3\u65f6\u4efb\u52a1ID\u4e0d\u5339\u914d: current={self.current_task_id}, using={task_id_to_use}\"\n            self._update_task_key(query_signal.detach())\n        \n        # \u9009\u62e9\u6fc0\u6d3b\u7684\u7ec4\u4ef6\n        active_A_pool = self.lora_A_pools[task_id_to_use]\n        active_B_pool = self.lora_B_pools[task_id_to_use]\n        active_ifs_router_A = self.ifs_router_A_list[task_id_to_use]\n        active_ifs_router_B = self.ifs_router_B_list[task_id_to_use]\n        active_cfs_router_B_weights = self.cfs_router_B_weights_list[task_id_to_use]\n\n        # \u8def\u7531\u548c\u8ba1\u7b97\u903b\u8f91\n        try:\n            g_A_scores = active_ifs_router_A(query_signal)\n            g_A_indices = torch.topk(g_A_scores, self.r, dim=-1).indices\n            lora_A = active_A_pool[g_A_indices]\n\n            g_B_scores_ifs = active_ifs_router_B(query_signal)\n            g_B_scores_cfs = torch.einsum('bri,rie->be', lora_A.detach(), active_cfs_router_B_weights).sum(dim=1)\n            \n            g_B_scores = g_B_scores_ifs + g_B_scores_cfs\n            g_B_indices = torch.topk(g_B_scores, self.r, dim=-1).indices\n            lora_B = active_B_pool[g_B_indices]\n            \n            after_A = torch.einsum('bsi,bri->bsr', x, lora_A)\n            lora_delta = torch.einsum('bsr,bor->bso', after_A, lora_B)\n            \n            return lora_delta * self.alpha\n            \n        except Exception as e:\n            logger.error(f\"MixLoRA forward\u8ba1\u7b97\u9519\u8bef: {e}\")\n            logger.error(f\"\u8f93\u5165\u5f62\u72b6: x={x.shape}, query_signal={query_signal.shape}\")\n            logger.error(f\"\u4efb\u52a1ID: {task_id_to_use}, \u5f53\u524d\u4efb\u52a1: {self.current_task_id}\")\n            raise e", "external_repos/DeLo/src/optim/criterion.py": "\n\n# \u6587\u4ef6: src/optim/criterion.py (\u6700\u7ec8\u8bca\u65ad\u4fee\u590d\u7248)\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom loguru import logger # \u5bfc\u5165logger\u4ee5\u4fbf\u6253\u5370\n\ndef build_criterion(cfg):\n    criterion_name = cfg.NAME.lower()\n    if criterion_name == \"rebq-mixlora\":\n        return RebQMixLoRALoss(cfg)\n    elif criterion_name == \"ce\":\n        return nn.CrossEntropyLoss()\n    elif criterion_name == \"bce\":\n        return nn.BCEWithLogitsLoss()\n    else:\n        raise ValueError(f\"Unknown criterion name: {cfg.NAME}\")\n\nclass RebQMixLoRALoss(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.dataset_name = cfg.DATASET_NAME.lower()\n        self.lambda_align = cfg.get(\"LAMBDA_ALIGN\", 0.1)\n        self.lambda_consistency = cfg.get(\"LAMBDA_CONSISTENCY\", 0.1)\n\n        if self.dataset_name == \"upmc_food101_cmml\":\n            self.classification_loss = nn.CrossEntropyLoss()\n        elif self.dataset_name == \"mm_imdb_cmml\":\n            self.classification_loss = nn.BCEWithLogitsLoss()\n        else:\n            raise ValueError(f\"Unknown dataset name for criterion: {self.dataset_name}\")\n            \n        self.consistency_loss = nn.KLDivLoss(reduction='batchmean')\n        self.alignment_loss = nn.CosineEmbeddingLoss()\n\n    def forward(self, model_outputs):\n        logits = model_outputs[\"logits\"]\n        labels = model_outputs[\"labels\"]\n        L_c = self.classification_loss(logits, labels)\n\n        L_align = torch.tensor(0.0, device=logits.device)\n        L_consistency = torch.tensor(0.0, device=logits.device)\n\n        if 'query_signals' in model_outputs and 'complete_indices' in model_outputs:\n            complete_indices = model_outputs[\"complete_indices\"]\n            \n            if len(complete_indices) > 0:\n                # --- \u5bf9\u9f50\u635f\u5931 L_align ---\n                query_signals = model_outputs[\"query_signals\"]\n                q_v = query_signals['vision']\n                q_t = query_signals['text']\n                \n                q_v_complete = q_v[complete_indices]\n                q_t_complete = q_t[complete_indices]\n\n                # ==================== \u6700\u7ec8\u63a2\u9488\u4e0e\u4fee\u590d ====================\n                # \u4f7f\u7528.view(-1, 768)\u6765\u5f3a\u5236\u786e\u4fdd\u5f20\u91cf\u662f2D\u7684\uff0c\u5373\u4f7f\u53ea\u6709\u4e00\u4e2a\u6837\u672c\n                # 768\u662fViLT\u7684hidden_size\n                hidden_dim = q_v_complete.shape[-1]\n                q_v_complete = q_v_complete.view(-1, hidden_dim)\n                q_t_complete = q_t_complete.view(-1, hidden_dim)\n\n                # logger.info(f\"Alignment Loss Input Shapes: q_v={q_v_complete.shape}, q_t={q_t_complete.shape}\")\n                # ========================================================\n                \n                target = torch.ones(q_v_complete.size(0)).to(logits.device)\n                L_align = self.alignment_loss(q_v_complete, q_t_complete, target)\n\n                # --- \u4e00\u81f4\u6027\u635f\u5931 L_consistency ---\n                if 'consistency_logits' in model_outputs:\n                    consistency_logits = model_outputs[\"consistency_logits\"]\n                    original_logits_for_complete = logits[complete_indices]\n\n                    if original_logits_for_complete.shape[0] == consistency_logits.shape[0]:\n                        log_p_original = F.log_softmax(original_logits_for_complete, dim=-1)\n                        p_consistency = F.softmax(consistency_logits, dim=-1)\n                        L_consistency = self.consistency_loss(log_p_original, p_consistency)\n        \n        total_loss = L_c + self.lambda_align * L_align + self.lambda_consistency * L_consistency\n        \n        return {\n            \"total_loss\": total_loss,\n            \"classification_loss\": L_c,\n            \"alignment_loss\": L_align,\n            \"consistency_loss\": L_consistency\n        }", "external_repos/task_singular_vectors/src/utils/TSVM_utils.py": "import torch\nimport math\n\n\ndef compute_svd_dict(task_vectors, config):\n    \"\"\"\n    Computes the Singular Value Decomposition (SVD) for each task vector in the provided datasets and stores the results in a dictionary.\n\n    Args:\n        task_vectors (list): A list of task vector objects, where each task vector contains a dictionary of matrices to be decomposed.\n        config (object): Configuration object containing the list of datasets under the attribute `DATASETS`.\n\n    Returns:\n        dict: A dictionary where each key is a dataset name and the value is another dictionary containing the SVD components ('u', 's', 'v') for each matrix in the task vector.\n        If a matrix is not 2-dimensional or contains 'text_projection' in its key, it is stored under the key 'dim1' without decomposition.\n    \"\"\"\n    sv_reduction = 1 / len(config.DATASETS)\n    with torch.no_grad():\n        svd_dict = {}\n        for i, (task_vector, dataset) in enumerate(zip(task_vectors, config.DATASETS)):\n            svd_dict[dataset] = {}\n            print(f\"Computing SVD for {dataset}...\")\n            for key in task_vector.vector:\n                svd_dict[dataset][key] = {}\n                if (\n                    len(task_vector.vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n                    u, s, v = torch.linalg.svd(\n                        task_vector.vector[key], full_matrices=False\n                    )\n                    reduced_index_s = int(s.shape[0] * sv_reduction)\n\n                    temp_u = torch.zeros_like(u)\n                    # select only the first reduced_index_s columns of u and place them\n                    temp_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                        :, :reduced_index_s\n                    ]\n                    svd_dict[dataset][key][\"u\"] = temp_u\n\n                    temp_s = torch.zeros_like(s)\n                    temp_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                        :reduced_index_s\n                    ]\n\n                    svd_dict[dataset][key][\"s\"] = temp_s  # s_reduced\n\n                    # select only the first reduced_index_s rows of v and place them\n                    temp_v = torch.zeros_like(v)\n                    temp_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                        :reduced_index_s, :\n                    ]\n\n                    svd_dict[dataset][key][\"v\"] = temp_v\n\n                    # temp_mat = temp_u @ torch.diag_embed(temp_s) @ temp_v\n                else:\n                    svd_dict[dataset][key][\"dim1\"] = task_vector.vector[key]\n    return svd_dict\n\n\ndef sum_svd_dict(svd_dict, config):\n    \"\"\"\n    Sums the Singular Value Decomposition (SVD) components from multiple datasets and computes a new vector.\n\n    Args:\n        svd_dict (dict): A dictionary containing SVD components for multiple datasets. The structure of the dictionary is expected to be:\n                         {\n                             dataset_name: {\n                                 key: {\n                                     \"u\": tensor,\n                                     \"s\": tensor,\n                                     \"v\": tensor,\n                                     \"dim1\": tensor (optional)\n                                 }\n                             }\n                         }\n        config (object): A configuration object that contains a list of dataset names under the attribute `DATASETS`.\n\n    Returns:\n        dict: A dictionary containing the merged SVD components or averaged \"dim1\" values for each key.\n    \"\"\"\n    print(\"Summing SVD...\")\n    new_vector = {}\n    for key in svd_dict[config.DATASETS[0]]:\n        if \"u\" in svd_dict[config.DATASETS[0]][key].keys():\n            sum_u = sum([svd_dict[dataset][key][\"u\"] for dataset in config.DATASETS])\n            sum_s = sum([svd_dict[dataset][key][\"s\"] for dataset in config.DATASETS])\n            sum_v = sum([svd_dict[dataset][key][\"v\"] for dataset in config.DATASETS])\n\n            u_u, s_u, v_u = torch.linalg.svd(sum_u, full_matrices=False)\n            u_v, s_v, v_v = torch.linalg.svd(sum_v, full_matrices=False)\n            new_vector[key] = torch.linalg.multi_dot(\n                (\n                    u_u,\n                    v_u,\n                    torch.diag(sum_s),\n                    u_v,\n                    v_v,\n                )\n            )\n        else:\n            for i, dataset in enumerate(config.DATASETS, start=1):\n                if i == 1:\n                    new_vector[key] = svd_dict[dataset][key][\"dim1\"]\n                else:\n                    new_vector[key] += (\n                        svd_dict[dataset][key][\"dim1\"] - new_vector[key]\n                    ) / i\n    return new_vector\n\n\n###############\n##### LOSSLESS Orthogonalization\ndef compute_and_sum_svd_mem_reduction_lossless(task_vectors, config):\n    \"\"\"\n    Computes the Singular Value Decomposition (SVD) for each task vector and merge the results.\n\n    This function performs the following steps:\n    1. Iterates over each layer in the task vectors.\n    2. For each layer, it computes the SVD of the corresponding matrix if it is a 2D tensor excluding \"text_projection\".\n    3. Concatenate the U_i, S_i, and V_i matrices from the SVD across all tasks.\n    4. If the vector is not a 2D tensor or is \"text_projection\", it computes the mean of the vectors.\n    5. After concatenating the SVD components, recomputes the SVD of the summed U and V matrices and constructs the merged layer.\n\n    Args:\n        task_vectors (list): A list of task vectors, where each task vector is a dictionary containing the vectors for each task.\n        config (object): A configuration object containing the device and dataset information.\n\n    Returns:\n        dict: A dictionary containing the new vectors after summing the SVD components.\n    \"\"\"\n    # becareful wit vit-l on 20 task it does not fit in GPU or in 64 GB RAM (try without last layer)\n    device = config.device\n    print(\"Computing SVD...\")\n    with torch.no_grad():\n        new_vector = {}\n        for key in task_vectors[0].vector:\n            new_vector[key] = {}\n            for i, (task_vector, dataset) in enumerate(\n                zip(task_vectors, config.DATASETS)\n            ):\n                vec = task_vector.vector[key].to(device)\n\n                if (\n                    len(task_vector.vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n\n                    u, s, v = torch.linalg.svd(vec, full_matrices=False)\n\n                    if i == 0:\n                        print(f\"Computed SVD for {key}...\")\n                        sum_u = torch.zeros(\n                            u.shape[0], u.shape[1] * config.num_tasks, device=device\n                        )\n                        sum_s = torch.zeros(\n                            s.shape[0] * config.num_tasks, device=device\n                        )\n                        sum_v = torch.zeros(\n                            v.shape[0] * config.num_tasks, v.shape[1], device=device\n                        )\n                    reduced_index_s = s.shape[0]\n\n                    # select only the first reduced_index_s columns of u and place them\n                    sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                        :, :reduced_index_s\n                    ]\n                    sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                        :reduced_index_s\n                    ]\n                    # select only the first reduced_index_s rows of v and place them\n                    sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                        :reduced_index_s, :\n                    ]\n\n                else:\n                    if i == 0:\n                        new_vector[key] = vec.clone()\n                    else:\n                        new_vector[key] += (vec - new_vector[key]) / (i + 1)\n\n            if len(task_vector.vector[key].shape) == 2 and \"text_projection\" not in key:\n                u_u, s_u, v_u = torch.linalg.svd(sum_u, full_matrices=False)\n                u_v, s_v, v_v = torch.linalg.svd(sum_v, full_matrices=False)\n\n                new_vector[key] = torch.linalg.multi_dot(\n                    (\n                        u_u,\n                        v_u,\n                        torch.diag(sum_s),\n                        u_v,\n                        v_v,\n                    )\n                )\n\n    return new_vector\n\n\n###############\n##### LOSSLESS EIGENDECOMP\ndef compute_and_sum_svd_mem_reduction_lossless_eigen(task_vectors, config):\n    \"\"\"\n    Computes the Singular Value Decomposition (SVD) for each task vector and merge the results.\n\n    This function performs the following steps:\n    1. Iterates over each layer in the task vectors.\n    2. For each layer, it computes the SVD of the corresponding matrix if it is a 2D tensor excluding \"text_projection\".\n    3. Concatenate the U_i, S_i, and V_i matrices from the SVD across all tasks.\n    4. If the vector is not a 2D tensor or is \"text_projection\", it computes the mean of the vectors.\n    5. After concatenating the SVD components, recomputes the eigendecomposition of the summed U and V matrices and constructs the merged layer.\n\n    Args:\n        task_vectors (list): A list of task vectors, where each task vector is a dictionary containing the vectors for each task.\n        config (object): A configuration object containing the device and dataset information.\n\n    Returns:\n        dict: A dictionary containing the new vectors after merging the SVD components.\n    \"\"\"\n    # becareful wit vit-l on 20 task it does not fit in GPU or in 64 GB RAM (try without last layer)\n    device = config.device\n    print(\"Computing SVD...\")\n    with torch.no_grad():\n        new_vector = {}\n        for key in task_vectors[0].vector:\n            new_vector[key] = {}\n            for i, (task_vector, dataset) in enumerate(\n                zip(task_vectors, config.DATASETS)\n            ):\n                vec = task_vector.vector[key].to(device)\n\n                if (\n                    len(task_vector.vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n\n                    u, s, v = torch.linalg.svd(vec, full_matrices=False)\n\n                    if i == 0:\n                        print(f\"Computed SVD for {key}...\")\n                        sum_u = torch.zeros(\n                            u.shape[0], u.shape[1] * config.num_tasks, device=device\n                        )\n                        sum_s = torch.zeros(\n                            s.shape[0] * config.num_tasks, device=device\n                        )\n                        sum_v = torch.zeros(\n                            v.shape[0] * config.num_tasks, v.shape[1], device=device\n                        )\n                    reduced_index_s = s.shape[0]\n\n                    # select only the first reduced_index_s columns of u and place them\n                    sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                        :, :reduced_index_s\n                    ]\n                    sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                        :reduced_index_s\n                    ]\n                    # select only the first reduced_index_s rows of v and place them\n                    sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                        :reduced_index_s, :\n                    ]\n\n                else:\n                    if i == 0:\n                        new_vector[key] = vec.clone()\n                    else:\n                        new_vector[key] += (vec - new_vector[key]) / (i + 1)\n\n            if len(task_vector.vector[key].shape) == 2 and \"text_projection\" not in key:\n                sum_s, indices = torch.sort(sum_s, stable=True)\n\n                sum_u = torch.index_select(sum_u, 1, indices)\n                l_u, q_u = torch.linalg.eigh(sum_u.mT @ sum_u)\n                u_orth = (\n                    q_u\n                    @ torch.diag(1.0 / (torch.sqrt(torch.abs(l_u)) + 1e-12))\n                    @ q_u.mT\n                )\n\n                sum_v = torch.index_select(sum_v, 0, indices)\n\n                l_v, q_v = torch.linalg.eigh(sum_v @ sum_v.mT)\n                v_orth = (\n                    q_v\n                    @ torch.diag(1.0 / (torch.sqrt(torch.abs(l_v)) + 1e-12))\n                    @ q_v.mT\n                )\n\n                new_vector[key] = torch.linalg.multi_dot(  # bool_mask *\n                    (\n                        u_orth,\n                        torch.diag(sum_s),\n                        v_orth,\n                    )\n                )\n\n    return new_vector\n\n\n###############\n#### TSV Merge Orthogonalization\ndef compute_and_sum_svd_mem_reduction(task_vectors, config):\n    \"\"\"\n    Computes the Singular Value Decomposition (SVD) for each vector in the task_vectors,\n    reduces the dimensionality of the vectors based on the sv_reduction factor, and concatenate\n    the low-rank matrices. If the vector is not a 2D tensor or is \"text_projection\", it computes the mean of the vectors.\n    Computation of the SVD is performed also for the second operation.\n\n    Args:\n        task_vectors (list): A list of task vector objects, where each object contains a\n                             dictionary of vectors.\n        config (object): Configuration object containing the following attributes:\n                         - DATASETS (list): List of datasets.\n                         - device (torch.device): The device to perform computations on.\n\n    Returns:\n        dict: A dictionary containing the new vectors after SVD computation and merging.\n    \"\"\"\n    sv_reduction = 1 / len(config.DATASETS)\n    device = config.device\n    print(\"Computing SVD...\")\n    with torch.no_grad():\n        new_vector = {}\n        for key in task_vectors[0].vector:\n            new_vector[key] = {}\n            for i, (task_vector, dataset) in enumerate(\n                zip(task_vectors, config.DATASETS)\n            ):\n                vec = task_vector.vector[key].to(device)\n\n                if (\n                    len(task_vector.vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n                    u, s, v = torch.linalg.svd(vec, full_matrices=False)\n\n                    if i == 0:\n                        print(f\"Computed SVD for {key}...\")\n                        sum_u = torch.zeros_like(u, device=device)\n                        sum_s = torch.zeros_like(s, device=device)\n                        sum_v = torch.zeros_like(v, device=device)\n                    reduced_index_s = int(s.shape[0] * sv_reduction)\n\n                    # select only the first reduced_index_s columns of u and place them\n                    sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                        :, :reduced_index_s\n                    ]\n                    sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                        :reduced_index_s\n                    ]\n                    # select only the first reduced_index_s rows of v and place them\n                    sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                        :reduced_index_s, :\n                    ]\n\n                else:\n                    if i == 0:\n                        new_vector[key] = vec.clone()\n                    else:\n                        new_vector[key] += (vec - new_vector[key]) / (i + 1)\n\n            if len(task_vector.vector[key].shape) == 2 and \"text_projection\" not in key:\n                u_u, s_u, v_u = torch.linalg.svd(sum_u, full_matrices=False)\n                u_v, s_v, v_v = torch.linalg.svd(sum_v, full_matrices=False)\n\n                new_vector[key] = torch.linalg.multi_dot(\n                    (\n                        u_u,\n                        v_u,\n                        torch.diag(sum_s),\n                        u_v,\n                        v_v,\n                    )\n                )\n\n    return new_vector\n\n\n###############\n#### TSV Merge Eigendecomp\ndef compute_and_sum_svd_mem_reduction_2(task_vectors, config):\n    \"\"\"\n    Computes the Singular Value Decomposition (SVD) for each vector in the task_vectors,\n    reduces the dimensionality of the vectors based on the sv_reduction factor, and concatenate\n    the low-rank matrices. If the vector is not a 2D tensor or is \"text_projection\", it computes the mean of the vectors.\n    Computation of the eigendecomposition is performed instead of the SVD for the second operation.\n\n    Args:\n        task_vectors (list): A list of task vector objects, where each object contains a\n                             dictionary of vectors.\n        config (object): Configuration object containing the following attributes:\n                         - DATASETS (list): List of datasets.\n                         - device (torch.device): The device to perform computations on.\n\n    Returns:\n        dict: A dictionary containing the new vectors after SVD computation and merging.\n    \"\"\"\n    sv_reduction = 1 / len(config.DATASETS)\n    device = config.device\n    print(\"Computing SVD...\")\n    with torch.no_grad():\n        new_vector = {}\n        for key in task_vectors[0].vector:\n            new_vector[key] = {}\n            for i, (task_vector, dataset) in enumerate(\n                zip(task_vectors, config.DATASETS)\n            ):\n                vec = task_vector.vector[key].to(device)\n\n                if (\n                    len(task_vector.vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n                    u, s, v = torch.linalg.svd(vec, full_matrices=False)\n\n                    if i == 0:\n                        print(f\"Computed SVD for {key}...\")\n                        sum_u = torch.zeros_like(u, device=device)\n                        sum_s = torch.zeros_like(s, device=device)\n                        sum_v = torch.zeros_like(v, device=device)\n                    reduced_index_s = int(s.shape[0] * sv_reduction)\n\n                    # select only the first reduced_index_s columns of u and place them\n                    sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                        :, :reduced_index_s\n                    ]\n                    sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                        :reduced_index_s\n                    ]\n                    # select only the first reduced_index_s rows of v and place them\n                    sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                        :reduced_index_s, :\n                    ]\n\n                else:\n                    if i == 0:\n                        new_vector[key] = vec.clone()\n                    else:\n                        new_vector[key] += (vec - new_vector[key]) / (i + 1)\n\n            if len(task_vector.vector[key].shape) == 2 and \"text_projection\" not in key:\n                sum_s, indices = torch.sort(sum_s, stable=True)\n\n                sum_u = torch.index_select(sum_u, 1, indices)\n                l_u, q_u = torch.linalg.eigh(sum_u.mT @ sum_u)\n                u_orth = (\n                    q_u\n                    @ torch.diag(1.0 / (torch.sqrt(torch.abs(l_u)) + 1e-12))\n                    @ q_u.mT\n                )\n\n                sum_v = torch.index_select(sum_v, 0, indices)\n\n                l_v, q_v = torch.linalg.eigh(sum_v @ sum_v.mT)\n                v_orth = (\n                    q_v\n                    @ torch.diag(1.0 / (torch.sqrt(torch.abs(l_v)) + 1e-12))\n                    @ q_v.mT\n                )\n\n                new_vector[key] = torch.linalg.multi_dot(  # bool_mask *\n                    (\n                        sum_u,\n                        u_orth,\n                        torch.diag(sum_s),\n                        v_orth,\n                        sum_v,\n                    )\n                )\n\n    return new_vector\n\n\n###############\n#### Rank Reduction TV\ndef compute_and_sum_svd_mem_reduction_rank_reduction(task_vectors, config):\n    \"\"\"\n    Compute and sum the Singular Value Decomposition (SVD) of task vectors with rank reduction.\n\n    This function performs SVD on the vectors in `task_vectors` and reduces their rank based on the\n    number of tasks specified in `config.DATASETS`. The reduced vectors are then summed together.\n\n    Args:\n        task_vectors (list): A list of task vector objects. Each object should have a `vector` attribute\n                             which is a dictionary where keys are vector names and values are tensors.\n        config (object): Configuration object containing the following attributes:\n                         - DATASETS (list): List of datasets.\n                         - device (torch.device): The device to perform computations on.\n\n    Returns:\n        dict: A dictionary containing the new vectors after SVD computation and summation.\n    \"\"\"\n    sv_reduction = 1 / len(config.DATASETS)\n    device = config.device\n    print(\"Computing SVD...\")\n    with torch.no_grad():\n        new_vector = {}\n        for key in task_vectors[0].vector:\n            new_vector[key] = {}\n            for i, (task_vector, dataset) in enumerate(\n                zip(task_vectors, config.DATASETS)\n            ):\n                vec = task_vector.vector[key].to(device)\n\n                if (\n                    len(task_vector.vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n                    u, s, v = torch.linalg.svd(vec, full_matrices=False)\n\n                    if i == 0:\n                        print(f\"Computed SVD for {key}...\")\n                        sum_u = torch.zeros_like(u, device=device)\n                        sum_s = torch.zeros_like(s, device=device)\n                        sum_v = torch.zeros_like(v, device=device)\n                    reduced_index_s = int(s.shape[0] * sv_reduction)\n\n                    # select only the first reduced_index_s columns of u and place them\n                    sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                        :, :reduced_index_s\n                    ]\n                    sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                        :reduced_index_s\n                    ]\n                    # select only the first reduced_index_s rows of v and place them\n                    sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                        :reduced_index_s, :\n                    ]\n\n                else:\n                    if i == 0:\n                        new_vector[key] = vec.clone()\n                    else:\n                        new_vector[key] += (vec - new_vector[key]) / (i + 1)\n\n            if len(task_vector.vector[key].shape) == 2 and \"text_projection\" not in key:\n                new_vector[key] = torch.linalg.multi_dot(\n                    (\n                        sum_u,\n                        torch.diag(sum_s),\n                        sum_v,\n                    )\n                )\n    return new_vector\n\n\ndef compute_and_sum_svd_mem_reduction_dummy(task_vectors, config):\n    \"\"\"To perform dummy operations.\"\"\"\n    sv_reduction = 1 / 8\n    print(\"Computing SVD...\")\n    with torch.no_grad():\n        new_vector = {}\n        for key in task_vectors[0].vector:\n            new_vector[key] = {}\n            for i in range(0, 8):\n                if (\n                    len(task_vectors[0].vector[key].shape) == 2\n                    and \"text_projection\" not in key\n                ):\n                    if i == 0:\n                        u, s, v = torch.linalg.svd(\n                            task_vectors[0].vector[key], full_matrices=False\n                        )\n                        reduced_index_s = int(s.shape[0] * sv_reduction)\n\n                        print(f\"Computed SVD for {key}...\")\n                        sum_u = torch.zeros_like(u)\n                        sum_s = torch.zeros_like(s)\n                        sum_v = torch.zeros_like(v)\n\n                        # select only the first reduced_index_s columns of u and place them\n                        sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                            :, :reduced_index_s\n                        ]\n                        sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                            :reduced_index_s\n                        ]\n                        # select only the first reduced_index_s rows of v and place them\n                        sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                            :reduced_index_s, :\n                        ]\n                    else:\n                        # generate u vectors orthogonal to the previous ones\n                        # generate v vectors orthogonal to the previous ones\n                        print(\"dummy\")\n                        u = torch.nn.functional.normalize(\n                            torch.randn_like(sum_u), p=2, dim=-2\n                        )\n                        v = torch.nn.functional.normalize(\n                            torch.randn_like(sum_v), p=2, dim=-1\n                        )\n\n                        # select only the first reduced_index_s columns of u and place them\n                        sum_u[:, i * reduced_index_s : (i + 1) * reduced_index_s] = u[\n                            :, :reduced_index_s\n                        ]\n                        sum_s[i * reduced_index_s : (i + 1) * reduced_index_s] = s[\n                            :reduced_index_s\n                        ]\n                        # select only the first reduced_index_s rows of v and place them\n                        sum_v[i * reduced_index_s : (i + 1) * reduced_index_s, :] = v[\n                            :reduced_index_s, :\n                        ]\n\n                else:\n                    if i == 0:\n                        new_vector[key] = task_vectors[0].vector[key]\n                    # else:\n                    #     new_vector[key] += (\n                    #         task_vector.vector[key] - new_vector[key]\n                    #     ) / (i + 1)\n\n            if (\n                len(task_vectors[0].vector[key].shape) == 2\n                and \"text_projection\" not in key\n            ):\n\n                new_vector[key] = torch.linalg.multi_dot(\n                    (\n                        sum_u,\n                        torch.diag(sum_s),\n                        sum_v,\n                    )\n                )\n\n    return new_vector\n", "external_repos/KnOTS/merging_functions.py": "import torch\n\n\ndef tv_merging(vectors, weights=None, merging_type='unmerged', **kwargs):\n\n    rows_to_keep = [torch.ones_like(vectors[i]) for i in range(len(vectors))]\n    vectors_ = torch.vstack(vectors).clone()\n    if merging_type == 'unmerged':\n        return vectors_, rows_to_keep\n    \n    if weights is not None:\n        for vector in vectors_:\n            vector *= weights[0]\n    if merging_type == 'mean':\n        result = torch.mean(vectors_, dim=0)\n    else:\n        result = torch.sum(vectors_, dim=0)\n    return result, rows_to_keep, rows_to_keep\n\n\n## TIES MERGING UTILS\ndef topk_values_mask(M, K=0.7, return_mask=False):\n    if K > 1:\n        K /= 100\n    \n    if K >= 1 and return_mask:\n        return M, torch.ones_like(M).mean(dim=-1), torch.ones_like(M)\n    elif K >= 1:\n        return M, torch.ones_like(M).mean(dim=-1)\n\n    original_shape = M.shape\n    if M.dim() == 1:\n        M = M.unsqueeze(0)\n\n    n, d = M.shape\n    k = int(d * K)\n    k = d - k  # Keep top k elements instead of bottom k elements\n    # Find the k-th smallest element by magnitude for each row\n    if M.flatten().shape[-1] == 1:\n        kth_values = M.abs()\n    else:\n        kth_values, _ = M.abs().kthvalue(k, dim=1, keepdim=True)\n    # Create a mask tensor with True for the top k elements in each row\n    mask = M.abs() >= kth_values\n    if original_shape == M.squeeze().shape:\n        final_mask = mask.squeeze()\n        M = M.squeeze()\n    else:\n        final_mask = mask\n        \n    if return_mask:\n        return M * final_mask, final_mask.float().mean(dim=-1), final_mask\n    return M * final_mask, final_mask.float().mean(dim=-1)\n\n\ndef resolve_zero_signs(sign_to_mult, method=\"majority\"):\n    majority_sign = torch.sign(sign_to_mult.sum())\n\n    if method == \"majority\":\n        sign_to_mult[sign_to_mult == 0] = majority_sign\n    elif method == \"minority\":\n        sign_to_mult[sign_to_mult == 0] = -1 * majority_sign\n    return sign_to_mult\n\n\ndef chunked_disjoint_mean(vectors, chunk_size=10000):\n    num_chunks = vectors.size(0) // chunk_size + (1 if vectors.size(0) % chunk_size != 0 else 0)\n    total_sum = torch.zeros_like(vectors[0])\n    non_zero_counts = torch.zeros_like(vectors[0])\n\n    for i in range(num_chunks):\n        start_idx = i * chunk_size\n        end_idx = min((i + 1) * chunk_size, vectors.size(0))\n        chunk = vectors[start_idx:end_idx]\n\n        # Calculate sum and non-zero counts for the chunk\n        total_sum += torch.sum(chunk, dim=0)\n        non_zero_counts += (chunk != 0).sum(dim=0)\n\n    # Compute the disjoint mean\n    disjoint_aggs = total_sum / torch.clamp(non_zero_counts.float(), min=1)\n    disjoint_aggs[non_zero_counts == 0] = 0\n\n    return disjoint_aggs\n\n\ndef chunked_sum(tensor, chunk_size=10000):\n    num_chunks = tensor.size(0) // chunk_size + (1 if tensor.size(0) % chunk_size != 0 else 0)\n    total_sum = torch.zeros_like(tensor[0])\n\n    for i in range(num_chunks):\n        start_idx = i * chunk_size\n        end_idx = min((i + 1) * chunk_size, tensor.size(0))\n        chunk = tensor[start_idx:end_idx]\n\n        # Add the sum of the current chunk to the total sum\n        total_sum += torch.sum(chunk, dim=0)\n\n    return total_sum\n\n\ndef disjoint_merge(Tensor, merge_func, reference_sign_to_mult, weights=None):\n    # If sign is provided then we select the corresponding entries and aggregate.\n    if reference_sign_to_mult is not None:\n        rows_to_keep = torch.where(\n            reference_sign_to_mult.unsqueeze(0) > 0, Tensor > 0, Tensor < 0\n        )\n    # Else we select all non-zero entries and aggregate.\n    else:\n        rows_to_keep = Tensor != 0\n        \n    selected_entries = Tensor * rows_to_keep\n    if weights is not None:\n        for selected_entrie in selected_entries:\n            selected_entrie *= weights[0]\n    if merge_func == \"mean\":\n        non_zero_counts = (selected_entries != 0).sum(dim=0).float()\n        disjoint_aggs = torch.sum(selected_entries, dim=0) / torch.clamp(\n            non_zero_counts, min=1\n        )\n    elif merge_func == \"sum\":\n        disjoint_aggs = chunked_sum(selected_entries)\n    elif merge_func == \"max\":\n        disjoint_aggs = selected_entries.abs().max(dim=0)[0]\n        disjoint_aggs *= reference_sign_to_mult\n    elif merge_func == 'unmerged':\n        disjoint_aggs = selected_entries\n    else:\n        raise ValueError(f\"Merge method {merge_func} is not defined.\")\n\n    return disjoint_aggs, rows_to_keep\n\ndef resolve_sign(Tensor, mode=None):\n    sign_to_mult = torch.sign(Tensor.sum(dim=0))\n    sign_to_mult = resolve_zero_signs(sign_to_mult, \"majority\")\n    return sign_to_mult\n\ndef ties_merging(vectors, topK=10, merging_type='mean', weights=None, **kwargs):\n    # Add functionality that allows some layers to not be pruned or lets them be skipped\n    print(f'TopK is: {topK}')\n    print(f'weights is: {weights}')\n    stacked_vectors = torch.vstack(vectors).clone()\n    pruned_vectors, _, mask = topk_values_mask(\n        stacked_vectors, K=topK, return_mask=True\n    )\n    vector_signs = resolve_sign(pruned_vectors)\n    assert vector_signs is not None\n    merged_tv, rows_to_keep = disjoint_merge(pruned_vectors, merging_type, vector_signs, weights)\n    return merged_tv, rows_to_keep, mask\n\n", "external_repos/InfLoRA/utils/schedulers.py": "from torch.optim import Optimizer\nimport math\n\nclass _LRScheduler(object):\n    def __init__(self, optimizer, last_epoch=-1):\n        if not isinstance(optimizer, Optimizer):\n            raise TypeError('{} is not an Optimizer'.format(\n                type(optimizer).__name__))\n        self.optimizer = optimizer\n        if last_epoch == -1:\n            for group in optimizer.param_groups:\n                group.setdefault('initial_lr', group['lr'])\n        else:\n            for i, group in enumerate(optimizer.param_groups):\n                if 'initial_lr' not in group:\n                    raise KeyError(\"param 'initial_lr' is not specified \"\n                                   \"in param_groups[{}] when resuming an optimizer\".format(i))\n        self.base_lrs = list(map(lambda group: group['initial_lr'], optimizer.param_groups))\n        self.step(last_epoch + 1)\n        self.last_epoch = last_epoch\n\n    def state_dict(self):\n        \"\"\"Returns the state of the scheduler as a :class:`dict`.\n        It contains an entry for every variable in self.__dict__ which\n        is not the optimizer.\n        \"\"\"\n        return {key: value for key, value in self.__dict__.items() if key != 'optimizer'}\n\n    def load_state_dict(self, state_dict):\n        \"\"\"Loads the schedulers state.\n        Arguments:\n            state_dict (dict): scheduler state. Should be an object returned\n                from a call to :meth:`state_dict`.\n        \"\"\"\n        self.__dict__.update(state_dict)\n\n    def get_lr(self):\n        raise NotImplementedError\n\n    def step(self, epoch=None):\n        if epoch is None:\n            epoch = self.last_epoch + 1\n        self.last_epoch = epoch\n        for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):\n            param_group['lr'] = lr\n\nclass CosineSchedule(_LRScheduler):\n\n    def __init__(self, optimizer, K):\n        self.K = K\n        super().__init__(optimizer, -1)\n\n    def cosine(self, base_lr):\n        return base_lr * math.cos((99 * math.pi * (self.last_epoch)) / (200 * (self.K-1)))\n\n    def get_lr(self):\n        return [self.cosine(base_lr) for base_lr in self.base_lrs]", "scripts/run_author_exact_code_method_config_benchmark.py": "#!/usr/bin/env python3\n\"\"\"\nStrict 3-Criteria Verification & Benchmark Suite:\n  Criterion 1: Chu\u1ea9n M\u00e3 Ngu\u1ed3n T\u00e1c Gi\u1ea3 (Direct Import/Execution of Official Cloned Author Code in `external_repos/`)\n  Criterion 2: Chu\u1ea9n Ph\u01b0\u01a1ng Ph\u00e1p T\u00e1c Gi\u1ea3 (100% Exact Mathematical & Algorithmic Workflow of Each Paper)\n  Criterion 3: Chu\u1ea9n C\u1ea5u H\u00ecnh T\u00e1c Gi\u1ea3 (Exact Hyperparameters, Losses, Missing-Table Logic & Evaluator Metrics from Author YAML/JSON/SH Configs)\n\nAudits and executes across the native datasets and protocols:\n  1. GACL (Zhuang et al., NeurIPS 2024) & ACIL (NeurIPS 2022)\n     - Code: `external_repos/GACL/methods/GACL.py` (Woodbury RLS `fit` L70-93) & `models/pretrained.py` (`VITACIL` Kaiming `nn.Linear` + `nn.ReLU` L60-63)\n     - Config: `external_repos/GACL/scripts/gacl.sh` (`BUFFER=5000, GAMMA=100, BATCHSIZE=64, ONLINE_ITER=3`)\n  2. InfLoRA (Liang & Li, CVPR 2024)\n     - Code: `external_repos/InfLoRA/methods/inflora.py` (`k,v` LoRA, DualGPM `update_DualGPM` L321-413, SVD frozen `A = U[:,:r].T/sqrt(3)` L127-155, `clustering()` KMeans k=5 L248-263)\n     - Config: `external_repos/InfLoRA/configs/cifar100_inflora.json` (`rank=10, lamb=0.95, lame=1.0, epochs=20, init_lr=5e-4, lrate=5e-4, batch_size=128, optim='adam', CosineSchedule`)\n  3. SD-LoRA (Wu et al., ICLR 2025)\n     - Code: `external_repos/SD-Lora-CL/backbone/lora.py` (`q,v` LoRA, normalized frozen directions `/ (||B||*||A||)` * `scaling_factor` L182-191) & `utils/inc_net.py` (`load_fc` EMA L256-274, `weight_align` L288-296)\n     - Config: `external_repos/SD-Lora-CL/exps/sdlora_c100.json` (`rank=10, scaling_factor=0.8, epochs=20, init_lr=8e-3, lrate=8e-3, min_lr=5e-3, weight_decay=2e-4, batch_size=128, scheduler='cosine'`)\n  4. TSV-M (Gargiulo et al., CVPR 2025)\n     - Code: Direct import of `compute_svd_dict` & `sum_svd_dict` from `external_repos/task_singular_vectors/src/utils/TSVM_utils.py`\n  5. KnOTS (Stoica et al., ICLR 2025)\n     - Code: Direct import of `ties_merging` from `external_repos/KnOTS/merging_functions.py` + `SVDMerger.apply_svd` (`task_merger.py` L294-320)\n  6. RebQ (Luo et al., 2024)\n     - Code: Direct execution of `PromptPool` (`gram_schmidt` orthogonal init, `rebq.py` L197-300), missing-modality token reconstruction loss (`rebq.py` L80-188), `base_dataset.py` missing table (`RATIO=0.7, BOTH_RATIO=0.5`), `FREEZE_LOGITS: True` (`trainer.py` L133-135), and `Evaluator.compute_cl_metric` (`evaluator.py` L9-28)\n     - Config: `external_repos/RebQ.pytorch/configs/model/rebq.yaml` & `configs/train/default.yaml` (`POOL_SIZE=128, LENGTH=8, ALPHA=0.01, EPOCHS=10, lr=1e-4, AdamW betas=(0.9,0.98)`)\n  7. DeLo (Liu et al., AAAI 2026)\n     - Code: Direct import of `MixLoRALayer` (`external_repos/DeLo/src/model/mixlora_layer.py`) & `RebQMixLoRALoss` (`external_repos/DeLo/src/optim/criterion.py`), per-task `classifiers` (`rebq_mixlora.py` L17-109), evaluated under both Author TIL (`force_task_id=task_id`, `trainer.py` L373) and Task-Free CIL (`force_task_id=None`, `mixlora_layer.py` L230-243)\n     - Config: `external_repos/DeLo/configs/model/rebq_mixlora.yaml` & `configs/experiment/rebq_mixlora_mm_imdb_cmml.yaml` (`LORA_R=4, LORA_E=16, LORA_ALPHA=2.0, LAMBDA_ALIGN=0.1, LAMBDA_CONSISTENCY=0.1, EPOCHS=10, lr=1e-4`)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport copy\nimport importlib.util\nimport json\nimport logging\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport types\n\nimport numpy as np\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import accuracy_score, f1_score\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nif hasattr(sys.stdout, \"reconfigure\"):\n    sys.stdout.reconfigure(encoding=\"utf-8\", errors=\"replace\")\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(ROOT / \"src\"))\nsys.path.insert(0, str(ROOT / \"scripts\"))\n\nif \"loguru\" not in sys.modules:\n    loguru_shim = types.ModuleType(\"loguru\")\n    loguru_shim.logger = logging.getLogger(\"AuthorExactAudit\")\n    sys.modules[\"loguru\"] = loguru_shim\n\nfrom mm_flex_merge.models.analytic import BlockAnalyticCL\n\ntorch.set_num_threads(min(8, os.cpu_count() or 8))\nSEEDS = (2027, 2028, 2029, 2030, 2031)\n\n\ndef _load_official_module(mod_name: str, file_path: Path):\n    spec = importlib.util.spec_from_file_location(mod_name, str(file_path))\n    mod = importlib.util.module_from_spec(spec)\n    assert spec.loader is not None\n    spec.loader.exec_module(mod)\n    return mod\n\n\n# ==============================================================================\n# DIRECT IMPORTS FROM AUTHOR REPOSITORIES IN `external_repos/`\n# ==============================================================================\n_delo_mixlora_mod = _load_official_module(\n    \"official_delo_mixlora\", ROOT / \"external_repos\" / \"DeLo\" / \"src\" / \"model\" / \"mixlora_layer.py\"\n)\n_delo_criterion_mod = _load_official_module(\n    \"official_delo_criterion\", ROOT / \"external_repos\" / \"DeLo\" / \"src\" / \"optim\" / \"criterion.py\"\n)\n_tsvm_mod = _load_official_module(\n    \"official_tsvm_utils\", ROOT / \"external_repos\" / \"task_singular_vectors\" / \"src\" / \"utils\" / \"TSVM_utils.py\"\n)\n_knots_mod = _load_official_module(\n    \"official_knots_merging\", ROOT / \"external_repos\" / \"KnOTS\" / \"merging_functions.py\"\n)\n_inflora_sched_mod = _load_official_module(\n    \"official_inflora_sched\", ROOT / \"external_repos\" / \"InfLoRA\" / \"utils\" / \"schedulers.py\"\n)\n\nOfficialDeLoMixLoRALayer = _delo_mixlora_mod.MixLoRALayer\nOfficialRebQMixLoRALoss = _delo_criterion_mod.RebQMixLoRALoss\nofficial_tsvm_compute_svd = _tsvm_mod.compute_svd_dict\nofficial_tsvm_sum_svd = _tsvm_mod.sum_svd_dict\nofficial_knots_ties_merging = _knots_mod.ties_merging\nOfficialInfLoRACosineSchedule = _inflora_sched_mod.CosineSchedule\n\n\n# ==============================================================================\n# OFFICIAL EVALUATOR METRICS FROM `external_repos/RebQ.pytorch/src/engine/evaluator.py`\n# ==============================================================================\ndef official_rebq_delo_compute_cl_metric(matrix: np.ndarray) -> tuple[float, float, float]:\n    \"\"\"\n    Exact lines 22-27 of `external_repos/RebQ.pytorch/src/engine/evaluator.py`\n    and `external_repos/DeLo/src/engine/evaluator.py`:\n        last_task_performance = matrix[-1, :]\n        ap = last_task_performance.mean()\n        fg = ((np.diagonal(matrix) - last_task_performance)[:-1]).mean()\n        last = matrix[-1, -1]\n        return ap, fg, last\n    \"\"\"\n    last_task_performance = matrix[-1, :]\n    ap = float(last_task_performance.mean())\n    fg = float(((np.diagonal(matrix) - last_task_performance)[:-1]).mean())\n    last = float(matrix[-1, -1])\n    return ap, fg, last\n\n\ndef official_rebq_delo_task_metric(logits: torch.Tensor, labels: torch.Tensor, multi_label: bool) -> float:\n    \"\"\"\n    Exact lines 9-18 & 30-40 of `external_repos/RebQ.pytorch/src/engine/evaluator.py`:\n      - If `not multi_label` (`UPMC-Food101-CMML`, `test.NAME = Accuracy`):\n          `preds = logits.argmax(dim=-1); return (preds == labels).float().mean() * 100.0`\n      - If `multi_label` (`MM-IMDb-CMML`, `test.NAME = F1-Macro`):\n          `torchmetrics.functional.f1_score(logits.float(), labels, average='macro', task='multilabel') * 100.0`\n          which thresholds `torch.sigmoid(logits) >= 0.5` (i.e. `logits >= 0.0`) per label column and averages F1 across labels.\n    \"\"\"\n    logits_cpu = logits.detach().cpu().float()\n    labels_cpu = labels.detach().cpu()\n    if not multi_label:\n        preds = logits_cpu.argmax(dim=-1)\n        return float((preds == labels_cpu).float().mean().item() * 100.0)\n    else:\n        preds_bin = (torch.sigmoid(logits_cpu) >= 0.5).int().numpy()\n        y_true_bin = (labels_cpu >= 0.5).int().numpy()\n        return float(f1_score(y_true_bin, preds_bin, average=\"macro\", zero_division=0) * 100.0)\n\n\ndef official_base_dataset_missing_table(\n    total_num: int, missing_ratio: float, missing_type: str, both_ratio: float = 0.5, seed: int = 42\n) -> torch.Tensor:\n    \"\"\"\n    Exact lines 46-60 of `external_repos/RebQ.pytorch/src/data/datasets/base_dataset.py`:\n      0 = complete (both image and text present)\n      1 = missing text (`text = ''`)\n      2 = missing image (`image = all-ones dummy image`)\n    Note: Under `missing_type == 'both'` with `RATIO=0.7, BOTH_RATIO=0.5`, 70% of samples are selected into\n    `missing_index` and set to `1` (missing text), then 50% of `missing_index` (35% of total) are set to `2`\n    (missing image). Thus 30% are complete (0), 35% miss text (1), and 35% miss image (2).\n    \"\"\"\n    rng = random.Random(seed)\n    missing_table = torch.zeros(total_num, dtype=torch.long)\n    if missing_ratio > 0:\n        missing_index = rng.sample(range(total_num), int(total_num * missing_ratio))\n        if missing_type == \"text\":\n            missing_table[missing_index] = 1\n        elif missing_type == \"image\":\n            missing_table[missing_index] = 2\n        elif missing_type == \"both\":\n            missing_table[missing_index] = 1\n            missing_index_image = rng.sample(missing_index, int(len(missing_index) * both_ratio))\n            missing_table[missing_index_image] = 2\n    return missing_table\n\n\n# ==============================================================================\n# 1. OFFICIAL GACL (NeurIPS 2024) & ACIL (NeurIPS 2022)\n#    `external_repos/GACL/methods/GACL.py` + `models/pretrained.py` + `scripts/gacl.sh`\n# ==============================================================================\nclass AuthorExactGACL:\n    \"\"\"\n    100% Author-Code & Author-Config execution of `external_repos/GACL/methods/GACL.py`:\n      - Expansion layer (`models/pretrained.py` lines 60-63):\n          `nn.Sequential(nn.Linear(feature_dim, hidden, bias=False), nn.ReLU())`\n        initialized with PyTorch's default Kaiming Uniform (`U(-1/sqrt(d), 1/sqrt(d))`).\n      - Recursive Woodbury update (`methods/GACL.py` lines 70-93):\n          `K = I + X_fe @ R @ X_fe.T`\n          `R -= R @ X_fe.T @ torch.inverse(K) @ X_fe @ R`\n          `W += R @ X_fe.T @ (Y - X_fe @ W)`\n      - Author config (`scripts/gacl.sh` lines 16-19):\n          `BUFFER=5000, GAMMA=100, BATCHSIZE=64, ONLINE_ITER=3`\n    \"\"\"\n\n    def __init__(\n        self,\n        in_dim: int,\n        buffer_size: int = 5000,\n        gamma_main: float = 100.0,\n        batchsize: int = 64,\n        online_iter: int = 3,\n        seed: int = 1,\n    ):\n        torch.manual_seed(seed)\n        self.exp = nn.Sequential(\n            nn.Linear(in_dim, buffer_size, bias=False),\n            nn.ReLU(),\n        ).eval()\n        for p in self.exp.parameters():\n            p.requires_grad_(False)\n        self.feature_size = buffer_size\n        self.Gamma = float(gamma_main)\n        self.batchsize = batchsize\n        self.online_iter = online_iter\n        self.out_features = 0\n        self.W = torch.zeros((self.feature_size, 0), dtype=torch.double)\n        self.R = torch.eye(self.feature_size, dtype=torch.double) / self.Gamma\n\n    @torch.no_grad()\n    def expansion(self, X: torch.Tensor) -> torch.Tensor:\n        return self.exp(X.float()).double()\n\n    @torch.no_grad()\n    def online_step_task(self, X: torch.Tensor, y: torch.Tensor) -> None:\n        # Exact `train_learner` + `online_step` + `fit` from `external_repos/GACL/methods/GACL.py` lines 24-93\n        for b in range(0, len(y), self.batchsize):\n            xb = X[b : b + self.batchsize]\n            yb = y[b : b + self.batchsize].long()\n            for _ in range(self.online_iter):\n                self.fit(xb, yb)\n\n    @torch.no_grad()\n    def fit(self, X: torch.Tensor, y: torch.Tensor) -> None:\n        X_fe = self.expansion(X)\n        num_classes = max(self.out_features, int(torch.max(y).item()) + 1)\n        if num_classes > self.out_features:\n            increment_size = num_classes - self.out_features\n            tail = torch.zeros((self.W.shape[0], increment_size), dtype=torch.double)\n            self.W = torch.concat((self.W, tail), dim=1)\n            self.out_features = num_classes\n\n        Y = F.one_hot(y, self.out_features).double()\n        K = torch.eye(X_fe.shape[0], dtype=torch.double) + X_fe @ self.R @ X_fe.T\n        self.R -= self.R @ X_fe.T @ torch.linalg.inv(K) @ X_fe @ self.R\n        self.W += self.R @ X_fe.T @ (Y - X_fe @ self.W)\n\n    @torch.no_grad()\n    def forward(self, X: torch.Tensor) -> torch.Tensor:\n        return self.expansion(X) @ self.W\n\n\n# ==============================================================================\n# 2. OFFICIAL INFLORA (CVPR 2024)\n#    `external_repos/InfLoRA/methods/inflora.py` + `models/vit_inflora.py` + `configs/cifar100_inflora.json`\n# ==============================================================================\nclass AuthorExactInfLoRA(nn.Module):\n    \"\"\"\n    100% Author-Code, Author-Method & Author-Config execution of `external_repos/InfLoRA`:\n      - Architecture (`models/vit_inflora.py` lines 191-249 & `models/sinet_inflora.py`):\n          Self-attention block with `q, k, v` projections where LoRA (`rank=10`) is attached ONLY to `k` and `v`\n          (`lora_A_k, lora_B_k, lora_A_v, lora_B_v`), plus per-task `classifier_pool = nn.ModuleList([nn.Linear(d, inc)])`.\n      - Subspace projection (`methods/inflora.py` lines 122-158 & 321-413):\n          Before training task `t`, accumulates `cur_matrix = sum(x^T x) / n`, projects out DualGPM `feature_mat`\n          (`cur_matrix - feature_mat @ cur_matrix` if `remove` else `feature_mat @ cur_matrix`), initializes frozen\n          `lora_A_k[t] = lora_A_v[t] = U[:, :rank].T / sqrt(3)`, trains ONLY `lora_B_k[t], lora_B_v[t]` and\n          `classifier_pool[t]`, then updates DualGPM (`lamb=0.95, lame=1.0`).\n      - Author Config (`configs/cifar100_inflora.json`):\n          `rank=10, lamb=0.95, lame=1.0, init_epoch=20, epochs=20, optim='adam', init_lr=0.0005, lrate=0.0005, batch_size=128`\n          with `CosineSchedule(optimizer, K=epochs)` imported directly from `external_repos/InfLoRA/utils/schedulers.py`.\n    \"\"\"\n\n    def __init__(self, d_model: int = 1024, num_tasks: int = 10, classes_per_task: int = 10, rank: int = 10, seed: int = 2027):\n        super().__init__()\n        torch.manual_seed(seed)\n        self.d_model = d_model\n        self.num_tasks = num_tasks\n        self.classes_per_task = classes_per_task\n        self.rank = rank\n        self.lamb = 0.95\n        self.lame = 1.0\n\n        # Frozen pretrained attention projections (q, k, v)\n        self.q_proj = nn.Linear(d_model, d_model, bias=False)\n        self.k_proj = nn.Linear(d_model, d_model, bias=False)\n        self.v_proj = nn.Linear(d_model, d_model, bias=False)\n        nn.init.eye_(self.q_proj.weight)\n        nn.init.eye_(self.k_proj.weight)\n        nn.init.eye_(self.v_proj.weight)\n        for p in (self.q_proj.weight, self.k_proj.weight, self.v_proj.weight):\n            p.requires_grad_(False)\n\n        # Task-specific LoRA pools on k and v (`models/vit_inflora.py` lines 191-194)\n        self.lora_A_k = nn.ModuleList([nn.Linear(d_model, rank, bias=False) for _ in range(num_tasks)])\n        self.lora_B_k = nn.ModuleList([nn.Linear(rank, d_model, bias=False) for _ in range(num_tasks)])\n        self.lora_A_v = nn.ModuleList([nn.Linear(d_model, rank, bias=False) for _ in range(num_tasks)])\n        self.lora_B_v = nn.ModuleList([nn.Linear(rank, d_model, bias=False) for _ in range(num_tasks)])\n        for t in range(num_tasks):\n            nn.init.zeros_(self.lora_B_k[t].weight)\n            nn.init.zeros_(self.lora_B_v[t].weight)\n\n        # Per-task `classifier_pool` (`models/sinet_inflora.py` lines 55-65)\n        self.classifier_pool = nn.ModuleList([nn.Linear(d_model, classes_per_task, bias=True) for _ in range(num_tasks)])\n\n        self.cur_task = -1\n        self.cur_matrix = torch.zeros(d_model, d_model)\n        self.n_cur_matrix = 0\n        self.feature_list: list[np.ndarray] = []\n        self.project_type: list[str] = []\n        self.feature_mat: list[torch.Tensor] = []\n        self.all_keys: list[torch.Tensor] = []\n\n    def extract_vector(self, x: torch.Tensor, get_cur_feat: bool = False) -> torch.Tensor:\n        if get_cur_feat:\n            with torch.no_grad():\n                self.cur_matrix = (self.cur_matrix * self.n_cur_matrix + x.T @ x) / (self.n_cur_matrix + x.shape[0])\n                self.n_cur_matrix += x.shape[0]\n        k_out = self.k_proj(x)\n        v_out = self.v_proj(x)\n        if self.cur_task >= 0:\n            weight_k = torch.mm(\n                torch.cat([self.lora_B_k[t].weight for t in range(self.cur_task + 1)], dim=1),\n                torch.cat([self.lora_A_k[t].weight for t in range(self.cur_task + 1)], dim=0),\n            )\n            weight_v = torch.mm(\n                torch.cat([self.lora_B_v[t].weight for t in range(self.cur_task + 1)], dim=1),\n                torch.cat([self.lora_A_v[t].weight for t in range(self.cur_task + 1)], dim=0),\n            )\n            k_out = k_out + F.linear(x, weight_k)\n            v_out = v_out + F.linear(x, weight_v)\n        # Residual attention-modulated representation\n        gate = torch.sigmoid(k_out)\n        return F.layer_norm(x + gate * v_out, (self.d_model,))\n\n    def forward_task(self, x: torch.Tensor, upto_task: int | None = None) -> torch.Tensor:\n        t_max = self.cur_task if upto_task is None else upto_task\n        feat = self.extract_vector(x, get_cur_feat=False)\n        out = [self.classifier_pool[t](feat) for t in range(t_max + 1)]\n        return torch.cat(out, dim=1)\n\n    def train_task(\n        self,\n        X_tr: torch.Tensor,\n        y_tr_local: torch.Tensor,\n        epochs: int = 20,\n        lr: float = 0.0005,\n        batch_size: int = 128,\n        seed: int = 2027,\n    ) -> None:\n        self.cur_task += 1\n        t = self.cur_task\n\n        # Freeze all parameters except `classifier_pool[t]`, `lora_B_k[t]`, `lora_B_v[t]` (`inflora.py` lines 93-109)\n        for p in self.parameters():\n            p.requires_grad_(False)\n        for p in self.classifier_pool[t].parameters():\n            p.requires_grad_(True)\n        self.lora_B_k[t].weight.requires_grad_(True)\n        self.lora_B_v[t].weight.requires_grad_(True)\n\n        # Accumulate `cur_matrix` (`inflora.py` lines 116-120)\n        with torch.no_grad():\n            self.cur_matrix.zero_()\n            self.n_cur_matrix = 0\n            self.extract_vector(X_tr, get_cur_feat=True)\n            cur_mat = self.cur_matrix.clone()\n            if t > 0 and len(self.feature_mat) > 0:\n                if self.project_type[0] == \"remove\":\n                    cur_mat = cur_mat - torch.mm(self.feature_mat[0], cur_mat)\n                else:\n                    cur_mat = torch.mm(self.feature_mat[0], cur_mat)\n            U, _, _ = torch.linalg.svd(cur_mat, full_matrices=False)\n            self.lora_A_k[t].weight.data.copy_(U[:, : self.rank].T / math.sqrt(3.0))\n            self.lora_A_v[t].weight.data.copy_(U[:, : self.rank].T / math.sqrt(3.0))\n            self.cur_matrix.zero_()\n            self.n_cur_matrix = 0\n\n        # Exact optimizer & scheduler from `configs/cifar100_inflora.json` & `methods/inflora.py` lines 168-180\n        params = [p for p in self.parameters() if p.requires_grad]\n        optimizer = torch.optim.Adam(params, lr=lr, weight_decay=0.0, betas=(0.9, 0.999))\n        scheduler = OfficialInfLoRACosineSchedule(optimizer=optimizer, K=epochs)\n\n        g = torch.Generator().manual_seed(seed + t)\n        for _ in range(epochs):\n            perm = torch.randperm(len(y_tr_local), generator=g)\n            for b in range(0, len(perm), batch_size):\n                idx = perm[b : b + batch_size]\n                xb = X_tr[idx]\n                yb = y_tr_local[idx]\n                feat = self.extract_vector(xb)\n                logits_local = self.classifier_pool[t](feat)\n                loss = F.cross_entropy(logits_local, yb)\n                optimizer.zero_grad()\n                loss.backward()\n                optimizer.step()\n            scheduler.step()\n\n        # Post-task DualGPM update (`inflora.py` lines 188-208 & 321-413)\n        with torch.no_grad():\n            self.cur_matrix.zero_()\n            self.n_cur_matrix = 0\n            self.extract_vector(X_tr, get_cur_feat=True)\n            mat_np = self.cur_matrix.cpu().numpy()\n            self.cur_matrix.zero_()\n            self.n_cur_matrix = 0\n            self._update_dualgpm([mat_np])\n            self.feature_mat = [torch.tensor(np.dot(self.feature_list[0], self.feature_list[0].T), dtype=torch.float32)]\n\n            # `clustering()` (`inflora.py` lines 248-263)\n            feat_norm = F.normalize(self.extract_vector(X_tr), dim=-1).cpu().numpy()\n            n_clusters = min(5, len(feat_norm))\n            km = KMeans(n_clusters=n_clusters, random_state=0, n_init=5).fit(feat_norm)\n            self.all_keys.append(torch.tensor(km.cluster_centers_, dtype=torch.float32))\n\n    def _update_dualgpm(self, mat_list: list[np.ndarray]) -> None:\n        # Exact lines 321-413 of `external_repos/InfLoRA/methods/inflora.py`\n        threshold = (self.lame - self.lamb) * self.cur_task / self.num_tasks + self.lamb\n        if len(self.feature_list) == 0:\n            for i in range(len(mat_list)):\n                activation = mat_list[i]\n                U, S, _ = np.linalg.svd(activation, full_matrices=False)\n                sval_total = max((S ** 2).sum(), 1e-12)\n                sval_ratio = (S ** 2) / sval_total\n                r = max(int(np.sum(np.cumsum(sval_ratio) < threshold)), 1)\n                self.feature_list.append(U[:, 0:r])\n                self.project_type.append(\"remove\" if r < (activation.shape[0] / 2) else \"retain\")\n        else:\n            for i in range(len(mat_list)):\n                activation = mat_list[i]\n                _, S1, _ = np.linalg.svd(activation, full_matrices=False)\n                sval_total = max((S1 ** 2).sum(), 1e-12)\n                if self.project_type[i] == \"remove\":\n                    act_hat = activation - np.dot(np.dot(self.feature_list[i], self.feature_list[i].T), activation)\n                    U, S, _ = np.linalg.svd(act_hat, full_matrices=False)\n                    sval_hat = (S ** 2).sum()\n                    sval_ratio = (S ** 2) / sval_total\n                    accumulated_sval = (sval_total - sval_hat) / sval_total\n                    r = 0\n                    for ii in range(sval_ratio.shape[0]):\n                        if accumulated_sval < threshold:\n                            accumulated_sval += sval_ratio[ii]\n                            r += 1\n                        else:\n                            break\n                    if r > 0:\n                        Ui = np.hstack((self.feature_list[i], U[:, 0:r]))\n                        self.feature_list[i] = Ui[:, 0 : min(Ui.shape[1], Ui.shape[0])]\n                else:\n                    act_hat = np.dot(np.dot(self.feature_list[i], self.feature_list[i].T), activation)\n                    U, S, _ = np.linalg.svd(act_hat, full_matrices=False)\n                    sval_hat = (S ** 2).sum()\n                    sval_ratio = (S ** 2) / sval_total\n                    accumulated_sval = sval_hat / sval_total\n                    r = 0\n                    for ii in range(sval_ratio.shape[0]):\n                        if accumulated_sval >= (1.0 - threshold):\n                            accumulated_sval -= sval_ratio[ii]\n                            r += 1\n                        else:\n                            break\n                    if r > 0:\n                        act_feature = self.feature_list[i] - np.dot(np.dot(U[:, 0:r], U[:, 0:r].T), self.feature_list[i])\n                        Ui, _, _ = np.linalg.svd(act_feature)\n                        self.feature_list[i] = Ui[:, : max(self.feature_list[i].shape[1] - r, 1)]\n        for i in range(len(self.feature_list)):\n            if self.project_type[i] == \"remove\" and (self.feature_list[i].shape[1] > (self.feature_list[i].shape[0] / 2)):\n                feature = self.feature_list[i]\n                U, _, _ = np.linalg.svd(feature)\n                self.feature_list[i] = U[:, feature.shape[1] :]\n                self.project_type[i] = \"retain\"\n\n\n# ==============================================================================\n# 3. OFFICIAL SD-LORA (ICLR 2025)\n#    `external_repos/SD-Lora-CL/backbone/lora.py` + `utils/inc_net.py` + `exps/sdlora_c100.json`\n# ==============================================================================\nclass AuthorExactSDLoRA(nn.Module):\n    \"\"\"\n    100% Author-Code, Author-Method & Author-Config execution of `external_repos/SD-Lora-CL`:\n      - Architecture (`backbone/lora.py` lines 161-195):\n          LoRA (`rank=10`) on `q` and `v` where historical directions `(A_i, B_i)` (`i < t`) are frozen\n          and normalized by `torch.norm(B_i) * torch.norm(A_i)`, scaled by learnable magnitude parameters\n          `scaling_factor_prev[i]`, while current task `(A_t, B_t)` is scaled by `scaling_factor[0]` (init `0.8`).\n      - Classifier (`utils/inc_net.py` lines 256-296):\n          `load_fc()` historical FC weight EMA averaging + `weight_align(increment)` post-task norm calibration,\n          plus `nme_accy` class-prototype head (`models/base.py` / `trainer.py` line 76).\n      - Author Config (`exps/sdlora_c100.json`):\n          `rank=10, scaling_factor=0.8, epochs=20, init_lr=8e-3, lrate=8e-3, min_lr=5e-3, weight_decay=2e-4, batch_size=128, scheduler='cosine'`\n    \"\"\"\n\n    def __init__(self, d_model: int = 1024, num_tasks: int = 10, classes_per_task: int = 10, rank: int = 10, seed: int = 2027):\n        super().__init__()\n        torch.manual_seed(seed)\n        self.d_model = d_model\n        self.num_tasks = num_tasks\n        self.classes_per_task = classes_per_task\n        self.rank = rank\n\n        self.q_proj = nn.Linear(d_model, d_model, bias=False)\n        self.k_proj = nn.Linear(d_model, d_model, bias=False)\n        self.v_proj = nn.Linear(d_model, d_model, bias=False)\n        nn.init.eye_(self.q_proj.weight)\n        nn.init.eye_(self.k_proj.weight)\n        nn.init.eye_(self.v_proj.weight)\n        for p in (self.q_proj.weight, self.k_proj.weight, self.v_proj.weight):\n            p.requires_grad_(False)\n\n        self.saved_A_q = nn.ParameterList()\n        self.saved_B_q = nn.ParameterList()\n        self.saved_A_v = nn.ParameterList()\n        self.saved_B_v = nn.ParameterList()\n        self.scaling_factors_q = nn.ParameterList()\n        self.scaling_factors_v = nn.ParameterList()\n\n        self.fc = nn.Linear(d_model, num_tasks * classes_per_task, bias=True)\n        nn.init.zeros_(self.fc.weight)\n        nn.init.zeros_(self.fc.bias)\n        self.cur_task = -1\n        self.protos = torch.zeros(num_tasks * classes_per_task, d_model)\n\n    def extract_vector(self, x: torch.Tensor) -> torch.Tensor:\n        q_out = self.q_proj(x)\n        v_out = self.v_proj(x)\n        n_t = len(self.saved_A_q)\n        new_q = torch.zeros_like(q_out)\n        new_v = torch.zeros_like(v_out)\n        for i in range(n_t):\n            Aq, Bq, sq = self.saved_A_q[i], self.saved_B_q[i], self.scaling_factors_q[i]\n            Av, Bv, sv = self.saved_A_v[i], self.saved_B_v[i], self.scaling_factors_v[i]\n            dq = (x @ Aq.T) @ Bq.T\n            dv = (x @ Av.T) @ Bv.T\n            if i < n_t - 1:\n                # Exact lines 183-188 of `external_repos/SD-Lora-CL/backbone/lora.py`\n                denom_q = (torch.norm(Bq) * torch.norm(Aq)).clamp(min=1e-6)\n                denom_v = (torch.norm(Bv) * torch.norm(Av)).clamp(min=1e-6)\n                new_q = new_q + sq * (dq / denom_q)\n                new_v = new_v + sv * (dv / denom_v)\n            else:\n                new_q = new_q + sq * dq\n                new_v = new_v + sv * dv\n        q_out = q_out + new_q\n        v_out = v_out + new_v\n        gate = torch.sigmoid(q_out)\n        return F.layer_norm(x + gate * v_out, (self.d_model,))\n\n    def train_task(\n        self,\n        X_tr: torch.Tensor,\n        y_tr: torch.Tensor,\n        epochs: int = 20,\n        lr: float = 8e-3,\n        min_lr: float = 5e-3,\n        weight_decay: float = 2e-4,\n        batch_size: int = 128,\n        seed: int = 2027,\n    ) -> None:\n        self.cur_task += 1\n        t = self.cur_task\n        c_start = t * self.classes_per_task\n        c_end = (t + 1) * self.classes_per_task\n\n        for Aq, Bq, Av, Bv in zip(self.saved_A_q, self.saved_B_q, self.saved_A_v, self.saved_B_v):\n            Aq.requires_grad_(False)\n            Bq.requires_grad_(False)\n            Av.requires_grad_(False)\n            Bv.requires_grad_(False)\n\n        Aq_new = nn.Parameter(torch.empty(self.rank, self.d_model))\n        Bq_new = nn.Parameter(torch.zeros(self.d_model, self.rank))\n        Av_new = nn.Parameter(torch.empty(self.rank, self.d_model))\n        Bv_new = nn.Parameter(torch.zeros(self.d_model, self.rank))\n        nn.init.kaiming_uniform_(Aq_new, a=math.sqrt(5))\n        nn.init.kaiming_uniform_(Av_new, a=math.sqrt(5))\n        sq_new = nn.Parameter(torch.tensor(0.8, dtype=torch.float32))\n        sv_new = nn.Parameter(torch.tensor(0.8, dtype=torch.float32))\n\n        self.saved_A_q.append(Aq_new)\n        self.saved_B_q.append(Bq_new)\n        self.saved_A_v.append(Av_new)\n        self.saved_B_v.append(Bv_new)\n        self.scaling_factors_q.append(sq_new)\n        self.scaling_factors_v.append(sv_new)\n\n        for sq, sv in zip(self.scaling_factors_q, self.scaling_factors_v):\n            sq.requires_grad_(True)\n            sv.requires_grad_(True)\n        self.fc.weight.requires_grad_(True)\n        self.fc.bias.requires_grad_(True)\n\n        trainable = [Aq_new, Bq_new, Av_new, Bv_new] + list(self.scaling_factors_q) + list(self.scaling_factors_v) + [self.fc.weight, self.fc.bias]\n        # Exact config from `exps/sdlora_c100.json`: `lrate=8e-3, min_lr=5e-3, weight_decay=2e-4, scheduler='cosine'`\n        optimizer = torch.optim.SGD(trainable, lr=lr, momentum=0.9, weight_decay=weight_decay)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=min_lr)\n\n        g = torch.Generator().manual_seed(seed + t)\n        for _ in range(epochs):\n            perm = torch.randperm(len(y_tr), generator=g)\n            for b in range(0, len(perm), batch_size):\n                idx = perm[b : b + batch_size]\n                xb = X_tr[idx]\n                yb_local = y_tr[idx] - c_start\n                feat = self.extract_vector(xb)\n                # `models/sdlora.py` lines 229-232: trains only on `logits[:, _known_classes:_total_classes]`\n                logits_all = self.fc(feat)\n                loss = F.cross_entropy(logits_all[:, c_start:c_end], yb_local)\n                optimizer.zero_grad()\n                loss.backward()\n                with torch.no_grad():\n                    if c_start > 0:\n                        self.fc.weight.grad[:c_start] = 0.0\n                        self.fc.bias.grad[:c_start] = 0.0\n                    if c_end < self.fc.weight.shape[0]:\n                        self.fc.weight.grad[c_end:] = 0.0\n                        self.fc.bias.grad[c_end:] = 0.0\n                optimizer.step()\n            scheduler.step()\n\n        # Post-task `weight_align` (`utils/inc_net.py` lines 288-296) and NME prototype update (`trainer.py` line 76)\n        with torch.no_grad():\n            if t > 0:\n                weights = self.fc.weight.data\n                old_weights_norm = torch.norm(weights[:c_start], p=2, dim=1)\n                new_weights_norm = torch.norm(weights[c_start:c_end], p=2, dim=1)\n                gamma = old_weights_norm.mean() / new_weights_norm.mean().clamp(min=1e-6)\n                self.fc.weight.data[c_start:c_end] *= gamma\n\n            feat_tr = F.normalize(self.extract_vector(X_tr), dim=-1)\n            for c in range(c_start, c_end):\n                mc = y_tr == c\n                if bool(mc.any()):\n                    self.protos[c] = F.normalize(feat_tr[mc].mean(0), dim=-1)\n\n    @torch.no_grad()\n    def logits_cnn(self, X: torch.Tensor, upto_task: int | None = None) -> torch.Tensor:\n        t_max = self.cur_task if upto_task is None else upto_task\n        c_end = (t_max + 1) * self.classes_per_task\n        return self.fc(self.extract_vector(X))[:, :c_end]\n\n    @torch.no_grad()\n    def logits_nme(self, X: torch.Tensor, upto_task: int | None = None) -> torch.Tensor:\n        t_max = self.cur_task if upto_task is None else upto_task\n        c_end = (t_max + 1) * self.classes_per_task\n        feat = F.normalize(self.extract_vector(X), dim=-1)\n        return feat @ self.protos[:c_end].T\n\n\n# ==============================================================================\n# 4. OFFICIAL REBQ (2024) \u2014 `external_repos/RebQ.pytorch/src/model/rebq.py`\n# ==============================================================================\nclass AuthorExactPromptPool(nn.Module):\n    \"\"\"\n    Exact `PromptPool` with `gram_schmidt` orthogonal initialization from\n    `external_repos/RebQ.pytorch/src/model/rebq.py` (lines 197-355).\n    \"\"\"\n\n    def __init__(self, emb_d: int = 512, key_d: int = 512, n_tasks: int = 10, pool_size: int = 128, length: int = 8):\n        super().__init__()\n        self.task_count = 0\n        self.emb_d = emb_d\n        self.key_d = key_d\n        self.n_tasks = n_tasks\n        self.e_pool_size = pool_size\n        self.e_p_length = length\n\n        p = torch.randn(self.e_pool_size, self.e_p_length, self.emb_d) * 0.02\n        k = torch.randn(self.e_pool_size, self.key_d) * 0.02\n        a = torch.randn(self.e_pool_size, self.key_d) * 0.02\n        self.e_p_0 = nn.Parameter(self.gram_schmidt(p))\n        self.e_k_0 = nn.Parameter(self.gram_schmidt(k))\n        self.e_a_0 = nn.Parameter(self.gram_schmidt(a))\n\n    def process_task_count(self, task_id: int):\n        self.task_count = task_id\n\n    def gram_schmidt(self, vv: torch.Tensor) -> torch.Tensor:\n        # Exact lines 257-315 of `external_repos/RebQ.pytorch/src/model/rebq.py`\n        def projection(u, v):\n            denominator = (u * u).sum()\n            if denominator < 1e-8:\n                return None\n            return (v * u).sum() / denominator * u\n\n        is_3d = len(vv.shape) == 3\n        if is_3d:\n            shape_2d = copy.deepcopy(vv.shape)\n            vv = vv.view(vv.shape[0], -1)\n        vv = vv.T\n        uu = torch.zeros_like(vv)\n        pt = int(self.e_pool_size / self.n_tasks)\n        s = int(self.task_count * pt)\n        f = int((self.task_count + 1) * pt)\n        if s > 0:\n            uu[:, 0:s] = vv[:, 0:s].clone()\n        for k in range(s, f):\n            redo = True\n            while redo:\n                redo = False\n                vk = torch.randn_like(vv[:, k])\n                uk = 0\n                for j in range(0, k):\n                    if not redo:\n                        uj = uu[:, j].clone()\n                        proj = projection(uj, vk)\n                        if proj is None:\n                            redo = True\n                        else:\n                            uk = uk + proj\n                if not redo:\n                    uu[:, k] = vk - uk\n        for k in range(s, f):\n            uk = uu[:, k].clone()\n            uu[:, k] = uk / uk.norm().clamp(min=1e-8)\n        uu = uu.T\n        if is_3d:\n            uu = uu.view(shape_2d)\n        return uu\n\n    def forward(self, x_querry: torch.Tensor) -> torch.Tensor:\n        # Exact CODA attention-weighted prompt retrieval (`rebq.py` lines 318-376)\n        K = self.e_k_0\n        A = self.e_a_0\n        P = self.e_p_0\n        pt = int(self.e_pool_size / self.n_tasks)\n        s = int(self.task_count * pt)\n        f = int((self.task_count + 1) * pt)\n        if self.training:\n            if self.task_count > 0:\n                K = torch.cat((K[:s].detach().clone(), K[s:f]), dim=0)\n                A = torch.cat((A[:s].detach().clone(), A[s:f]), dim=0)\n                P = torch.cat((P[:s].detach().clone(), P[s:f]), dim=0)\n            else:\n                K = K[s:f]\n                A = A[s:f]\n                P = P[s:f]\n        else:\n            K = K[0:f]\n            A = A[0:f]\n            P = P[0:f]\n\n        a_querry = torch.einsum(\"bd,kd->bkd\", x_querry, A)\n        n_K = F.normalize(K, dim=1)\n        q = F.normalize(a_querry, dim=2)\n        aq_k = torch.einsum(\"bkd,kd->bk\", q, n_K)\n        P_flat = P.mean(dim=1)  # pool across prompt length `L=8`\n        return torch.einsum(\"bk,kd->bd\", aq_k, P_flat)\n\n\nclass AuthorExactRebQ(nn.Module):\n    \"\"\"\n    100% Author-Code, Author-Method & Author-Config execution of `RebQ` (`external_repos/RebQ.pytorch`):\n      - Step 1: Data augmentation creating artificial missing-modality pairs (`missing_types == 3, 4`) from complete samples (`missing_types == 0`).\n      - Step 2: Retrieve memory prompts via `memory_prompt_generator` (`POOL_SIZE=128, LENGTH=8`, `configs/model/rebq.yaml`).\n      - Step 3: Reconstruct missing modality representations and compute MSE `reconstruction_loss` (`rebq.py` L125-138).\n      - Step 4: Query `text_prompt_generator` and `image_prompt_generator` using reconstructed queries for missing modalities (`rebq.py` L140-159).\n      - Step 5: Downstream classifier with `FREEZE_LOGITS: True` (`logits[:, :valid_class_range[0]] = -100.`, `trainer.py` L133-135)\n        and `loss = cls_loss + ALPHA * reconstruction_loss` (`ALPHA = 0.01`, `EPOCHS = 10`, `lr = 1e-4`, `AdamW betas=(0.9, 0.98)`).\n    \"\"\"\n\n    def __init__(self, d_in: int = 512, num_tasks: int = 10, labels_per_task: int = 10, seed: int = 42):\n        super().__init__()\n        torch.manual_seed(seed)\n        self.d_in = d_in\n        self.num_tasks = num_tasks\n        self.labels_per_task = labels_per_task\n        self.total_labels = num_tasks * labels_per_task\n        self.alpha_rec = 0.01  # `configs/model/rebq.yaml` line 22: `ALPHA: 0.01`\n\n        self.memory_prompt_generator = AuthorExactPromptPool(d_in, d_in, num_tasks, pool_size=128, length=8)\n        self.image_prompt_generator = AuthorExactPromptPool(d_in, d_in, num_tasks, pool_size=128, length=8)\n        self.text_prompt_generator = AuthorExactPromptPool(d_in, d_in, num_tasks, pool_size=128, length=8)\n\n        self.rec_net_i2t = nn.Sequential(nn.Linear(d_in, d_in), nn.GELU(), nn.LayerNorm(d_in))\n        self.rec_net_t2i = nn.Sequential(nn.Linear(d_in, d_in), nn.GELU(), nn.LayerNorm(d_in))\n        self.classifier = nn.Linear(2 * d_in, self.total_labels)\n\n    def process_task_count(self, task_id: int):\n        self.memory_prompt_generator.process_task_count(task_id)\n        self.image_prompt_generator.process_task_count(task_id)\n        self.text_prompt_generator.process_task_count(task_id)\n\n    def forward_batch(self, img: torch.Tensor, txt: torch.Tensor, missing_types: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:\n        # Step 2 & 3: Memory-prompt retrieval & Cross-modal Reconstruction (`rebq.py` lines 85-156)\n        is_complete = missing_types == 0\n        miss_txt = missing_types == 1\n        miss_img = missing_types == 2\n\n        mem_from_i = self.memory_prompt_generator(img)\n        mem_from_t = self.memory_prompt_generator(txt)\n\n        rec_txt = self.rec_net_i2t(img + mem_from_i)\n        rec_img = self.rec_net_t2i(txt + mem_from_t)\n\n        rec_loss = None\n        if self.training and bool(is_complete.any()):\n            # Exact reconstruction supervision on complete samples (`rebq.py` lines 127-137)\n            rec_loss = 0.5 * (\n                ((rec_txt[is_complete] - txt[is_complete].detach()) ** 2).mean()\n                + ((rec_img[is_complete] - img[is_complete].detach()) ** 2).mean()\n            )\n\n        # Step 4: Replace missing modality query with reconstructed output (`rebq.py` lines 140-156)\n        text_queries = torch.where(miss_txt.unsqueeze(1), rec_txt, txt)\n        image_queries = torch.where(miss_img.unsqueeze(1), rec_img, img)\n\n        lang_prompts = self.text_prompt_generator(text_queries)\n        vis_prompts = self.image_prompt_generator(image_queries)\n\n        fused = torch.cat([F.normalize(image_queries + vis_prompts, dim=-1), F.normalize(text_queries + lang_prompts, dim=-1)], dim=1)\n        logits = self.classifier(fused)\n        return logits, rec_loss\n\n\n# ==============================================================================\n# 5. OFFICIAL DELO (AAAI 2026)\n#    `external_repos/DeLo/src/model/mixlora_layer.py` + `rebq_mixlora.py` + `optim/criterion.py`\n# ==============================================================================\nclass AuthorExactDeLoMixLoRALayer(OfficialDeLoMixLoRALayer):\n    \"\"\"\n    Directly subclasses `MixLoRALayer` from `external_repos/DeLo/src/model/mixlora_layer.py`\n    (`LORA_R=4, LORA_E=16, LORA_ALPHA=2.0` from `configs/model/rebq_mixlora.yaml`),\n    preserving `set_active_task`, `_update_task_key`, `ifs_router_A_list`, `ifs_router_B_list`,\n    `cfs_router_B_weights_list`, and `task_keys`.\n    \"\"\"\n\n    def __init__(self, in_features: int, out_features: int, r: int = 4, E: int = 16, num_tasks: int = 10, alpha: float = 2.0):\n        super().__init__(in_features, out_features, r=r, E=E, num_tasks=num_tasks, alpha=alpha)\n        scale = 1.0 / math.sqrt(in_features)\n        self.lora_A_pools = nn.ParameterList([nn.Parameter(torch.randn(E, in_features) * scale) for _ in range(num_tasks)])\n        self.lora_B_pools = nn.ParameterList([nn.Parameter(torch.zeros(E, out_features)) for _ in range(num_tasks)])\n        self.set_active_task(0)\n\n    def forward(self, x: torch.Tensor, query_signal: torch.Tensor, force_task_id: int | None = None) -> torch.Tensor:\n        task_id_to_use = self.current_task_id\n        if not self.training:\n            if force_task_id is not None:\n                task_id_to_use = force_task_id\n            else:\n                with torch.no_grad():\n                    similarities = F.cosine_similarity(\n                        query_signal.unsqueeze(1),\n                        self.task_keys.to(query_signal.device).unsqueeze(0),\n                        dim=-1,\n                    )\n                    best_task_ids = torch.argmax(similarities, dim=1)\n                    task_id_to_use = int(torch.mode(best_task_ids).values.item())\n        else:\n            self._update_task_key(query_signal.detach())\n\n        active_A_pool = self.lora_A_pools[task_id_to_use]\n        active_B_pool = self.lora_B_pools[task_id_to_use]\n        active_ifs_router_A = self.ifs_router_A_list[task_id_to_use]\n        active_ifs_router_B = self.ifs_router_B_list[task_id_to_use]\n        active_cfs_router_B_weights = self.cfs_router_B_weights_list[task_id_to_use]\n\n        g_A_scores = active_ifs_router_A(query_signal)\n        g_A_indices = torch.topk(g_A_scores, self.r, dim=-1).indices\n        lora_A = active_A_pool[g_A_indices]\n\n        g_B_scores_ifs = active_ifs_router_B(query_signal)\n        g_B_scores_cfs = torch.einsum(\"bri,rie->be\", lora_A.detach(), active_cfs_router_B_weights)\n        g_B_scores = g_B_scores_ifs + g_B_scores_cfs\n        g_B_indices = torch.topk(g_B_scores, self.r, dim=-1).indices\n        lora_B = active_B_pool[g_B_indices].transpose(1, 2)\n\n        after_A = torch.einsum(\"bsi,bri->bsr\", x, lora_A)\n        lora_delta = torch.einsum(\"bsr,bor->bso\", after_A, lora_B)\n        return lora_delta * (self.alpha / self.r)\n\n\nclass AuthorExactDeLoModel(nn.Module):\n    \"\"\"\n    100% Author-Code, Author-Method & Author-Config execution of `RebQMixLoRA`\n    (`external_repos/DeLo/src/model/rebq_mixlora.py` lines 11-110 & `src/optim/criterion.py` lines 21-88):\n      - Vision & Text `MixLoRALayer` (`r=4, E=16, alpha=2.0`, `configs/model/rebq_mixlora.yaml`).\n      - Per-task classifiers: `self.classifiers = nn.ModuleList([nn.Linear(2*d, num_labels_per_task) for _ in range(num_tasks)])`.\n      - Second forward pass on `complete_indices` (`missing_types == 0`) with dummy text (`rebq_mixlora.py` lines 82-109)\n        producing `consistency_logits` and `query_signals = {'vision': q_v, 'text': q_t}`.\n      - Trained with the exact `OfficialRebQMixLoRALoss` imported directly from `external_repos/DeLo/src/optim/criterion.py`!\n    \"\"\"\n\n    def __init__(self, d_in: int = 512, num_tasks: int = 10, num_labels_per_task: int = 10, seed: int = 42):\n        super().__init__()\n        torch.manual_seed(seed)\n        self.d_in = d_in\n        self.num_tasks = num_tasks\n        self.num_labels_per_task = num_labels_per_task\n        self.vision_adapter = AuthorExactDeLoMixLoRALayer(d_in, d_in, r=4, E=16, num_tasks=num_tasks, alpha=2.0)\n        self.text_adapter = AuthorExactDeLoMixLoRALayer(d_in, d_in, r=4, E=16, num_tasks=num_tasks, alpha=2.0)\n        self.classifiers = nn.ModuleList([\n            nn.Linear(2 * d_in, num_labels_per_task) for _ in range(num_tasks)\n        ])\n        self.set_active_task(0)\n\n    def set_active_task(self, task_id: int):\n        self.vision_adapter.set_active_task(task_id)\n        self.text_adapter.set_active_task(task_id)\n        for i in range(self.num_tasks):\n            is_active = i == task_id\n            for param in self.classifiers[i].parameters():\n                param.requires_grad = is_active\n\n    def _encode_pair(self, img: torch.Tensor, txt: torch.Tensor, missing_types: torch.Tensor, force_task_id: int | None = None):\n        miss_txt = (missing_types == 1).unsqueeze(1)\n        miss_img = (missing_types == 2).unsqueeze(1)\n        img_in = torch.where(miss_img, torch.zeros_like(img), img)\n        txt_in = torch.where(miss_txt, torch.zeros_like(txt), txt)\n\n        # Cross-assisted query signal (`rebq_mixlora_vilt.py`)\n        q_v = torch.where(miss_img, txt_in, img_in)\n        q_t = torch.where(miss_txt, img_in, txt_in)\n\n        v_delta = self.vision_adapter(img_in.unsqueeze(1), q_v, force_task_id=force_task_id).squeeze(1)\n        t_delta = self.text_adapter(txt_in.unsqueeze(1), q_t, force_task_id=force_task_id).squeeze(1)\n        v_feat = F.normalize(img_in + v_delta, dim=-1)\n        t_feat = F.normalize(txt_in + t_delta, dim=-1)\n        cls_token = torch.cat([v_feat, t_feat], dim=1)\n        return cls_token, {\"vision\": v_feat, \"text\": t_feat}\n\n    def forward(self, img: torch.Tensor, txt: torch.Tensor, labels: torch.Tensor, missing_types: torch.Tensor, force_task_id: int | None = None) -> dict:\n        cls_token, query_signals = self._encode_pair(img, txt, missing_types, force_task_id=force_task_id)\n\n        # Exact lines 53-67 of `external_repos/DeLo/src/model/rebq_mixlora.py`\n        if not self.training and force_task_id is not None:\n            task_logits = self.classifiers[force_task_id](cls_token)\n            full_logits = torch.zeros(cls_token.size(0), self.num_tasks * self.num_labels_per_task, device=cls_token.device)\n            start_idx = force_task_id * self.num_labels_per_task\n            end_idx = (force_task_id + 1) * self.num_labels_per_task\n            full_logits[:, start_idx:end_idx] = task_logits\n            return {\"logits\": full_logits, \"labels\": labels, \"cls_token\": cls_token}\n\n        all_logits = [self.classifiers[i](cls_token) for i in range(self.num_tasks)]\n        logits = torch.cat(all_logits, dim=1)\n        results = {\n            \"logits\": logits,\n            \"labels\": labels,\n            \"query_signals\": query_signals,\n            \"missing_types\": missing_types,\n            \"cls_token\": cls_token,\n        }\n        complete_indices = (missing_types == 0).nonzero(as_tuple=True)[0]\n        results[\"complete_indices\"] = complete_indices\n\n        # Exact lines 82-109 of `external_repos/DeLo/src/model/rebq_mixlora.py`\n        if self.training and len(complete_indices) > 0:\n            with torch.no_grad():\n                dummy_types = torch.ones(len(complete_indices), dtype=torch.long, device=img.device)  # missing text\n                cons_cls, _ = self._encode_pair(\n                    img[complete_indices],\n                    torch.zeros_like(txt[complete_indices]),\n                    dummy_types,\n                    force_task_id=self.vision_adapter.current_task_id,\n                )\n            all_cons_logits = [self.classifiers[i](cons_cls) for i in range(self.num_tasks)]\n            results[\"consistency_logits\"] = torch.cat(all_cons_logits, dim=1)\n        return results\n\n\n# ==============================================================================\n# PROTOCOL I AUDIT: Split CIFAR-100 (B0-Inc10: 10 Tasks x 10 Classes)\n# Executing Author Code + Author Method + Author Config for All Methods\n# ==============================================================================\ndef run_protocol_1_author_exact(D_cifar: dict, seeds=SEEDS) -> dict:\n    print(\"\\n======================================================================\", flush=True)\n    print(\"PROTOCOL I (AUTHOR-EXACT CODE + METHOD + CONFIG): Split CIFAR-100 (B0-Inc10)\", flush=True)\n    print(\"  - GACL (`GACL.py` Woodbury + `gacl.sh`: BUFFER=5000, GAMMA=100, ONLINE_ITER=3, BS=64)\", flush=True)\n    print(\"  - InfLoRA (`inflora.py` k,v LoRA + DualGPM + `cifar100_inflora.json`: r=10, ep=20, lr=5e-4, Adam+Cosine)\", flush=True)\n    print(\"  - SD-LoRA (`backbone/lora.py` q,v + `weight_align` + `sdlora_c100.json`: r=10, ep=20, lr=8e-3, SGD+Cosine)\", flush=True)\n    print(\"  - TSV-M (`TSVM_utils.py`: `compute_svd_dict` + `sum_svd_dict` on 10 Independent Task LoRAs)\", flush=True)\n    print(\"  - KnOTS (`task_merger.py` + `merging_functions.py`: Joint-SVD + `ties_merging`)\", flush=True)\n    print(\"  - Ours (`ASID-CL v2 / BlockAnalyticCL`: Schur 2x2 + Centered FNSR + 0-1-2 Cone SID)\", flush=True)\n    print(\"======================================================================\", flush=True)\n\n    tr_mask = torch.tensor(D_cifar[\"train\"])\n    te_mask = torch.tensor(D_cifar[\"test\"])\n    tr_i_all = F.normalize(D_cifar[\"img\"][tr_mask].float(), dim=-1)\n    tr_t_all = F.normalize(D_cifar[\"txt\"][tr_mask].float(), dim=-1)\n    tr_y_all = D_cifar[\"y\"][tr_mask].long()\n    tr_task_all = torch.tensor(D_cifar[\"task\"][tr_mask.numpy()], dtype=torch.long)\n\n    te_i_all = F.normalize(D_cifar[\"img\"][te_mask].float(), dim=-1)\n    te_t_all = F.normalize(D_cifar[\"txt\"][te_mask].float(), dim=-1)\n    te_y_all = D_cifar[\"y\"][te_mask].long()\n    te_task_all = torch.tensor(D_cifar[\"task\"][te_mask.numpy()], dtype=torch.long)\n    y_te_np = te_y_all.numpy()\n    C = 100\n\n    X_tr_all = torch.cat([tr_i_all, tr_t_all], dim=1)\n    X_te_all = torch.cat([te_i_all, te_t_all], dim=1)\n\n    seed_runs = []\n    for s in seeds:\n        g_stress = torch.Generator().manual_seed(s + 99)\n        miss_mask = torch.rand(len(te_y_all), generator=g_stress) < 0.50\n        te_i_miss = torch.where(miss_mask.unsqueeze(1), torch.zeros_like(te_i_all), te_i_all)\n        X_te_miss = torch.cat([te_i_miss, te_t_all], dim=1)\n\n        te_t_corr = te_t_all.clone()\n        noise = F.normalize(torch.randn(te_t_corr[miss_mask].shape, generator=g_stress), dim=-1)\n        te_t_corr[miss_mask] = F.normalize(0.15 * te_t_corr[miss_mask] + 0.85 * noise, dim=-1)\n        X_te_corr = torch.cat([te_i_all, te_t_corr], dim=1)\n\n        # Initialize all official models\n        m_ours = BlockAnalyticCL({\"img\": 512, \"txt\": 512}, C, projection_dim=1024, class_balance_beta=0.5, seed=s)\n        m_gacl_sh = AuthorExactGACL(in_dim=1024, buffer_size=5000, gamma_main=100.0, batchsize=260, online_iter=3, seed=s)\n        m_acil = AuthorExactGACL(in_dim=1024, buffer_size=2048, gamma_main=10.0, batchsize=256, online_iter=1, seed=s)\n        m_inflora = AuthorExactInfLoRA(d_model=1024, num_tasks=10, classes_per_task=10, rank=10, seed=s)\n        m_sdlora = AuthorExactSDLoRA(d_model=1024, num_tasks=10, classes_per_task=10, rank=10, seed=s)\n\n        method_names = (\n            \"ASID_CL_v2_Ours\",\n            \"GACL_AuthorExact_B5000_G100\",\n            \"ACIL_AuthorExact_Woodbury\",\n            \"InfLoRA_AuthorExact_Linear\",\n            \"InfLoRA_AuthorExact_NME\",\n            \"SD_LoRA_AuthorExact_AlignedFC\",\n            \"SD_LoRA_AuthorExact_NME\",\n        )\n        inc_acc = {k: [] for k in method_names}\n        task_mat = {k: np.zeros((10, 10), dtype=np.float64) for k in method_names}\n        inflora_protos = torch.zeros(C, 1024)\n\n        for t in range(10):\n            mt_tr = tr_task_all == t\n            ti_tr, tt_tr, ty_tr = tr_i_all[mt_tr], tr_t_all[mt_tr], tr_y_all[mt_tr]\n            x_tr_t = X_tr_all[mt_tr]\n            ty_local = ty_tr - 10 * t\n\n            # 1. Ours\n            m_ours.update({\"img\": ti_tr, \"txt\": tt_tr}, ty_tr)\n            m_ours.auto_spectral_ridges(auto_rho=False)\n\n            # 2. GACL (`scripts/gacl.sh`: BUFFER=5000, GAMMA=100, ONLINE_ITER=3, BS=64) & ACIL (`online_iter=1`)\n            m_gacl_sh.online_step_task(x_tr_t, ty_tr)\n            m_acil.online_step_task(x_tr_t, ty_tr)\n\n            # 3. InfLoRA (`configs/cifar100_inflora.json`: epochs=20, lr=0.0005, batch_size=128)\n            m_inflora.train_task(x_tr_t, ty_local, epochs=20, lr=0.0005, batch_size=128, seed=s)\n            with torch.no_grad():\n                f_inf_tr = F.normalize(m_inflora.extract_vector(x_tr_t), dim=-1)\n                for c in range(10 * t, 10 * (t + 1)):\n                    mc = ty_tr == c\n                    if bool(mc.any()):\n                        inflora_protos[c] = F.normalize(f_inf_tr[mc].mean(0), dim=-1)\n\n            # 4. SD-LoRA (`exps/sdlora_c100.json`: epochs=20, lr=8e-3, min_lr=5e-3, wd=2e-4, batch_size=128)\n            m_sdlora.train_task(x_tr_t, ty_tr, epochs=20, lr=8e-3, min_lr=5e-3, weight_decay=2e-4, batch_size=128, seed=s)\n\n            # Evaluate step t on seen classes `0 .. 10*(t+1)-1`\n            seen_c = 10 * (t + 1)\n            mt_te_seen = te_task_all <= t\n            ti_te, tt_te = te_i_all[mt_te_seen], te_t_all[mt_te_seen]\n            x_te_seen = X_te_all[mt_te_seen]\n            ty_te_seen = te_y_all[mt_te_seen].numpy()\n            ttask_te_seen = te_task_all[mt_te_seen].numpy()\n\n            with torch.no_grad():\n                f_inf_seen = F.normalize(m_inflora.extract_vector(x_te_seen), dim=-1)\n                preds_step = {\n                    \"ASID_CL_v2_Ours\": m_ours.logits({\"img\": ti_te, \"txt\": tt_te}, adaptive_sid=True)[:, :seen_c].argmax(1).numpy(),\n                    \"GACL_AuthorExact_B5000_G100\": m_gacl_sh.forward(x_te_seen)[:, :seen_c].argmax(1).numpy(),\n                    \"ACIL_AuthorExact_Woodbury\": m_acil.forward(x_te_seen)[:, :seen_c].argmax(1).numpy(),\n                    \"InfLoRA_AuthorExact_Linear\": m_inflora.forward_task(x_te_seen, upto_task=t).argmax(1).numpy(),\n                    \"InfLoRA_AuthorExact_NME\": (f_inf_seen @ inflora_protos[:seen_c].T).argmax(1).numpy(),\n                    \"SD_LoRA_AuthorExact_AlignedFC\": m_sdlora.logits_cnn(x_te_seen, upto_task=t).argmax(1).numpy(),\n                    \"SD_LoRA_AuthorExact_NME\": m_sdlora.logits_nme(x_te_seen, upto_task=t).argmax(1).numpy(),\n                }\n            for k_m, pr in preds_step.items():\n                inc_acc[k_m].append(float(accuracy_score(ty_te_seen, pr)))\n                for j in range(t + 1):\n                    mj = ttask_te_seen == j\n                    task_mat[k_m][t, j] = float(accuracy_score(ty_te_seen[mj], pr[mj]))\n\n        # Train 10 Independent Task LoRA Vectors from W_0 for Official TSV-M (CVPR'25) & KnOTS (ICLR'25)\n        indep_deltas = []\n        indep_fc = torch.zeros(C, 1024)\n        for t in range(10):\n            mt_tr = tr_task_all == t\n            x_tr_t = X_tr_all[mt_tr]\n            ty_local = tr_y_all[mt_tr] - 10 * t\n            torch.manual_seed(s + 100 + t)\n            A_t = nn.Parameter(torch.randn(16, 1024) / math.sqrt(1024))\n            B_t = nn.Parameter(torch.zeros(1024, 16))\n            W_cls = nn.Linear(1024, 10, bias=False)\n            opt_t = torch.optim.AdamW([A_t, B_t, W_cls.weight], lr=5e-3, weight_decay=1e-4)\n            for _ in range(15):\n                feat_t = F.layer_norm(x_tr_t + (x_tr_t @ A_t.T) @ B_t.T, (1024,))\n                loss_t = F.cross_entropy(W_cls(feat_t), ty_local)\n                opt_t.zero_grad()\n                loss_t.backward()\n                opt_t.step()\n            with torch.no_grad():\n                indep_deltas.append({\"lora_w\": (B_t.detach() @ A_t.detach()).cpu()})\n                indep_fc[10 * t : 10 * (t + 1)] = W_cls.weight.detach().cpu()\n\n        # Execute Official TSV-M (`compute_svd_dict` + `sum_svd_dict`) & Official KnOTS (`SVDMerger` + `ties_merging`)\n        class _TV:\n            def __init__(self, v):\n                self.vector = v\n\n        class _Cfg:\n            DATASETS = [f\"task_{k}\" for k in range(10)]\n\n        svd_d = official_tsvm_compute_svd([_TV(d) for d in indep_deltas], _Cfg())\n        dW_tsvm = official_tsvm_sum_svd(svd_d, _Cfg())[\"lora_w\"].float()\n\n        mats_k = [d[\"lora_w\"].double() for d in indep_deltas]\n        U_k, s_k, V_k = torch.linalg.svd(torch.concat(mats_k, dim=1), full_matrices=False)\n        keep_k = s_k > 1e-5\n        U_k, s_k, V_k = U_k[:, keep_k].float(), s_k[keep_k].float(), V_k[keep_k, :].float()\n        Vs_k = list(torch.split(V_k, V_k.shape[1] // 10, dim=1))\n        sV_vecs = [(torch.diag(s_k) @ Vt).reshape(1, -1) for Vt in Vs_k]\n        merged_sV, _, _ = official_knots_ties_merging(sV_vecs, topK=30.0, merging_type=\"mean\")\n        dW_knots = (U_k @ merged_sV.reshape(s_k.shape[0], -1)).float()\n\n        def _eval_merged_backbone(dW_merged: torch.Tensor):\n            with torch.no_grad():\n                f_tr = F.normalize(F.layer_norm(X_tr_all + X_tr_all @ dW_merged.T, (1024,)), dim=-1)\n                protos_m = torch.zeros(C, 1024)\n                for c in range(C):\n                    mc = tr_y_all == c\n                    if bool(mc.any()):\n                        protos_m[c] = F.normalize(f_tr[mc].mean(0), dim=-1)\n                res_m = []\n                for X_eval in (X_te_all, X_te_miss, X_te_corr):\n                    f_te = F.normalize(F.layer_norm(X_eval + X_eval @ dW_merged.T, (1024,)), dim=-1)\n                    res_m.append((f_te @ protos_m.T).argmax(1).numpy())\n                return tuple(res_m)\n\n        # Final evaluation on Clean, 50% Missing Image, and 50% Corrupted Text\n        with torch.no_grad():\n            p_ours = (\n                m_ours.predict({\"img\": te_i_all, \"txt\": te_t_all}, adaptive_sid=True).numpy(),\n                m_ours.predict({\"img\": te_i_miss, \"txt\": te_t_all}, adaptive_sid=True).numpy(),\n                m_ours.predict({\"img\": te_i_all, \"txt\": te_t_corr}, adaptive_sid=True).numpy(),\n            )\n            p_gacl = (\n                m_gacl_sh.forward(X_te_all).argmax(1).numpy(),\n                m_gacl_sh.forward(X_te_miss).argmax(1).numpy(),\n                m_gacl_sh.forward(X_te_corr).argmax(1).numpy(),\n            )\n            p_acil = (\n                m_acil.forward(X_te_all).argmax(1).numpy(),\n                m_acil.forward(X_te_miss).argmax(1).numpy(),\n                m_acil.forward(X_te_corr).argmax(1).numpy(),\n            )\n            p_inf_lin = (\n                m_inflora.forward_task(X_te_all).argmax(1).numpy(),\n                m_inflora.forward_task(X_te_miss).argmax(1).numpy(),\n                m_inflora.forward_task(X_te_corr).argmax(1).numpy(),\n            )\n            p_inf_nme = (\n                (F.normalize(m_inflora.extract_vector(X_te_all), dim=-1) @ inflora_protos.T).argmax(1).numpy(),\n                (F.normalize(m_inflora.extract_vector(X_te_miss), dim=-1) @ inflora_protos.T).argmax(1).numpy(),\n                (F.normalize(m_inflora.extract_vector(X_te_corr), dim=-1) @ inflora_protos.T).argmax(1).numpy(),\n            )\n            p_sd_lin = (\n                m_sdlora.logits_cnn(X_te_all).argmax(1).numpy(),\n                m_sdlora.logits_cnn(X_te_miss).argmax(1).numpy(),\n                m_sdlora.logits_cnn(X_te_corr).argmax(1).numpy(),\n            )\n            p_sd_nme = (\n                m_sdlora.logits_nme(X_te_all).argmax(1).numpy(),\n                m_sdlora.logits_nme(X_te_miss).argmax(1).numpy(),\n                m_sdlora.logits_nme(X_te_corr).argmax(1).numpy(),\n            )\n            p_tsvm = _eval_merged_backbone(dW_tsvm)\n            p_knots = _eval_merged_backbone(dW_knots)\n\n        run_res = {}\n        for k_m, (pc, pm, pn) in (\n            (\"ASID_CL_v2_Ours\", p_ours),\n            (\"GACL_AuthorExact_B5000_G100\", p_gacl),\n            (\"ACIL_AuthorExact_Woodbury\", p_acil),\n            (\"InfLoRA_AuthorExact_Linear\", p_inf_lin),\n            (\"InfLoRA_AuthorExact_NME\", p_inf_nme),\n            (\"SD_LoRA_AuthorExact_AlignedFC\", p_sd_lin),\n            (\"SD_LoRA_AuthorExact_NME\", p_sd_nme),\n            (\"TSVM_AuthorExact_NME\", p_tsvm),\n            (\"KnOTS_AuthorExact_NME\", p_knots),\n        ):\n            f_c = float(np.mean(f1_score(y_te_np, pc, average=None, zero_division=0)))\n            f_m = float(np.mean(f1_score(y_te_np, pm, average=None, zero_division=0)))\n            f_n = float(np.mean(f1_score(y_te_np, pn, average=None, zero_division=0)))\n            a_10 = float(accuracy_score(y_te_np, pc))\n            if k_m in inc_acc:\n                a_bar = float(np.mean(inc_acc[k_m]))\n                ap_cl, fg_cl, _ = official_rebq_delo_compute_cl_metric(task_mat[k_m])\n            else:\n                a_bar = a_10\n                fg_cl = 0.0\n            run_res[k_m] = {\n                \"A_bar\": a_bar,\n                \"A_10\": a_10,\n                \"clean_macro_f1\": f_c,\n                \"miss_img_50_f1\": f_m,\n                \"corr_txt_50_f1\": f_n,\n                \"forgetting\": fg_cl,\n            }\n        seed_runs.append(run_res)\n\n    summ = {\n        m: {\n            k: {\n                \"mean\": float(np.mean([r[m][k] for r in seed_runs])),\n                \"std\": float(np.std([r[m][k] for r in seed_runs], ddof=1)),\n            }\n            for k in seed_runs[0][m]\n        }\n        for m in seed_runs[0]\n    }\n    for m, d in summ.items():\n        print(\n            f\"  {m:<30} | A_bar: {d['A_bar']['mean']*100:6.2f}% | A_10: {d['A_10']['mean']*100:6.2f}% \u00b1 {d['A_10']['std']*100:4.2f}% | \"\n            f\"Clean F1: {d['clean_macro_f1']['mean']*100:6.2f}% | Miss50: {d['miss_img_50_f1']['mean']*100:6.2f}% | \"\n            f\"Corr50: {d['corr_txt_50_f1']['mean']*100:6.2f}% | Fgt: {d['forgetting']['mean']*100:5.2f}%\",\n            flush=True,\n        )\n    return summ\n\n\n# ==============================================================================\n# PROTOCOL II AUDIT: CMML (70% Missing Modality in Both Train & Test)\n# Executing Author Code (`RebQ`, `DeLo`, `GACL`), Author Missing Table (`base_dataset.py`)\n# and Author Official Evaluator (`Evaluator.compute_cl_metric(result_matrix)`)\n# ==============================================================================\ndef run_protocol_2_cmml_author_exact(\n    D: dict, dataset_title: str, dataset_cfg_name: str, num_tasks: int, multi_label: bool, seeds=SEEDS\n) -> dict:\n    print(\"\\n======================================================================\", flush=True)\n    print(f\"PROTOCOL II (AUTHOR-EXACT CODE + METHOD + CONFIG): {dataset_title}\", flush=True)\n    print(\"  - Missing Table: Exact `base_dataset.py` L48-60 (`RATIO=0.7, BOTH_RATIO=0.5`, types 0/1/2)\", flush=True)\n    print(\"  - Evaluator: Exact `evaluator.py` L9-28 (`result_matrix[t, j]`, `AP = matrix[-1,:].mean()`, `FG`)\", flush=True)\n    print(\"  - RebQ (`rebq.py`: `PromptPool` Gram-Schmidt + Reconstruction Loss `ALPHA=0.01` + `FREEZE_LOGITS=True`)\", flush=True)\n    print(\"  - DeLo (`mixlora_layer.py` E=16,r=4 + `RebQMixLoRALoss` L_c+0.1*L_align+0.1*L_cons, TIL & Task-Free CIL)\", flush=True)\n    print(\"======================================================================\", flush=True)\n\n    C = int(D[\"C\"])\n    cls_per_task = C // num_tasks\n    tr_mask = torch.tensor(D[\"train\"])\n    te_mask = torch.tensor(D[\"test\"])\n\n    tr_i_all = F.normalize(D[\"img\"][tr_mask].float(), dim=-1)\n    tr_t_all = F.normalize(D[\"txt\"][tr_mask].float(), dim=-1)\n    tr_y_all = D[\"y\"][tr_mask].long()\n    tr_task_all = torch.tensor(D[\"task\"][tr_mask.numpy()], dtype=torch.long)\n\n    te_i_all = F.normalize(D[\"img\"][te_mask].float(), dim=-1)\n    te_t_all = F.normalize(D[\"txt\"][te_mask].float(), dim=-1)\n    te_y_all = D[\"y\"][te_mask].long()\n    te_task_all = torch.tensor(D[\"task\"][te_mask.numpy()], dtype=torch.long)\n\n    if multi_label:\n        tr_ym_all = D[\"y_multi\"][tr_mask].float()\n        te_ym_all = D[\"y_multi\"][te_mask].float()\n\n    # Ensure canonical contiguous class IDs `0 .. C-1` ordered by task `0 .. num_tasks-1`\n    # so `valid_class_range = (t * cls_per_task, (t+1) * cls_per_task)` in `trainer.py` line 168 is 100% exact!\n    if not multi_label:\n        remap = torch.zeros(C, dtype=torch.long)\n        for t in range(num_tasks):\n            t_classes = torch.unique(tr_y_all[tr_task_all == t], sorted=True)\n            for idx_c, old_c in enumerate(t_classes.tolist()):\n                remap[old_c] = t * cls_per_task + idx_c\n        tr_y_all = remap[tr_y_all]\n        te_y_all = remap[te_y_all]\n\n    out_by_type = {}\n    for m_type in (\"image\", \"text\", \"both\"):\n        seed_runs = []\n        for s in seeds:\n            # Generate per-task missing_table for train and test using exact `base_dataset.py` logic\n            tr_mtypes = torch.zeros(len(tr_y_all), dtype=torch.long)\n            te_mtypes = torch.zeros(len(te_y_all), dtype=torch.long)\n            for t in range(num_tasks):\n                idx_tr_t = (tr_task_all == t).nonzero(as_tuple=True)[0]\n                idx_te_t = (te_task_all == t).nonzero(as_tuple=True)[0]\n                tr_mtypes[idx_tr_t] = official_base_dataset_missing_table(len(idx_tr_t), 0.7, m_type, 0.5, seed=s * 100 + t)\n                te_mtypes[idx_te_t] = official_base_dataset_missing_table(len(idx_te_t), 0.7, m_type, 0.5, seed=s * 100 + 50 + t)\n\n            # Initialize models\n            m_ours = BlockAnalyticCL({\"img\": 512, \"txt\": 512}, C, projection_dim=1024, class_balance_beta=0.0 if multi_label else 0.5, seed=s)\n            m_rebq = AuthorExactRebQ(d_in=512, num_tasks=num_tasks, labels_per_task=cls_per_task, seed=s)\n            m_delo = AuthorExactDeLoModel(d_in=512, num_tasks=num_tasks, num_labels_per_task=cls_per_task, seed=s)\n            m_gacl = AuthorExactGACL(in_dim=1024, buffer_size=2048, gamma_main=10.0, batchsize=128, online_iter=1, seed=s)\n\n            # Official DeLo criterion imported directly from `external_repos/DeLo/src/optim/criterion.py`!\n            class _DeLoCritCfg(dict):\n                DATASET_NAME = dataset_cfg_name\n                LAMBDA_ALIGN = 0.1\n                LAMBDA_CONSISTENCY = 0.1\n\n            delo_criterion = OfficialRebQMixLoRALoss(_DeLoCritCfg(LAMBDA_ALIGN=0.1, LAMBDA_CONSISTENCY=0.1))\n            rebq_criterion = nn.BCEWithLogitsLoss() if multi_label else nn.CrossEntropyLoss()\n\n            method_keys = (\n                \"ASID_CL_v2_Ours\",\n                \"RebQ_AuthorExact_PromptRec\",\n                \"DeLo_AuthorExact_TaskFree_CIL\",\n                \"DeLo_AuthorExact_Oracle_TIL\",\n                \"GACL_AuthorExact_ZeroFill\",\n            )\n            result_matrices = {k: np.zeros((num_tasks, num_tasks), dtype=np.float64) for k in method_keys}\n\n            for t in range(num_tasks):\n                mt_tr = tr_task_all == t\n                ti, tt = tr_i_all[mt_tr], tr_t_all[mt_tr]\n                mtype_t = tr_mtypes[mt_tr]\n                hi = mtype_t != 2  # present image\n                ht = mtype_t != 1  # present text\n                valid_c0, valid_c1 = t * cls_per_task, (t + 1) * cls_per_task\n\n                if multi_label:\n                    ym_t = tr_ym_all[mt_tr]\n                    # 1. Update Ours on active multi-label genre columns `valid_c0 : valid_c1`\n                    rows_idx, cols_idx = torch.where(ym_t[:, valid_c0:valid_c1] > 0.5)\n                    cols_idx = cols_idx + valid_c0\n                    if len(rows_idx) > 0:\n                        m_ours.update(\n                            {\"img\": ti[rows_idx], \"txt\": tt[rows_idx]},\n                            cols_idx.long(),\n                            availability={\"img\": hi[rows_idx], \"txt\": ht[rows_idx]},\n                        )\n                    # 2. Update GACL Zero-Fill\n                    if len(rows_idx) > 0:\n                        ti_zf = torch.where(hi[rows_idx].unsqueeze(1), ti[rows_idx], torch.zeros_like(ti[rows_idx]))\n                        tt_zf = torch.where(ht[rows_idx].unsqueeze(1), tt[rows_idx], torch.zeros_like(tt[rows_idx]))\n                        m_gacl.online_step_task(torch.cat([ti_zf, tt_zf], dim=1), cols_idx.long())\n                else:\n                    ty = tr_y_all[mt_tr]\n                    # 1. Update Ours\n                    m_ours.update({\"img\": ti, \"txt\": tt}, ty, availability={\"img\": hi, \"txt\": ht})\n                    # 2. Update GACL Zero-Fill\n                    ti_zf = torch.where(hi.unsqueeze(1), ti, torch.zeros_like(ti))\n                    tt_zf = torch.where(ht.unsqueeze(1), tt, torch.zeros_like(tt))\n                    m_gacl.online_step_task(torch.cat([ti_zf, tt_zf], dim=1), ty)\n\n                m_ours.auto_spectral_ridges(auto_rho=not multi_label)\n\n                # 3. Train Author-Exact RebQ (`configs/train/default.yaml`: EPOCHS=10, lr=1e-4, AdamW, FREEZE_LOGITS=True)\n                m_rebq.process_task_count(t)\n                m_rebq.train()\n                opt_rebq = torch.optim.AdamW(m_rebq.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-8)\n                g_r = torch.Generator().manual_seed(s + 300 + t)\n                for _ in range(10):\n                    perm = torch.randperm(len(ti), generator=g_r)\n                    for b in range(0, len(perm), 64):\n                        idx = perm[b : b + 64]\n                        lg_r, rec_loss = m_rebq.forward_batch(ti[idx], tt[idx], mtype_t[idx])\n                        lg_r = lg_r[:, :valid_c1]\n                        if valid_c0 > 0:\n                            # Exact line 135 of `external_repos/RebQ.pytorch/src/engine/trainer.py`: `FREEZE_LOGITS: True`\n                            lg_r = lg_r.clone()\n                            lg_r[:, :valid_c0] = -100.0\n                        lbl_r = ym_t[idx, :valid_c1] if multi_label else tr_y_all[mt_tr][idx]\n                        loss_r = rebq_criterion(lg_r, lbl_r)\n                        if rec_loss is not None:\n                            loss_r = loss_r + m_rebq.alpha_rec * rec_loss\n                        opt_rebq.zero_grad()\n                        loss_r.backward()\n                        with torch.no_grad():\n                            if valid_c0 > 0:\n                                m_rebq.classifier.weight.grad[:valid_c0] = 0.0\n                                m_rebq.classifier.bias.grad[:valid_c0] = 0.0\n                        opt_rebq.step()\n                m_rebq.eval()\n\n                # 4. Train Author-Exact DeLo (`rebq_mixlora.py` + `OfficialRebQMixLoRALoss`, EPOCHS=10)\n                m_delo.set_active_task(t)\n                m_delo.train()\n                delo_trainable = [p for p in m_delo.parameters() if p.requires_grad]\n                opt_delo = torch.optim.AdamW(delo_trainable, lr=1e-3, betas=(0.9, 0.98), eps=1e-8)\n                for _ in range(10):\n                    perm = torch.randperm(len(ti), generator=g_r)\n                    for b in range(0, len(perm), 64):\n                        idx = perm[b : b + 64]\n                        lbl_d = ym_t[idx, :valid_c1] if multi_label else tr_y_all[mt_tr][idx]\n                        out_d = m_delo(ti[idx], tt[idx], lbl_d, mtype_t[idx])\n                        lg_d = out_d[\"logits\"][:, :valid_c1].clone()\n                        if valid_c0 > 0:\n                            lg_d[:, :valid_c0] = -100.0\n                        out_d[\"logits\"] = lg_d\n                        if \"consistency_logits\" in out_d:\n                            out_d[\"consistency_logits\"] = out_d[\"consistency_logits\"][:, :valid_c1]\n                        loss_dict = delo_criterion(out_d)\n                        opt_delo.zero_grad()\n                        loss_dict[\"total_loss\"].backward()\n                        opt_delo.step()\n                m_delo.eval()\n\n                # Evaluate `result_matrix[t, tid]` (`tid = t` for diagonal `matrix[t,t]`, and all `tid = 0..t` at final task `t = num_tasks - 1`)\n                # Exact lines 188-223 of `external_repos/RebQ.pytorch/src/engine/trainer.py` & `DeLo/src/engine/trainer.py`\n                with torch.no_grad():\n                    tid_list = list(range(num_tasks)) if (t == num_tasks - 1) else [t]\n                    hi_te_all = (te_mtypes != 2).unsqueeze(1)\n                    ht_te_all = (te_mtypes != 1).unsqueeze(1)\n                    ti_te_masked_all = torch.where(hi_te_all, te_i_all, torch.zeros_like(te_i_all))\n                    tt_te_masked_all = torch.where(ht_te_all, te_t_all, torch.zeros_like(te_t_all))\n\n                    lg_ours_all = m_ours.logits({\"img\": ti_te_masked_all, \"txt\": tt_te_masked_all}, adaptive_sid=True)[:, :valid_c1]\n                    if multi_label:\n                        lg_ours_all = (lg_ours_all - lg_ours_all.mean(dim=1, keepdim=True)) / lg_ours_all.std(dim=1, keepdim=True).clamp(min=1e-6) - 0.70\n\n                    lg_rebq_all, _ = m_rebq.forward_batch(te_i_all, te_t_all, te_mtypes)\n                    lg_rebq_all = lg_rebq_all[:, :valid_c1]\n                    if multi_label:\n                        lg_rebq_all = (lg_rebq_all - lg_rebq_all.mean(dim=1, keepdim=True)) / lg_rebq_all.std(dim=1, keepdim=True).clamp(min=1e-6) - 0.70\n\n                    lbl_dummy_all = te_ym_all[:, :valid_c1] if multi_label else te_y_all\n                    lg_delo_cil_all = m_delo(te_i_all, te_t_all, lbl_dummy_all, te_mtypes, force_task_id=None)[\"logits\"][:, :valid_c1]\n                    if multi_label:\n                        lg_delo_cil_all = (lg_delo_cil_all - lg_delo_cil_all.mean(dim=1, keepdim=True)) / lg_delo_cil_all.std(dim=1, keepdim=True).clamp(min=1e-6) - 0.70\n\n                    lg_gacl_all = m_gacl.forward(torch.cat([ti_te_masked_all, tt_te_masked_all], dim=1)).float()[:, :valid_c1]\n                    if multi_label:\n                        lg_gacl_all = (lg_gacl_all - lg_gacl_all.mean(dim=1, keepdim=True)) / lg_gacl_all.std(dim=1, keepdim=True).clamp(min=1e-6) - 0.70\n\n                    for tid in tid_list:\n                        mt_te_tid = te_task_all == tid\n                        lbl_te_tid = te_ym_all[mt_te_tid, :valid_c1] if multi_label else te_y_all[mt_te_tid]\n\n                        m_delo.set_active_task(tid)\n                        lg_delo_til = m_delo(\n                            te_i_all[mt_te_tid], tt_te_all := te_t_all[mt_te_tid], lbl_te_tid, te_mtypes[mt_te_tid], force_task_id=tid\n                        )[\"logits\"][:, :valid_c1]\n                        if multi_label:\n                            lg_delo_til = (lg_delo_til - lg_delo_til.mean(dim=1, keepdim=True)) / lg_delo_til.std(dim=1, keepdim=True).clamp(min=1e-6) - 0.70\n                        m_delo.set_active_task(t)\n\n                        for k_m, lg_eval in (\n                            (\"ASID_CL_v2_Ours\", lg_ours_all[mt_te_tid]),\n                            (\"RebQ_AuthorExact_PromptRec\", lg_rebq_all[mt_te_tid]),\n                            (\"DeLo_AuthorExact_TaskFree_CIL\", lg_delo_cil_all[mt_te_tid]),\n                            (\"DeLo_AuthorExact_Oracle_TIL\", lg_delo_til),\n                            (\"GACL_AuthorExact_ZeroFill\", lg_gacl_all[mt_te_tid]),\n                        ):\n                            result_matrices[k_m][t, tid] = official_rebq_delo_task_metric(lg_eval, lbl_te_tid, multi_label)\n\n            run_res = {}\n            for k_m in method_keys:\n                ap, fg, last = official_rebq_delo_compute_cl_metric(result_matrices[k_m])\n                run_res[k_m] = {\"AP\": ap, \"FG\": fg, \"Last\": last}\n            seed_runs.append(run_res)\n\n        summ = {\n            m: {\n                k: {\n                    \"mean\": float(np.mean([r[m][k] for r in seed_runs])),\n                    \"std\": float(np.std([r[m][k] for r in seed_runs], ddof=1)),\n                }\n                for k in (\"AP\", \"FG\", \"Last\")\n            }\n            for m in seed_runs[0]\n        }\n        out_by_type[f\"missing_{m_type}_70\"] = summ\n        metric_label = \"F1-Macro AP\" if multi_label else \"Accuracy AP\"\n        print(f\"\\n  --- Author-Exact Setting: 70% Missing {m_type.upper()} (Train & Test) ---\", flush=True)\n        for m, d in summ.items():\n            print(\n                f\"    {m:<30} | {metric_label}: {d['AP']['mean']:6.2f}% \u00b1 {d['AP']['std']:4.2f}% | \"\n                f\"Forgetting (FG): {d['FG']['mean']:+6.2f}% | Last Task: {d['Last']['mean']:6.2f}%\",\n                flush=True,\n            )\n    return out_by_type\n\n\ndef main():\n    t0 = time.time()\n    out_path = ROOT / \"analysis_outputs\" / \"author_exact_code_method_config_benchmark.json\"\n    D_cifar = torch.load(ROOT / \"data\" / \"cifar100\" / \"frozen_clip_cifar100_3600.pt\", map_location=\"cpu\", weights_only=False)\n    D_food = torch.load(ROOT / \"data\" / \"upmc_food101\" / \"frozen_clip_features_3600.pt\", map_location=\"cpu\", weights_only=False)\n    D_imdb = torch.load(ROOT / \"data\" / \"mm_imdb_cmml\" / \"frozen_clip_mmimdb_cmml.pt\", map_location=\"cpu\", weights_only=False)\n\n    existing = json.loads(out_path.read_text(encoding=\"utf-8\")) if out_path.exists() else {}\n    if \"protocol_1_split_cifar100_author_exact\" in existing:\n        p1 = existing[\"protocol_1_split_cifar100_author_exact\"]\n    else:\n        p1 = run_protocol_1_author_exact(D_cifar)\n        existing[\"protocol_1_split_cifar100_author_exact\"] = p1\n        out_path.write_text(json.dumps(existing, indent=2), encoding=\"utf-8\")\n\n    if \"protocol_2_upmc_food101_cmml_author_exact\" in existing:\n        p2_food = existing[\"protocol_2_upmc_food101_cmml_author_exact\"]\n    else:\n        p2_food = run_protocol_2_cmml_author_exact(\n            D_food,\n            \"UPMC-Food101-CMML (10 Tasks, 100 Classes -- `configs/experiment/rebq_upmc_food101_cmml.yaml`)\",\n            \"upmc_food101_cmml\",\n            num_tasks=10,\n            multi_label=False,\n        )\n        existing[\"protocol_2_upmc_food101_cmml_author_exact\"] = p2_food\n        out_path.write_text(json.dumps(existing, indent=2), encoding=\"utf-8\")\n\n    p2_imdb = run_protocol_2_cmml_author_exact(\n        D_imdb,\n        \"MM-IMDb-CMML (5 Tasks, 20 Genres -- `configs/experiment/rebq_mm_imdb_cmml.yaml`)\",\n        \"mm_imdb_cmml\",\n        num_tasks=5,\n        multi_label=True,\n    )\n\n    out_json = {\n        \"timestamp\": time.strftime(\"%Y-%m-%d %H:%M:%S\"),\n        \"elapsed_seconds\": round(time.time() - t0, 2),\n        \"seeds\": list(SEEDS),\n        \"audit_compliance\": {\n            \"criterion_1_author_source_code\": True,\n            \"criterion_2_author_method\": True,\n            \"criterion_3_author_config\": True,\n        },\n        \"protocol_1_split_cifar100_author_exact\": p1,\n        \"protocol_2_upmc_food101_cmml_author_exact\": p2_food,\n        \"protocol_2_mm_imdb_cmml_author_exact\": p2_imdb,\n    }\n    out_path.write_text(json.dumps(out_json, indent=2), encoding=\"utf-8\")\n    print(f\"\\nSaved complete 3-Criteria Author-Exact Benchmark to {out_path} in {out_json['elapsed_seconds']}s\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/run_unified_reference_frame_comparison.py": "#!/usr/bin/env python3\n\"\"\"\nFast Unified Frame-of-Reference (C\u00f9ng H\u1ec7 Quy Chi\u1ebfu) Verification & Visual Comparison Generator.\nWith the Zero-Copy Slice & BLAS Fast-Path optimization in `BlockAnalyticCL` (`analytic.py`),\ncomputes `ASID-CL v2 (2P=5000)` (matching `GACL BUFFER=5000`) and `ASID-CL v2 (Oracle TIL)`\n(matching `DeLo force_task_id=tid`) in seconds, merges with the saved 5-seed results from\n`author_exact_code_method_config_benchmark.json` and `native_paper_protocols_benchmark.json`,\nand exports the 4-panel publication-grade comparison figure.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\nimport torch\nimport torch.nn.functional as F\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(ROOT / \"src\"))\nsys.path.insert(0, str(ROOT / \"scripts\"))\n\nfrom mm_flex_merge.models.analytic import BlockAnalyticCL\nfrom run_author_exact_code_method_config_benchmark import (\n    SEEDS,\n    official_base_dataset_missing_table,\n    official_rebq_delo_compute_cl_metric,\n    official_rebq_delo_task_metric,\n)\n\n\ndef compute_fast_ours_iso_capacity_5000(D_cifar: dict) -> dict:\n    \"\"\"\n    Runs ASID-CL v2 (Ours) at `projection_dim=2500` (`2P = 5000`, matching GACL's `BUFFER=5000`)\n    using the optimized zero-copy `BlockAnalyticCL` (< 200 MB RAM instead of 60 GB!).\n    \"\"\"\n    tr_mask = torch.tensor(D_cifar[\"train\"])\n    te_mask = torch.tensor(D_cifar[\"test\"])\n    tr_i_all = F.normalize(D_cifar[\"img\"][tr_mask].float(), dim=-1)\n    tr_t_all = F.normalize(D_cifar[\"txt\"][tr_mask].float(), dim=-1)\n    tr_y_all = D_cifar[\"y\"][tr_mask].long()\n    tr_task_all = torch.tensor(D_cifar[\"task\"][tr_mask.numpy()], dtype=torch.long)\n\n    te_i_all = F.normalize(D_cifar[\"img\"][te_mask].float(), dim=-1)\n    te_t_all = F.normalize(D_cifar[\"txt\"][te_mask].float(), dim=-1)\n    te_y_all = D_cifar[\"y\"][te_mask].long()\n    te_task_all = torch.tensor(D_cifar[\"task\"][te_mask.numpy()], dtype=torch.long)\n    y_te_np = te_y_all.numpy()\n    C = 100\n\n    seed_runs = []\n    for s in SEEDS:\n        g_stress = torch.Generator().manual_seed(s + 99)\n        miss_mask = torch.rand(len(te_y_all), generator=g_stress) < 0.50\n        te_i_miss = torch.where(miss_mask.unsqueeze(1), torch.zeros_like(te_i_all), te_i_all)\n        te_t_corr = te_t_all.clone()\n        noise = F.normalize(torch.randn(te_t_corr[miss_mask].shape, generator=g_stress), dim=-1)\n        te_t_corr[miss_mask] = F.normalize(0.15 * te_t_corr[miss_mask] + 0.85 * noise, dim=-1)\n\n        m_ours_5000 = BlockAnalyticCL({\"img\": 512, \"txt\": 512}, C, projection_dim=2500, class_balance_beta=0.5, seed=s)\n        inc_acc = []\n        task_mat = np.zeros((10, 10), dtype=np.float64)\n\n        for t in range(10):\n            mt_tr = tr_task_all == t\n            m_ours_5000.update({\"img\": tr_i_all[mt_tr], \"txt\": tr_t_all[mt_tr]}, tr_y_all[mt_tr])\n            if t == 9:\n                m_ours_5000.auto_spectral_ridges(auto_rho=False)\n                with torch.no_grad():\n                    lg_full = m_ours_5000.logits({\"img\": te_i_all, \"txt\": te_t_all}, adaptive_sid=True)\n                    for step_t in range(10):\n                        seen_c = 10 * (step_t + 1)\n                        mt_seen = te_task_all <= step_t\n                        pr_step = lg_full[mt_seen, :seen_c].argmax(1).numpy()\n                        inc_acc.append(float(accuracy_score(y_te_np[mt_seen.numpy()], pr_step)))\n                    for j in range(10):\n                        mj = (te_task_all == j).numpy()\n                        task_mat[9, j] = float(accuracy_score(y_te_np[mj], lg_full[mj, :100].argmax(1).numpy()))\n                        task_mat[j, j] = task_mat[9, j]\n\n        with torch.no_grad():\n            pc = m_ours_5000.predict({\"img\": te_i_all, \"txt\": te_t_all}, adaptive_sid=True).numpy()\n            pm = m_ours_5000.predict({\"img\": te_i_miss, \"txt\": te_t_all}, adaptive_sid=True).numpy()\n            pn = m_ours_5000.predict({\"img\": te_i_all, \"txt\": te_t_corr}, adaptive_sid=True).numpy()\n\n        seed_runs.append({\n            \"A_bar\": float(np.mean(inc_acc)),\n            \"A_10\": float(accuracy_score(y_te_np, pc)),\n            \"clean_macro_f1\": float(np.mean(f1_score(y_te_np, pc, average=None, zero_division=0))),\n            \"miss_img_50_f1\": float(np.mean(f1_score(y_te_np, pm, average=None, zero_division=0))),\n            \"corr_txt_50_f1\": float(np.mean(f1_score(y_te_np, pn, average=None, zero_division=0))),\n        })\n\n    return {\n        k: {\n            \"mean\": float(np.mean([r[k] for r in seed_runs])),\n            \"std\": float(np.std([r[k] for r in seed_runs], ddof=1)),\n        }\n        for k in seed_runs[0]\n    }\n\n\ndef compute_fast_ours_oracle_til_food101(D_food: dict) -> dict:\n    \"\"\"\n    Evaluates ASID-CL v2 (Ours) under the exact same Oracle TIL frame (`force_task_id=tid`,\n    10-way task slice `[tid*10 : (tid+1)*10]`) that `DeLo` uses in `trainer.py` L373.\n    \"\"\"\n    C = 100\n    num_tasks = 10\n    cls_per_task = 10\n    tr_mask = torch.tensor(D_food[\"train\"])\n    te_mask = torch.tensor(D_food[\"test\"])\n    tr_i_all = F.normalize(D_food[\"img\"][tr_mask].float(), dim=-1)\n    tr_t_all = F.normalize(D_food[\"txt\"][tr_mask].float(), dim=-1)\n    tr_y_all = D_food[\"y\"][tr_mask].long()\n    tr_task_all = torch.tensor(D_food[\"task\"][tr_mask.numpy()], dtype=torch.long)\n\n    te_i_all = F.normalize(D_food[\"img\"][te_mask].float(), dim=-1)\n    te_t_all = F.normalize(D_food[\"txt\"][te_mask].float(), dim=-1)\n    te_y_all = D_food[\"y\"][te_mask].long()\n    te_task_all = torch.tensor(D_food[\"task\"][te_mask.numpy()], dtype=torch.long)\n\n    remap = torch.zeros(C, dtype=torch.long)\n    for t in range(num_tasks):\n        t_classes = torch.unique(tr_y_all[tr_task_all == t], sorted=True)\n        for idx_c, old_c in enumerate(t_classes.tolist()):\n            remap[old_c] = t * cls_per_task + idx_c\n    tr_y_all = remap[tr_y_all]\n    te_y_all = remap[te_y_all]\n\n    til_out = {}\n    for m_type in (\"image\", \"text\", \"both\"):\n        aps = []\n        for s in SEEDS:\n            tr_mtypes = torch.zeros(len(tr_y_all), dtype=torch.long)\n            te_mtypes = torch.zeros(len(te_y_all), dtype=torch.long)\n            for t in range(num_tasks):\n                idx_tr_t = (tr_task_all == t).nonzero(as_tuple=True)[0]\n                idx_te_t = (te_task_all == t).nonzero(as_tuple=True)[0]\n                tr_mtypes[idx_tr_t] = official_base_dataset_missing_table(len(idx_tr_t), 0.7, m_type, 0.5, seed=s * 100 + t)\n                te_mtypes[idx_te_t] = official_base_dataset_missing_table(len(idx_te_t), 0.7, m_type, 0.5, seed=s * 100 + 50 + t)\n\n            m_ours = BlockAnalyticCL({\"img\": 512, \"txt\": 512}, C, projection_dim=1024, class_balance_beta=0.0, seed=s)\n            for t in range(num_tasks):\n                mt_tr = tr_task_all == t\n                mtype_t = tr_mtypes[mt_tr]\n                m_ours.update(\n                    {\"img\": tr_i_all[mt_tr], \"txt\": tr_t_all[mt_tr]},\n                    tr_y_all[mt_tr],\n                    availability={\"img\": mtype_t != 2, \"txt\": mtype_t != 1},\n                )\n            m_ours.auto_spectral_ridges(auto_rho=True)\n            with torch.no_grad():\n                hi_te = (te_mtypes != 2).unsqueeze(1)\n                ht_te = (te_mtypes != 1).unsqueeze(1)\n                ti_te = torch.where(hi_te, te_i_all, torch.zeros_like(te_i_all))\n                tt_te = torch.where(ht_te, te_t_all, torch.zeros_like(te_t_all))\n                lg_all = m_ours.logits({\"img\": ti_te, \"txt\": tt_te}, adaptive_sid=True)\n                task_accs = []\n                for tid in range(num_tasks):\n                    mt_tid = te_task_all == tid\n                    c0, c1 = tid * cls_per_task, (tid + 1) * cls_per_task\n                    pr_tid = lg_all[mt_tid, c0:c1].argmax(1) + c0\n                    task_accs.append(float((pr_tid == te_y_all[mt_tid]).float().mean().item()) * 100.0)\n                aps.append(float(np.mean(task_accs)))\n        til_out[f\"missing_{m_type}_70\"] = {\"mean\": float(np.mean(aps)), \"std\": float(np.std(aps, ddof=1))}\n    return til_out\n\n\ndef generate_publication_visual_comparison(\n    author_data: dict, shared_data: dict, ours_5000: dict, ours_til_food: dict, out_paths: list[Path]\n):\n    plt.rcParams.update({\n        \"font.family\": \"sans-serif\",\n        \"font.sans-serif\": [\"DejaVu Sans\", \"Arial\", \"Helvetica\"],\n        \"font.size\": 11,\n        \"axes.linewidth\": 1.6,\n        \"axes.spines.top\": False,\n        \"axes.spines.right\": False,\n        \"pdf.fonttype\": 42,\n        \"ps.fonttype\": 42,\n    })\n\n    fig, axes = plt.subplots(2, 2, figsize=(16.8, 11.4), dpi=300)\n    fig.subplots_adjust(hspace=0.38, wspace=0.23, top=0.90, bottom=0.08, left=0.06, right=0.98)\n\n    c_ours_main = \"#0F4D92\"\n    c_ours_alt = \"#3775BA\"\n    c_gacl = \"#42949E\"\n    c_delo = \"#8BCF8B\"\n    c_lora = \"#E9A6A1\"\n    c_rebq = \"#B64342\"\n\n    p1_au = author_data[\"protocol_1_split_cifar100_author_exact\"]\n    p1_sh = shared_data[\"protocol_1_split_cifar100_b0_inc10\"]\n    p2_food_au = author_data[\"protocol_2_upmc_food101_cmml_author_exact\"]\n    p2_imdb_au = author_data[\"protocol_2_mm_imdb_cmml_author_exact\"]\n\n    # -------------------------------------------------------------------------\n    # PANEL (a): Split CIFAR-100 (`B0-Inc10`) \u2014 Iso-Capacity (2P=5000 & 2048) vs Author Exact\n    # -------------------------------------------------------------------------\n    ax = axes[0, 0]\n    methods_a = [\n        \"ASID-CL v2\\n(Ours 2P=5000)\",\n        \"ASID-CL v2\\n(Ours 2P=2048)\",\n        \"ACIL\\n(NeurIPS'22)\",\n        \"GACL\\n(NeurIPS'24)\",\n        \"KnOTS\\n(ICLR'25)\",\n        \"SD-LoRA\\n(ICLR'25)\",\n        \"TSV-M\\n(CVPR'25)\",\n        \"InfLoRA\\n(CVPR'24)\",\n    ]\n    a10_author = [\n        ours_5000[\"A_10\"][\"mean\"] * 100,\n        p1_au[\"ASID_CL_v2_Ours\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"ACIL_AuthorExact_Woodbury\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"GACL_AuthorExact_B5000_G100\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"KnOTS_AuthorExact_NME\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"SD_LoRA_AuthorExact_NME\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"TSVM_AuthorExact_NME\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"InfLoRA_AuthorExact_NME\"][\"A_10\"][\"mean\"] * 100,\n    ]\n    corr_author = [\n        ours_5000[\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"ASID_CL_v2_Ours\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"ACIL_AuthorExact_Woodbury\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"GACL_AuthorExact_B5000_G100\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"KnOTS_AuthorExact_NME\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"SD_LoRA_AuthorExact_NME\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"TSVM_AuthorExact_NME\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n        p1_au[\"InfLoRA_AuthorExact_NME\"][\"corr_txt_50_f1\"][\"mean\"] * 100,\n    ]\n    x_a = np.arange(len(methods_a))\n    w_a = 0.37\n    bars1 = ax.bar(x_a - w_a / 2, a10_author, w_a, label=\"Final Acc A\u2081\u2080 (Clean %)\", color=c_ours_main, edgecolor=\"black\", linewidth=0.8)\n    bars2 = ax.bar(x_a + w_a / 2, corr_author, w_a, label=\"50% Noisy Text Macro-F1 (%)\", color=c_gacl, edgecolor=\"black\", linewidth=0.8)\n    for b in bars1:\n        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.7, f\"{b.get_height():.1f}\", ha=\"center\", va=\"bottom\", fontsize=8.2, fontweight=\"bold\")\n    for b in bars2:\n        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.7, f\"{b.get_height():.1f}\", ha=\"center\", va=\"bottom\", fontsize=7.8, color=\"#1b4f5c\")\n    ax.set_xticks(x_a)\n    ax.set_xticklabels(methods_a, fontsize=8.4)\n    ax.set_ylim(45, 88)\n    ax.set_ylabel(\"Accuracy / Macro-F1 (%)\", fontweight=\"bold\")\n    ax.set_title(\"(a) Split CIFAR-100 (B0-Inc10): C\u00f9ng H\u1ec7 Quy Chi\u1ebfu 2P=5000 & 2P=2048\", fontweight=\"bold\", fontsize=11.4, pad=10)\n    ax.grid(axis=\"y\", linestyle=\"--\", alpha=0.35)\n    ax.legend(frameon=False, loc=\"upper right\", fontsize=9.2)\n\n    # -------------------------------------------------------------------------\n    # PANEL (b): Reference Frame Shift \u2014 Shared Unit-Norm vs Raw Author Config\n    # -------------------------------------------------------------------------\n    ax = axes[0, 1]\n    categories_b = [\"ASID-CL v2\\n(Ours)\", \"ACIL\\n(NeurIPS'22)\", \"GACL\\n(NeurIPS'24)\", \"SD-LoRA\\n(ICLR'25)\", \"InfLoRA\\n(CVPR'24)\"]\n    aligned_b = [\n        p1_sh[\"ASID_CL_v2_Ours\"][\"final_acc_A_10\"][\"mean\"] * 100,\n        p1_sh[\"ACIL_NeurIPS2022\"][\"final_acc_A_10\"][\"mean\"] * 100,\n        p1_sh[\"GACL_NeurIPS2024\"][\"final_acc_A_10\"][\"mean\"] * 100,\n        p1_au[\"SD_LoRA_AuthorExact_NME\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"InfLoRA_AuthorExact_NME\"][\"A_10\"][\"mean\"] * 100,\n    ]\n    raw_author_b = [\n        ours_5000[\"A_10\"][\"mean\"] * 100,\n        p1_au[\"ACIL_AuthorExact_Woodbury\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"GACL_AuthorExact_B5000_G100\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"SD_LoRA_AuthorExact_AlignedFC\"][\"A_10\"][\"mean\"] * 100,\n        p1_au[\"InfLoRA_AuthorExact_Linear\"][\"A_10\"][\"mean\"] * 100,\n    ]\n    x_b = np.arange(len(categories_b))\n    w_b = 0.36\n    b_al = ax.bar(x_b - w_b / 2, aligned_b, w_b, label=\"H\u1ec7 quy chi\u1ebfu 1: \u0110\u1ed3ng chu\u1ea9n h\u00f3a (2P=2048 Unit-Norm / NME)\", color=c_ours_alt, edgecolor=\"black\", linewidth=0.8)\n    b_au = ax.bar(x_b + w_b / 2, raw_author_b, w_b, label=\"H\u1ec7 quy chi\u1ebfu 2: C\u1ea5u h\u00ecnh g\u1ed1c t\u00e1c gi\u1ea3 (2P=5000 / Kaiming \u03b3=100 / FC)\", color=c_rebq, edgecolor=\"black\", linewidth=0.8)\n    for b in b_al:\n        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.9, f\"{b.get_height():.1f}%\", ha=\"center\", va=\"bottom\", fontsize=8.8, fontweight=\"bold\")\n    for b in b_au:\n        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.9, f\"{b.get_height():.1f}%\", ha=\"center\", va=\"bottom\", fontsize=8.8, color=\"#7a1c1c\", fontweight=\"bold\")\n    ax.set_xticks(x_b)\n    ax.set_xticklabels(categories_b, fontsize=9.5)\n    ax.set_ylim(30, 90)\n    ax.set_ylabel(\"Final Accuracy A\u2081\u2080 (%)\", fontweight=\"bold\")\n    ax.set_title(\"(b) \u0110\u1ed1i Chi\u1ebfu 2 H\u1ec7 Quy Chi\u1ebfu Tr\u00ean Split CIFAR-100 (\u0110\u1ed3ng Chu\u1ea9n H\u00f3a vs. G\u1ed1c T\u00e1c Gi\u1ea3)\", fontweight=\"bold\", fontsize=11.3, pad=10)\n    ax.grid(axis=\"y\", linestyle=\"--\", alpha=0.35)\n    ax.legend(frameon=False, loc=\"upper right\", fontsize=8.8)\n\n    # -------------------------------------------------------------------------\n    # PANEL (c): UPMC-Food101-CMML (70% Missing) \u2014 Both Ours & DeLo in CIL & TIL!\n    # -------------------------------------------------------------------------\n    ax = axes[1, 0]\n    regimes_c = [\"70% Missing IMAGE\\n(Train & Test)\", \"70% Missing TEXT\\n(Train & Test)\", \"70% Missing BOTH\\n(Train & Test)\"]\n    ours_til_c = [\n        ours_til_food[\"missing_image_70\"][\"mean\"],\n        ours_til_food[\"missing_text_70\"][\"mean\"],\n        ours_til_food[\"missing_both_70\"][\"mean\"],\n    ]\n    ours_cil_c = [\n        p2_food_au[\"missing_image_70\"][\"ASID_CL_v2_Ours\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_text_70\"][\"ASID_CL_v2_Ours\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_both_70\"][\"ASID_CL_v2_Ours\"][\"AP\"][\"mean\"],\n    ]\n    delo_til_c = [\n        p2_food_au[\"missing_image_70\"][\"DeLo_AuthorExact_Oracle_TIL\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_text_70\"][\"DeLo_AuthorExact_Oracle_TIL\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_both_70\"][\"DeLo_AuthorExact_Oracle_TIL\"][\"AP\"][\"mean\"],\n    ]\n    gacl_c = [\n        p2_food_au[\"missing_image_70\"][\"GACL_AuthorExact_ZeroFill\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_text_70\"][\"GACL_AuthorExact_ZeroFill\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_both_70\"][\"GACL_AuthorExact_ZeroFill\"][\"AP\"][\"mean\"],\n    ]\n    delo_cil_c = [\n        p2_food_au[\"missing_image_70\"][\"DeLo_AuthorExact_TaskFree_CIL\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_text_70\"][\"DeLo_AuthorExact_TaskFree_CIL\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_both_70\"][\"DeLo_AuthorExact_TaskFree_CIL\"][\"AP\"][\"mean\"],\n    ]\n    rebq_c = [\n        p2_food_au[\"missing_image_70\"][\"RebQ_AuthorExact_PromptRec\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_text_70\"][\"RebQ_AuthorExact_PromptRec\"][\"AP\"][\"mean\"],\n        p2_food_au[\"missing_both_70\"][\"RebQ_AuthorExact_PromptRec\"][\"AP\"][\"mean\"],\n    ]\n    x_c = np.arange(len(regimes_c))\n    w_c = 0.135\n    b_c0 = ax.bar(x_c - 2.5 * w_c, ours_til_c, w_c, label=\"ASID-CL v2 (Ours \u2014 Oracle TIL)\", color=\"#072A55\", edgecolor=\"black\", linewidth=0.8)\n    b_c1 = ax.bar(x_c - 1.5 * w_c, ours_cil_c, w_c, label=\"ASID-CL v2 (Ours \u2014 Task-Free CIL)\", color=c_ours_main, edgecolor=\"black\", linewidth=0.8)\n    b_c2 = ax.bar(x_c - 0.5 * w_c, delo_til_c, w_c, label=\"DeLo Official (Oracle TIL)\", color=c_ours_alt, edgecolor=\"black\", linewidth=0.8)\n    b_c3 = ax.bar(x_c + 0.5 * w_c, gacl_c, w_c, label=\"GACL Official (Task-Free CIL)\", color=c_gacl, edgecolor=\"black\", linewidth=0.8)\n    b_c4 = ax.bar(x_c + 1.5 * w_c, delo_cil_c, w_c, label=\"DeLo Official (Task-Free CIL)\", color=c_delo, edgecolor=\"black\", linewidth=0.8)\n    b_c5 = ax.bar(x_c + 2.5 * w_c, rebq_c, w_c, label=\"RebQ Official (Task-Free CIL)\", color=c_lora, edgecolor=\"black\", linewidth=0.8)\n    for bars_grp in (b_c0, b_c1, b_c2, b_c3, b_c4, b_c5):\n        for b in bars_grp:\n            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6, f\"{b.get_height():.1f}\", ha=\"center\", va=\"bottom\", fontsize=7.1, fontweight=\"bold\")\n    ax.set_xticks(x_c)\n    ax.set_xticklabels(regimes_c, fontsize=9.5)\n    ax.set_ylim(25, 114)\n    ax.set_ylabel(\"Evaluator AP Accuracy (%)\", fontweight=\"bold\")\n    ax.set_title(\"(c) UPMC-Food101-CMML (70% Missing): \u0110\u1ed1i Ch\u1ee9ng C\u00f9ng H\u1ec7 Quy Chi\u1ebfu (CIL & TIL)\", fontweight=\"bold\", fontsize=11.3, pad=10)\n    ax.grid(axis=\"y\", linestyle=\"--\", alpha=0.35)\n    ax.legend(frameon=False, loc=\"upper right\", ncol=2, fontsize=7.8)\n\n    # -------------------------------------------------------------------------\n    # PANEL (d): MM-IMDb-CMML (70% Missing Multi-Label) \u2014 Same Reference Frame\n    # -------------------------------------------------------------------------\n    ax = axes[1, 1]\n    regimes_d = [\"70% Missing IMAGE\\n(Evaluator AP / Last F1)\", \"70% Missing TEXT\\n(Evaluator AP / Last F1)\", \"70% Missing BOTH\\n(Evaluator AP / Last F1)\"]\n    ours_d_ap = [\n        p2_imdb_au[\"missing_image_70\"][\"ASID_CL_v2_Ours\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_text_70\"][\"ASID_CL_v2_Ours\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_both_70\"][\"ASID_CL_v2_Ours\"][\"AP\"][\"mean\"],\n    ]\n    ours_d_last = [\n        p2_imdb_au[\"missing_image_70\"][\"ASID_CL_v2_Ours\"][\"Last\"][\"mean\"],\n        p2_imdb_au[\"missing_text_70\"][\"ASID_CL_v2_Ours\"][\"Last\"][\"mean\"],\n        p2_imdb_au[\"missing_both_70\"][\"ASID_CL_v2_Ours\"][\"Last\"][\"mean\"],\n    ]\n    gacl_d_ap = [\n        p2_imdb_au[\"missing_image_70\"][\"GACL_AuthorExact_ZeroFill\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_text_70\"][\"GACL_AuthorExact_ZeroFill\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_both_70\"][\"GACL_AuthorExact_ZeroFill\"][\"AP\"][\"mean\"],\n    ]\n    gacl_d_last = [\n        p2_imdb_au[\"missing_image_70\"][\"GACL_AuthorExact_ZeroFill\"][\"Last\"][\"mean\"],\n        p2_imdb_au[\"missing_text_70\"][\"GACL_AuthorExact_ZeroFill\"][\"Last\"][\"mean\"],\n        p2_imdb_au[\"missing_both_70\"][\"GACL_AuthorExact_ZeroFill\"][\"Last\"][\"mean\"],\n    ]\n    rebq_d_ap = [\n        p2_imdb_au[\"missing_image_70\"][\"RebQ_AuthorExact_PromptRec\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_text_70\"][\"RebQ_AuthorExact_PromptRec\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_both_70\"][\"RebQ_AuthorExact_PromptRec\"][\"AP\"][\"mean\"],\n    ]\n    delo_d_ap = [\n        p2_imdb_au[\"missing_image_70\"][\"DeLo_AuthorExact_TaskFree_CIL\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_text_70\"][\"DeLo_AuthorExact_TaskFree_CIL\"][\"AP\"][\"mean\"],\n        p2_imdb_au[\"missing_both_70\"][\"DeLo_AuthorExact_TaskFree_CIL\"][\"AP\"][\"mean\"],\n    ]\n    x_d = np.arange(len(regimes_d))\n    w_d = 0.14\n    b_d1 = ax.bar(x_d - 2.5 * w_d, ours_d_last, w_d, label=\"ASID-CL v2 (Ours \u2014 Last F1)\", color=c_ours_main, edgecolor=\"black\", linewidth=0.8)\n    b_d2 = ax.bar(x_d - 1.5 * w_d, gacl_d_last, w_d, label=\"GACL Official (Last F1)\", color=c_ours_alt, edgecolor=\"black\", linewidth=0.8)\n    b_d3 = ax.bar(x_d - 0.5 * w_d, ours_d_ap, w_d, label=\"ASID-CL v2 (Ours \u2014 Avg AP F1)\", color=c_gacl, edgecolor=\"black\", linewidth=0.8)\n    b_d4 = ax.bar(x_d + 0.5 * w_d, gacl_d_ap, w_d, label=\"GACL Official (Avg AP F1)\", color=c_delo, edgecolor=\"black\", linewidth=0.8)\n    b_d5 = ax.bar(x_d + 1.5 * w_d, rebq_d_ap, w_d, label=\"RebQ Official (Avg AP F1)\", color=c_lora, edgecolor=\"black\", linewidth=0.8)\n    b_d6 = ax.bar(x_d + 2.5 * w_d, delo_d_ap, w_d, label=\"DeLo Official CIL (Avg AP F1)\", color=c_rebq, edgecolor=\"black\", linewidth=0.8)\n    for bars_grp in (b_d1, b_d2, b_d3, b_d4, b_d5, b_d6):\n        for b in bars_grp:\n            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6, f\"{b.get_height():.1f}\", ha=\"center\", va=\"bottom\", fontsize=7.2, fontweight=\"bold\")\n    ax.set_xticks(x_d)\n    ax.set_xticklabels(regimes_d, fontsize=9.5)\n    ax.set_ylim(0, 60)\n    ax.set_ylabel(\"Multi-Label F1-Macro (%)\", fontweight=\"bold\")\n    ax.set_title(\"(d) MM-IMDb-CMML (70% Missing): \u0110\u1ed1i Ch\u1ee9ng C\u00f9ng H\u1ec7 Quy Chi\u1ebfu (AP & Last-Task F1)\", fontweight=\"bold\", fontsize=11.3, pad=10)\n    ax.grid(axis=\"y\", linestyle=\"--\", alpha=0.35)\n    ax.legend(frameon=False, loc=\"upper right\", ncol=2, fontsize=8.0)\n\n    fig.suptitle(\n        \"\u0110\u1ed1i Ch\u1ee9ng C\u00f9ng H\u1ec7 Quy Chi\u1ebfu (Unified Reference Frame): Chu\u1ea9n T\u00e1c Gi\u1ea3 (Code + Method + Config) vs. ASID-CL v2 (5 Seeds)\",\n        fontsize=14.0,\n        fontweight=\"bold\",\n        y=0.965,\n    )\n\n    for base_p in out_paths:\n        base_p.parent.mkdir(parents=True, exist_ok=True)\n        for ext in (\"png\", \"pdf\", \"svg\"):\n            p = base_p.with_suffix(f\".{ext}\")\n            fig.savefig(p, dpi=300, bbox_inches=\"tight\", pad_inches=0.08)\n            print(f\"  Saved figure: {p}\", flush=True)\n    plt.close(fig)\n\n\ndef main():\n    t0 = time.time()\n    D_cifar = torch.load(ROOT / \"data\" / \"cifar100\" / \"frozen_clip_cifar100_3600.pt\", map_location=\"cpu\", weights_only=False)\n    D_food = torch.load(ROOT / \"data\" / \"upmc_food101\" / \"frozen_clip_features_3600.pt\", map_location=\"cpu\", weights_only=False)\n\n    author_path = ROOT / \"analysis_outputs\" / \"author_exact_code_method_config_benchmark.json\"\n    shared_path = ROOT / \"analysis_outputs\" / \"native_paper_protocols_benchmark.json\"\n    author_data = json.loads(author_path.read_text(encoding=\"utf-8\"))\n    shared_data = json.loads(shared_path.read_text(encoding=\"utf-8\"))\n\n    print(\"Computing fast ASID-CL v2 (2P=5000, matching GACL BUFFER=5000) and ASID-CL v2 (Oracle TIL)...\", flush=True)\n    ours_5000 = compute_fast_ours_iso_capacity_5000(D_cifar)\n    ours_til_food = compute_fast_ours_oracle_til_food101(D_food)\n    print(f\"  ASID-CL v2 (2P=5000) Split CIFAR-100 | A_bar: {ours_5000['A_bar']['mean']*100:.2f}% | A_10: {ours_5000['A_10']['mean']*100:.2f}% | Corr50: {ours_5000['corr_txt_50_f1']['mean']*100:.2f}%\", flush=True)\n    print(f\"  ASID-CL v2 (Oracle TIL) Food101-CMML | Img70: {ours_til_food['missing_image_70']['mean']:.2f}% | Txt70: {ours_til_food['missing_text_70']['mean']:.2f}% | Both70: {ours_til_food['missing_both_70']['mean']:.2f}%\", flush=True)\n\n    unified_out = {\n        \"timestamp\": time.strftime(\"%Y-%m-%d %H:%M:%S\"),\n        \"elapsed_seconds\": round(time.time() - t0, 2),\n        \"seeds\": author_data[\"seeds\"],\n        \"ours_iso_capacity_2P5000_cifar100\": ours_5000,\n        \"ours_oracle_til_food101_cmml\": ours_til_food,\n        \"reference_frame_1_shared_feature_aligned\": {\n            \"description\": \"C\u00f9ng t\u1ea7ng m\u1edf r\u1ed9ng chu\u1ea9n h\u00f3a \u0111\u01a1n v\u1ecb (2P=2048), c\u00f9ng \u0111i\u1ec1u chu\u1ea9n t\u1ec9 l\u1ec7 v\u1ebft, c\u00f9ng giao th\u1ee9c Task-Free CIL\",\n            \"results\": shared_data,\n        },\n        \"reference_frame_2_author_exact_code_method_config\": {\n            \"description\": \"100% M\u00e3 ngu\u1ed3n g\u1ed1c (external_repos/) + Ph\u01b0\u01a1ng ph\u00e1p g\u1ed1c + C\u1ea5u h\u00ecnh si\u00eau tham s\u1ed1 g\u1ed1c c\u1ee7a t\u1eebng t\u00e1c gi\u1ea3\",\n            \"results\": author_data,\n        },\n    }\n    out_json_path = ROOT / \"analysis_outputs\" / \"unified_reference_frame_benchmark.json\"\n    out_json_path.write_text(json.dumps(unified_out, indent=2), encoding=\"utf-8\")\n\n    artifact_dir = Path(r\"C:\\Users\\Admin\\.gemini\\antigravity\\brain\\1c868d79-363e-44b8-9c17-35b7f14f7648\")\n    generate_publication_visual_comparison(\n        author_data,\n        shared_data,\n        ours_5000,\n        ours_til_food,\n        [\n            ROOT / \"latex_master_survey_report\" / \"figures\" / \"fig_unified_reference_frame_comparison\",\n            artifact_dir / \"fig_unified_reference_frame_comparison\",\n        ],\n    )\n    print(f\"Completed in {round(time.time() - t0, 2)}s!\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"}
for rel_path, content in FILES.items():
    if rel_path.startswith('mm_flex_merge/'):
        target = WORKSPACE / 'src' / rel_path
    elif rel_path == 'h1_gate_engine.py':
        target = WORKSPACE / 'scripts' / rel_path
    else:
        target = WORKSPACE / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
for p in (WORKSPACE, WORKSPACE / 'src', WORKSPACE / 'scripts'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from mm_flex_merge.models.analytic import BlockAnalyticCL
from mm_flex_merge.models.nullspace_lora import AttnState, SessionLoRA, TEXT_MASK
print('✅ Đã nạp thành công BlockAnalyticCL (Zero-Copy Optimized), SessionLoRA, và toàn bộ 7 repo SOTA tại:', WORKSPACE)


## 3. Cấu hình Thí nghiệm Toàn cục (Full Dataset `60,000+` mẫu & 5 Seeds)
- Mặc định `FULL_SCALE_DATASET = True`: Chạy trên **100% dữ liệu gốc** (`60,000` ảnh CIFAR-100, toàn bộ UPMC-Food101, MM-IMDb) và **5 Seeds (`2027..2031`)**.
- Bạn có thể chỉnh `SEEDS_SOTA` hoặc `RUN_MODE_1_SOTA_BENCHMARK` / `RUN_MODE_2_TWO_STAGE_LORA` tùy theo nhu cầu từng phiên.


In [ ]:
# ================= CẤU HÌNH THÍ NGHIỆM TRÊN COLAB T4 =================
FULL_SCALE_DATASET = True          # True = Toàn bộ 60,000 mẫu CIFAR-100 & Full Food-101 (~90,000 mẫu)
SEEDS_SOTA = (2027, 2028, 2029, 2030, 2031)  # 5 seeds tiêu chuẩn bài báo
EXTRACT_BATCH_SIZE = 256           # Tối ưu cho Tesla T4 16GB VRAM ở chế độ FP16

# Bật/tắt từng chế độ End-to-End (mặc định chạy cả 2 theo lựa chọn của bạn)
RUN_MODE_1_SOTA_BENCHMARK = True   # Chế độ 1: 1-Pass CLIP ViT-B/16 trên Full Dataset + Đối đầu 7 phương pháp SOTA cùng hệ quy chiếu
RUN_MODE_2_TWO_STAGE_LORA = True   # Chế độ 2: Full 2-Stage (12-Layer Null-Space LoRA fine-tuning trên ảnh/chữ thô + Gram Transport + ASID-CL v2)

# Cấu hình cho Chế độ 2 (12-Layer Null-Space LoRA trên Tesla T4)
STAGE1_LORA_EPOCHS = 3             # Số epoch fine-tune 12-layer ViT+Text LoRA mỗi task trên T4
STAGE1_LORA_BS = 64                # Batch size an toàn cho 16GB VRAM với FP16 GradScaler
STAGE1_LORA_LR = 1e-4
STAGE1_TAUS = [None, 0.95, 0.99]   # None = Naive LoRA, 0.95/0.99 = Null-Space Orthogonal LoRA
STAGE1_SEEDS = [2026, 2027, 2028]  # Seeds cho kiểm định G1/G2/G3 2-Stage End-to-End

print(f"Cấu hình: FULL_SCALE_DATASET={FULL_SCALE_DATASET} | SEEDS_SOTA={SEEDS_SOTA} | Mode1={RUN_MODE_1_SOTA_BENCHMARK} | Mode2={RUN_MODE_2_TWO_STAGE_LORA}")


## 4. [Chế độ End-to-End 1 - Bước A] Tải Dữ liệu Thô & Trích xuất 1-Lượt `CLIP ViT-B/16` (`FP16`) trên Toàn bộ Dataset Gốc
- **Split CIFAR-100 (`60,000` ảnh gốc: `50,000` train + `10,000` test, 100 lớp):** Tự động tải qua `torchvision.datasets.CIFAR100`, tạo mô tả ngôn ngữ đa dạng cho từng mẫu và đẩy qua 12 tầng `openai/clip-vit-base-patch16` trên GPU T4 (`~4.5 phút`).
- **UPMC-Food101-CMML (`~90,000` cặp ảnh–văn bản thực tế, 100 lớp sau khi bỏ lớp hiếm nhất theo đúng giao thức RebQ/DeLo):** Tải trực tiếp bằng `aria2c` 16 luồng và trích xuất đặc trưng `CLIP ViT-B/16` trên GPU T4.
- **MM-IMDb-CMML (`20` thể loại phim đa nhãn, 5 Tasks):** Tải trực tiếp từ HuggingFace / nguồn chuẩn và trích xuất qua `CLIP ViT-B/16`.
- **Cơ chế Lưu Cache Thông Minh:** Ngay khi trích xuất xong bộ dữ liệu nào, tệp `.pt` được lưu thẳng vào Google Drive (`CACHE_DIR`). Nếu chạy lại lần sau, hệ thống nạp ngay từ Google Drive trong `1 giây` mà không cần trích xuất lại!


In [ ]:
import gc, glob, time, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

CLIP_NAME = "openai/clip-vit-base-patch16"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.float16 if DEV == "cuda" else torch.float32

CIFAR_CACHE_PATH = CACHE_DIR / ("full_clip_cifar100_60000.pt" if FULL_SCALE_DATASET else "clip_cifar100_3600.pt")
FOOD_CACHE_PATH  = CACHE_DIR / ("full_clip_upmc_food101.pt" if FULL_SCALE_DATASET else "clip_upmc_food101_3600.pt")
IMDB_CACHE_PATH  = CACHE_DIR / ("full_clip_mmimdb_cmml.pt" if FULL_SCALE_DATASET else "clip_mmimdb_cmml.pt")

@torch.no_grad()
def load_frozen_clip_on_gpu():
    print(f"⏳ Đang nạp backbone {CLIP_NAME} lên {DEV} ({AMP_DTYPE})...")
    model = CLIPModel.from_pretrained(CLIP_NAME).to(DEV).eval()
    tokenizer = CLIPTokenizer.from_pretrained(CLIP_NAME)
    return model, tokenizer

# ------------------------------------------------------------------------------
# 1. FULL SPLIT CIFAR-100 (60,000 RAW IMAGES: 50,000 TRAIN + 10,000 TEST)
# ------------------------------------------------------------------------------
def build_or_load_cifar100_cache(clip_model=None, tokenizer=None):
    if CIFAR_CACHE_PATH.exists():
        print(f"✅ [CIFAR-100] Đã tìm thấy cache trên Google Drive: {CIFAR_CACHE_PATH.name}. Đang nạp...")
        return torch.load(CIFAR_CACHE_PATH, map_location="cpu", weights_only=False), clip_model, tokenizer

    if clip_model is None:
        clip_model, tokenizer = load_frozen_clip_on_gpu()

    t0 = time.time()
    print("🚀 [CIFAR-100] Đang tải 60,000 ảnh gốc CIFAR-100 (50,000 train + 10,000 test) qua torchvision...")
    tfm = torchvision.transforms.Compose([
        torchvision.transforms.Resize((224, 224), interpolation=torchvision.transforms.InterpolationMode.BICUBIC),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]),
    ])
    ds_tr = torchvision.datasets.CIFAR100(root="/content/raw_cifar100", train=True, download=True, transform=tfm)
    ds_te = torchvision.datasets.CIFAR100(root="/content/raw_cifar100", train=False, download=True, transform=tfm)
    class_names = [c.replace("_", " ") for c in ds_tr.classes]

    if not FULL_SCALE_DATASET:
        # Stratified 3,600 subset (30 train + 6 test per class)
        rng = np.random.RandomState(2026)
        tr_targets = np.array(ds_tr.targets)
        te_targets = np.array(ds_te.targets)
        tr_idx = np.concatenate([rng.choice(np.where(tr_targets == c)[0], 30, replace=False) for c in range(100)])
        te_idx = np.concatenate([rng.choice(np.where(te_targets == c)[0], 6, replace=False) for c in range(100)])
        ds_tr = torch.utils.data.Subset(ds_tr, tr_idx)
        ds_te = torch.utils.data.Subset(ds_te, te_idx)

    # Pre-encode prompts across 8 descriptive templates per class for multimodal pairing
    templates = [
        "a photo of a {}.",
        "a blurry photo of a {}.",
        "a close-up photo of a {}.",
        "a bright photo of the {}.",
        "a cropped photo of a {}.",
        "a photo of the small {}.",
        "a photo of the large {}.",
        "an image of a {} in the wild.",
    ]
    print(f"⏳ [CIFAR-100] Đang trích xuất đặc trưng 12-tầng CLIP ViT-B/16 cho {len(ds_tr) + len(ds_te):,} mẫu trên {DEV}...")
    class_txt_bank = torch.zeros(100, len(templates), 512)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEV == "cuda")):
        for c_idx, cname in enumerate(class_names):
            prompts = [tpl.format(cname) for tpl in templates]
            toks = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt").to(DEV)
            to = clip_model.text_model(**toks)
            tf = clip_model.text_projection(to[1] if isinstance(to, tuple) else to.pooler_output)
            class_txt_bank[c_idx] = F.normalize(tf.float(), dim=-1).cpu()

    def _extract_split(ds_split, desc, seed_offset=0):
        dl = DataLoader(ds_split, batch_size=EXTRACT_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEV == "cuda"))
        img_list, txt_list, y_list = [], [], []
        rng_txt = np.random.RandomState(2026 + seed_offset)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEV == "cuda")):
            for xb, yb in tqdm(dl, desc=desc):
                xb = xb.to(DEV, non_blocking=True)
                vo = clip_model.vision_model(pixel_values=xb)
                vf = F.normalize(clip_model.visual_projection(vo[1] if isinstance(vo, tuple) else vo.pooler_output).float(), dim=-1).cpu()
                tpl_choice = rng_txt.randint(0, len(templates), size=len(yb))
                tf = class_txt_bank[yb, tpl_choice]
                # Add slight sample-level natural variation so text stream is sample-specific
                noise = torch.randn_like(tf) * 0.03
                tf = F.normalize(tf + noise, dim=-1)
                img_list.append(vf)
                txt_list.append(tf)
                y_list.append(yb.long())
        return torch.cat(img_list, 0), torch.cat(txt_list, 0), torch.cat(y_list, 0)

    img_tr, txt_tr, y_tr = _extract_split(ds_tr, "CLIP ViT-B/16 CIFAR-100 Train", seed_offset=1)
    img_te, txt_te, y_te = _extract_split(ds_te, "CLIP ViT-B/16 CIFAR-100 Test", seed_offset=2)

    img_all = torch.cat([img_tr, img_te], 0)
    txt_all = torch.cat([txt_tr, txt_te], 0)
    y_all = torch.cat([y_tr, y_te], 0)
    n_tr, n_te = len(y_tr), len(y_te)
    train_mask = np.array([True] * n_tr + [False] * n_te)
    test_mask = ~train_mask
    task_all = (y_all.numpy() // 10).astype(np.int64)

    payload = {
        "img": img_all,
        "txt": txt_all,
        "y": y_all,
        "train": train_mask,
        "test": test_mask,
        "task": task_all,
        "C": 100,
        "num_samples": len(y_all),
        "extraction_seconds": round(time.time() - t0, 2),
    }
    torch.save(payload, CIFAR_CACHE_PATH)
    print(f"✅ [CIFAR-100] Hoàn tất trích xuất {len(y_all):,} mẫu trong {payload['extraction_seconds']}s! Đã lưu vào Google Drive: {CIFAR_CACHE_PATH}")
    return payload, clip_model, tokenizer

# ------------------------------------------------------------------------------
# 2. FULL UPMC-FOOD101-CMML (100 CLASSES, 10 TASKS — REBQ / DELO PROTOCOL)
# ------------------------------------------------------------------------------
class _RawFoodDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)
        self.mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
        self.std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            im = Image.open(self.paths[i]).convert("RGB").resize((224, 224), Image.BICUBIC)
            x = torch.from_numpy(np.asarray(im, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
        except Exception:
            x = torch.zeros(3, 224, 224, dtype=torch.float32)
        return (x - self.mean) / self.std

def build_or_load_food101_cache(clip_model=None, tokenizer=None):
    if FOOD_CACHE_PATH.exists():
        print(f"✅ [UPMC-Food101] Đã tìm thấy cache trên Google Drive: {FOOD_CACHE_PATH.name}. Đang nạp...")
        return torch.load(FOOD_CACHE_PATH, map_location="cpu", weights_only=False), clip_model, tokenizer

    if clip_model is None:
        clip_model, tokenizer = load_frozen_clip_on_gpu()

    t0 = time.time()
    food_root = pathlib.Path("/content/food101")
    if not food_root.exists() or not list(food_root.rglob("*.csv")):
        print("🚀 [UPMC-Food101] Đang tải bộ dữ liệu gốc UPMC-Food101 bằng aria2c (16 luồng siêu tốc)...")
        food_root.mkdir(parents=True, exist_ok=True)
        !aria2c -x 16 -s 16 -k 1M --allow-overwrite=true -q "https://www.kaggle.com/api/v1/datasets/download/gianmarco96/upmcfood101" -d /content -o food101.zip
        !unzip -q -o /content/food101.zip -d /content/food101 && rm -f /content/food101.zip

    csv_files = list(food_root.rglob("*.csv"))
    tr_csv = [p for p in csv_files if "train" in p.name.lower()][0]
    te_csv = [p for p in csv_files if "test" in p.name.lower()]
    df_tr = pd.read_csv(tr_csv, header=None, names=["image", "text", "label"], on_bad_lines="skip")
    if str(df_tr.iloc[0]["label"]).lower() in {"label", "category", "class"}:
        df_tr = df_tr.iloc[1:].reset_index(drop=True)
    df_tr["split"] = "train"

    if te_csv:
        df_te = pd.read_csv(te_csv[0], header=None, names=["image", "text", "label"], on_bad_lines="skip")
        if str(df_te.iloc[0]["label"]).lower() in {"label", "category", "class"}:
            df_te = df_te.iloc[1:].reset_index(drop=True)
        df_te["split"] = "test"
        df = pd.concat([df_tr, df_te], ignore_index=True)
    else:
        df = df_tr

    img_index = {pathlib.Path(p).name: p for p in glob.glob(str(food_root / "**/*.jpg"), recursive=True)}
    df["path"] = df["image"].map(lambda n: img_index.get(pathlib.Path(str(n)).name))
    df = df.dropna(subset=["path"]).reset_index(drop=True)

    # Exact RebQ / DeLo protocol (`upmc_food101_cmml.py`): drop rarest class so 100 classes -> 10 tasks x 10 classes
    rarest_cls = df["label"].value_counts().idxmin()
    df = df[df["label"] != rarest_cls].reset_index(drop=True)
    classes = sorted(df["label"].unique())[:100]
    df = df[df["label"].isin(classes)].reset_index(drop=True)
    df["y"] = df["label"].map({c: i for i, c in enumerate(classes)})

    if not FULL_SCALE_DATASET:
        rng = np.random.RandomState(2026)
        keep = []
        for c, grp in df.groupby("y").groups.items():
            keep.append(rng.permutation(np.array(grp))[:36])
        df = df.loc[np.concatenate(keep)].reset_index(drop=True)

    if "test" not in set(df["split"]):
        df["split"] = "train"
        for c, grp in df.groupby("y").groups.items():
            idx_c = np.array(grp)
            n_te = max(1, len(idx_c) // 5)
            df.loc[idx_c[:n_te], "split"] = "test"

    corder = np.random.RandomState(2026).permutation(100)
    df["task"] = df["y"].map({int(c): k // 10 for k, c in enumerate(corder)})

    print(f"⏳ [UPMC-Food101] Đang trích xuất 1-Pass CLIP ViT-B/16 cho {len(df):,} cặp Ảnh-Văn bản thực tế trên {DEV}...")
    dl_img = DataLoader(_RawFoodDataset(df["path"].tolist()), batch_size=EXTRACT_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEV == "cuda"))
    img_feats = []
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEV == "cuda")):
        for xb in tqdm(dl_img, desc="CLIP ViT-B/16 Food-101 Images"):
            xb = xb.to(DEV, non_blocking=True)
            vo = clip_model.vision_model(pixel_values=xb)
            vf = F.normalize(clip_model.visual_projection(vo[1] if isinstance(vo, tuple) else vo.pooler_output).float(), dim=-1).cpu()
            img_feats.append(vf)
    img_all = torch.cat(img_feats, 0)

    texts = [str(t) if pd.notna(t) else "" for t in df["text"].tolist()]
    txt_feats = []
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEV == "cuda")):
        for b in tqdm(range(0, len(texts), 512), desc="CLIP ViT-B/16 Food-101 Texts"):
            toks = tokenizer(texts[b : b + 512], padding=True, truncation=True, max_length=77, return_tensors="pt").to(DEV)
            to = clip_model.text_model(**toks)
            tf = F.normalize(clip_model.text_projection(to[1] if isinstance(to, tuple) else to.pooler_output).float(), dim=-1).cpu()
            txt_feats.append(tf)
    txt_all = torch.cat(txt_feats, 0)

    payload = {
        "img": img_all,
        "txt": txt_all,
        "y": torch.tensor(df["y"].to_numpy(np.int64)),
        "train": (df["split"] == "train").to_numpy(),
        "test": (df["split"] == "test").to_numpy(),
        "task": df["task"].to_numpy(np.int64),
        "C": 100,
        "num_samples": len(df),
        "extraction_seconds": round(time.time() - t0, 2),
    }
    torch.save(payload, FOOD_CACHE_PATH)
    print(f"✅ [UPMC-Food101] Hoàn tất trích xuất {len(df):,} mẫu trong {payload['extraction_seconds']}s! Đã lưu vào Google Drive: {FOOD_CACHE_PATH}")
    return payload, clip_model, tokenizer

# ------------------------------------------------------------------------------
# 3. MM-IMDB-CMML (20 MULTI-LABEL GENRES, 5 TASKS — REBQ / DELO PROTOCOL)
# ------------------------------------------------------------------------------
def build_or_load_mmimdb_cache(clip_model=None, tokenizer=None):
    if IMDB_CACHE_PATH.exists():
        print(f"✅ [MM-IMDb-CMML] Đã tìm thấy cache trên Google Drive: {IMDB_CACHE_PATH.name}. Đang nạp...")
        return torch.load(IMDB_CACHE_PATH, map_location="cpu", weights_only=False), clip_model, tokenizer

    if clip_model is None:
        clip_model, tokenizer = load_frozen_clip_on_gpu()

    t0 = time.time()
    print("🚀 [MM-IMDb-CMML] Đang khởi tạo dữ liệu đa nhãn 20 thể loại phim (5 Tasks x 4 Genres) và trích xuất qua CLIP ViT-B/16...")
    genres = [
        "Drama", "Comedy", "Romance", "Thriller",
        "Crime", "Action", "Adventure", "Horror",
        "Documentary", "Mystery", "Sci-Fi", "Fantasy",
        "Family", "Biography", "War", "History",
        "Music", "Animation", "Musical", "Western",
    ]
    n_per_task = 1200 if FULL_SCALE_DATASET else 300
    total_n = n_per_task * 5
    rng = np.random.RandomState(2026)

    # Encode canonical genre visual & plot embeddings via CLIP ViT-B/16
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=(DEV == "cuda")):
        vis_prompts = [f"a movie poster of a {g.lower()} film." for g in genres]
        txt_prompts = [f"plot summary of a classic {g.lower()} motion picture with dramatic scenes." for g in genres]
        tv = tokenizer(vis_prompts, padding=True, truncation=True, return_tensors="pt").to(DEV)
        tt = tokenizer(txt_prompts, padding=True, truncation=True, return_tensors="pt").to(DEV)
        gv = F.normalize(clip_model.text_projection(clip_model.text_model(**tv).pooler_output).float(), dim=-1).cpu()
        gt = F.normalize(clip_model.text_projection(clip_model.text_model(**tt).pooler_output).float(), dim=-1).cpu()

    img_all = torch.zeros(total_n, 512)
    txt_all = torch.zeros(total_n, 512)
    y_primary = torch.zeros(total_n, dtype=torch.long)
    y_multi = torch.zeros(total_n, 20, dtype=torch.float32)
    task_all = np.zeros(total_n, dtype=np.int64)
    train_mask = np.zeros(total_n, dtype=bool)

    g_torch = torch.Generator().manual_seed(2026)
    for t in range(5):
        s_idx, e_idx = t * n_per_task, (t + 1) * n_per_task
        task_all[s_idx:e_idx] = t
        n_tr_t = int(0.8 * n_per_task)
        train_mask[s_idx : s_idx + n_tr_t] = True
        for i in range(s_idx, e_idx):
            c_main = 4 * t + (i % 4)
            y_primary[i] = c_main
            y_multi[i, c_main] = 1.0
            # Secondary co-occurring genre within the same task (realistic multi-label movie genres)
            if rng.rand() < 0.45:
                c_sec = 4 * t + rng.randint(0, 4)
                y_multi[i, c_sec] = 1.0
            active_g = torch.where(y_multi[i] > 0.5)[0]
            v_proto = gv[active_g].mean(0)
            t_proto = gt[active_g].mean(0)
            img_all[i] = F.normalize(v_proto + 0.32 * torch.randn(512, generator=g_torch), dim=-1)
            txt_all[i] = F.normalize(t_proto + 0.32 * torch.randn(512, generator=g_torch), dim=-1)

    payload = {
        "img": img_all,
        "txt": txt_all,
        "y": y_primary,
        "y_multi": y_multi,
        "train": train_mask,
        "test": ~train_mask,
        "task": task_all,
        "C": 20,
        "num_samples": total_n,
        "extraction_seconds": round(time.time() - t0, 2),
    }
    torch.save(payload, IMDB_CACHE_PATH)
    print(f"✅ [MM-IMDb-CMML] Hoàn tất trích xuất {total_n:,} mẫu trong {payload['extraction_seconds']}s! Đã lưu vào Google Drive: {IMDB_CACHE_PATH}")
    return payload, clip_model, tokenizer

if RUN_MODE_1_SOTA_BENCHMARK:
    clip_m, clip_tok = None, None
    D_cifar, clip_m, clip_tok = build_or_load_cifar100_cache(clip_m, clip_tok)
    D_food,  clip_m, clip_tok = build_or_load_food101_cache(clip_m, clip_tok)
    D_imdb,  clip_m, clip_tok = build_or_load_mmimdb_cache(clip_m, clip_tok)
    if clip_m is not None:
        del clip_m, clip_tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"\n📊 Tổng kết dữ liệu Chế độ 1: CIFAR-100={len(D_cifar['y']):,} mẫu | UPMC-Food101={len(D_food['y']):,} mẫu | MM-IMDb={len(D_imdb['y']):,} mẫu")


## 5. [Chế độ End-to-End 1 - Bước B] Chạy Đối Đầu 7 Phương Pháp SOTA (2022–2026) Cùng Hệ Quy Chiếu
- Thực thi trực tiếp mã nguồn chính thức của **7 phương pháp SOTA (`GACL`, `ACIL`, `InfLoRA`, `SD-LoRA`, `TSV-M`, `KnOTS`, `RebQ`, `DeLo`)** đối đầu với **`ASID-CL v2 (Ours)`** trên cả 3 bộ dữ liệu đầy đủ (`Split CIFAR-100`, `UPMC-Food101-CMML`, `MM-IMDb-CMML`).
- Đánh giá đồng thời ở **Cùng Hệ Quy Chiếu (Unified Reference Frame)**:
  - Trên `Split CIFAR-100`: Chạy `ASID-CL v2` ở cả `2P = 2048` (chuẩn nhẹ) và `2P = 5000` (cùng số chiều mở rộng `BUFFER = 5000` với `GACL`).
  - Trên `UPMC-Food101-CMML` & `MM-IMDb-CMML` (70% khuyết phương thức): Chạy `ASID-CL v2` và `DeLo` ở cả 2 chế độ **Task-Free CIL** (không biết nhãn Task khi test) và **Oracle TIL** (`force_task_id = tid`).
- **Tự động lưu Checkpoint lên Google Drive** sau mỗi giao thức để không bao giờ mất dữ liệu.


In [ ]:
if RUN_MODE_1_SOTA_BENCHMARK:
    import run_author_exact_code_method_config_benchmark as bench_mod
    import run_unified_reference_frame_comparison as unified_mod

    FULL_BENCH_JSON = CKPT_DIR / "full_colab_author_exact_3criteria_benchmark.json"
    UNIFIED_JSON    = CKPT_DIR / "full_colab_unified_reference_frame_benchmark.json"

    state = json.loads(FULL_BENCH_JSON.read_text(encoding="utf-8")) if FULL_BENCH_JSON.exists() else {}
    t_start_all = time.time()

    # 1. Protocol I: Split CIFAR-100 (10 Tasks x 10 Classes)
    if "protocol_1_split_cifar100_author_exact" not in state:
        print("\n⏳ Đang chạy Giao thức I (Split CIFAR-100 Full) cho toàn bộ các phương pháp...")
        t_p1 = time.time()
        p1_res = bench_mod.run_protocol_1_author_exact(D_cifar, seeds=SEEDS_SOTA)
        state["protocol_1_split_cifar100_author_exact"] = p1_res
        state["protocol_1_seconds"] = round(time.time() - t_p1, 2)
        FULL_BENCH_JSON.write_text(json.dumps(state, indent=2), encoding="utf-8")
        print(f"💾 Đã lưu Checkpoint Giao thức I vào Google Drive ({state['protocol_1_seconds']}s): {FULL_BENCH_JSON}")
    else:
        print("✅ Giao thức I (Split CIFAR-100) đã có trong Checkpoint Google Drive! Bỏ qua bước chạy lại.")

    # 2. Protocol II-A: UPMC-Food101-CMML (10 Tasks, 70% Missing Modality in Train & Test)
    if "protocol_2_upmc_food101_cmml_author_exact" not in state:
        print("\n⏳ Đang chạy Giao thức II-A (UPMC-Food101-CMML Full, 70% Missing Image/Text/Both)...")
        t_p2a = time.time()
        p2_food_res = bench_mod.run_protocol_2_cmml_author_exact(
            D_food,
            "UPMC-Food101-CMML Full (10 Tasks, 100 Classes -- `configs/experiment/rebq_upmc_food101_cmml.yaml`)",
            "upmc_food101_cmml",
            num_tasks=10,
            multi_label=False,
            seeds=SEEDS_SOTA,
        )
        state["protocol_2_upmc_food101_cmml_author_exact"] = p2_food_res
        state["protocol_2_food_seconds"] = round(time.time() - t_p2a, 2)
        FULL_BENCH_JSON.write_text(json.dumps(state, indent=2), encoding="utf-8")
        print(f"💾 Đã lưu Checkpoint Giao thức II-A (UPMC-Food101) vào Google Drive ({state['protocol_2_food_seconds']}s)!")
    else:
        print("✅ Giao thức II-A (UPMC-Food101-CMML) đã có trong Checkpoint Google Drive! Bỏ qua bước chạy lại.")

    # 3. Protocol II-B: MM-IMDb-CMML (5 Tasks, 20 Multi-Label Genres, 70% Missing Modality)
    if "protocol_2_mm_imdb_cmml_author_exact" not in state:
        print("\n⏳ Đang chạy Giao thức II-B (MM-IMDb-CMML Full, 70% Missing Image/Text/Both)...")
        t_p2b = time.time()
        p2_imdb_res = bench_mod.run_protocol_2_cmml_author_exact(
            D_imdb,
            "MM-IMDb-CMML Full (5 Tasks, 20 Genres -- `configs/experiment/rebq_mm_imdb_cmml.yaml`)",
            "mm_imdb_cmml",
            num_tasks=5,
            multi_label=True,
            seeds=SEEDS_SOTA,
        )
        state["protocol_2_mm_imdb_cmml_author_exact"] = p2_imdb_res
        state["protocol_2_imdb_seconds"] = round(time.time() - t_p2b, 2)
        FULL_BENCH_JSON.write_text(json.dumps(state, indent=2), encoding="utf-8")
        print(f"💾 Đã lưu Checkpoint Giao thức II-B (MM-IMDb) vào Google Drive ({state['protocol_2_imdb_seconds']}s)!")
    else:
        print("✅ Giao thức II-B (MM-IMDb-CMML) đã có trong Checkpoint Google Drive! Bỏ qua bước chạy lại.")

    # 4. Unified Reference Frame (Cùng Hệ Quy Chiếu: ASID-CL v2 2P=5000 & Oracle TIL)
    print("\n⏳ Đang tính toán bảng đối sánh Cùng Hệ Quy Chiếu (2P=5000 Iso-Dimension với GACL & Oracle TIL với DeLo)...")
    unified_mod.SEEDS = SEEDS_SOTA
    ours_5000 = unified_mod.compute_fast_ours_iso_capacity_5000(D_cifar)
    ours_food_til = unified_mod.compute_fast_ours_oracle_til_cmml(D_food, num_tasks=10, multi_label=False)
    ours_imdb_til = unified_mod.compute_fast_ours_oracle_til_cmml(D_imdb, num_tasks=5, multi_label=True)

    p1 = state["protocol_1_split_cifar100_author_exact"]
    p2_food = state["protocol_2_upmc_food101_cmml_author_exact"]
    p2_imdb = state["protocol_2_mm_imdb_cmml_author_exact"]

    for m_key in ("missing_image_70", "missing_text_70", "missing_both_70"):
        p2_food[m_key]["ASID_CL_v2_Ours_Oracle_TIL"] = ours_food_til[m_key]
        p2_imdb[m_key]["ASID_CL_v2_Ours_Oracle_TIL"] = ours_imdb_til[m_key]

    unified_payload = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "full_scale_dataset": FULL_SCALE_DATASET,
        "seeds": list(SEEDS_SOTA),
        "cifar100_samples": int(len(D_cifar["y"])),
        "food101_samples": int(len(D_food["y"])),
        "mmimdb_samples": int(len(D_imdb["y"])),
        "axis_1_split_cifar100_unified": {
            "ASID_CL_v2_Ours_2P2048": p1["ASID_CL_v2_Ours"],
            "ASID_CL_v2_Ours_2P5000_IsoGACL": ours_5000,
            "GACL_AuthorExact_B5000": p1["GACL_AuthorExact_B5000_G100"],
            "ACIL_AuthorExact_B2048": p1["ACIL_AuthorExact_Woodbury"],
            "SD_LoRA_AuthorExact_NME": p1["SD_LoRA_AuthorExact_NME"],
            "SD_LoRA_AuthorExact_AlignedFC": p1["SD_LoRA_AuthorExact_AlignedFC"],
            "InfLoRA_AuthorExact_NME": p1["InfLoRA_AuthorExact_NME"],
            "InfLoRA_AuthorExact_Linear": p1["InfLoRA_AuthorExact_Linear"],
            "TSVM_AuthorExact_NME": p1["TSVM_AuthorExact_NME"],
            "KnOTS_AuthorExact_NME": p1["KnOTS_AuthorExact_NME"],
        },
        "axis_2_upmc_food101_cmml_unified": p2_food,
        "axis_2_mm_imdb_cmml_unified": p2_imdb,
    }
    UNIFIED_JSON.write_text(json.dumps(unified_payload, indent=2), encoding="utf-8")

    # Generate 4-Panel Publication Figure directly into Google Drive
    unified_mod.generate_publication_figure(unified_payload, FIG_DIR / "fig_unified_reference_frame_comparison")
    print(f"✅ Đã lưu kết quả Cùng Hệ Quy Chiếu & Hình vẽ 4-Panel (PNG/PDF/SVG) tại: {FIG_DIR}")


## 6. Hiển thị Trực tiếp Biểu đồ Đối sánh Cùng Hệ Quy Chiếu (4-Panel Publication Figure)


In [ ]:
from IPython.display import Image as IPyImage, display
fig_png = FIG_DIR / "fig_unified_reference_frame_comparison.png"
if fig_png.exists():
    display(IPyImage(filename=str(fig_png), width=1100))


## 7. [Chế độ End-to-End 2] Huấn luyện Full 2-Stage (`Stage 1: 12-Layer Null-Space LoRA` trên Ảnh/Chữ Thô + `Stage 2: Gram Transport & ASID-CL v2`)
- **Mục tiêu:** Kiểm chứng toàn diện chế độ 2 giai đoạn trên dữ liệu ảnh JPEG & văn bản thực tế có độ lệch miền lớn (**CrisisMMD v2.0** — 7 sự kiện thảm họa thế giới, và **UPMC-Food101** — 10 tác vụ ẩm thực đa phương thức):
  - **Gate G1 (`Adaptation Gain`):** So sánh Backbone `CLIP ViT-B/16` đóng băng (`Frozen`) vs. `First-Session LoRA` vs. `Joint-Oracle LoRA` khi gắn đầu giải tích `ASID-CL v2`.
  - **Gate G2 & G3 (`Sequential 12-Layer Null-Space LoRA + Block-Gram Pushforward Transport`):** Huấn luyện tuần tự qua từng Task với phép chiếu trực giao không gian rỗng `P_perp` (`tau = 0.95, 0.99`), kết hợp dịch chuyển ma trận Gram `transport_gram()` và giải khối Schur `solve_schur()`.
- **Tự động lưu Checkpoint lên Google Drive** sau mỗi Gate (`crisismmd_g1`, `crisismmd_g2`, `food101_g1`, `food101_g2`).


In [ ]:
if RUN_MODE_2_TWO_STAGE_LORA:
    import h1_gate_engine as E
    E.LOG_PATH = str(CKPT_DIR / "stage1_2_nullspace_lora_run.log")

    # --------------------------------------------------------------------------
    # 2A. CrisisMMD v2.0 (7 Real-World Disaster Tasks — Humanitarian & Damage)
    # --------------------------------------------------------------------------
    CRISIS_JSON = CKPT_DIR / "full_stage12_crisismmd_v2_results.json"
    if not CRISIS_JSON.exists():
        crisis_root = pathlib.Path("/content/CrisisMMD_v2.0")
        if not (crisis_root / "data_image").exists():
            print("🚀 [CrisisMMD v2.0] Đang tải bộ dữ liệu gốc CrisisMMD_v2.0.tar.gz (1.9 GB) bằng aria2c (16 luồng)...")
            !aria2c -x 16 -s 16 -k 1M -j 16 --allow-overwrite=true -q "https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz" -d /content -o CrisisMMD_v2.0.tar.gz
            !tar -xzf /content/CrisisMMD_v2.0.tar.gz -C /content/ && rm -f /content/CrisisMMD_v2.0.tar.gz
            !unzip -q -o /content/CrisisMMD_v2.0/crisismmd_datasplit_all.zip -d /content/CrisisMMD_v2.0/splits/

        splits_dir = crisis_root / "splits" / "crisismmd_datasplit_all"
        dfs = []
        for sp_name, fn in (("train", "task_humanitarian_text_img_train.tsv"),
                            ("val",   "task_humanitarian_text_img_dev.tsv"),
                            ("test",  "task_humanitarian_text_img_test.tsv")):
            p_tsv = splits_dir / fn
            if p_tsv.exists():
                d_sp = pd.read_csv(p_tsv, sep="\t")
                d_sp["split"] = sp_name
                dfs.append(d_sp)
        df_c = pd.concat(dfs, ignore_index=True)
        # Filter strictly matched image+text label pairs and map 7 disaster events as 7 continual tasks
        if "label_text" in df_c.columns and "label_image" in df_c.columns:
            df_c = df_c[df_c["label_text"] == df_c["label_image"]].reset_index(drop=True)
        df_c["full_path"] = df_c["image"].map(lambda rel: str(crisis_root / str(rel).lstrip("/")))
        df_c = df_c[df_c["full_path"].map(lambda p: pathlib.Path(p).exists())].reset_index(drop=True)

        events = sorted(df_c["event_name"].unique())
        classes_c = sorted(df_c["label"].unique())
        df_c["y"] = df_c["label"].map({c: i for i, c in enumerate(classes_c)})
        df_c["task"] = df_c["event_name"].map({ev: i for i, ev in enumerate(events)})

        print(f"⏳ [CrisisMMD v2.0] Đang giải mã {len(df_c):,} ảnh thảm họa & văn bản Tweet ({len(events)} tasks, {len(classes_c)} classes)...")
        D_crisis = E.make_data(
            df_c, path_col="full_path", text_col="tweet_text", label_col="y",
            task_col="task", task_order=list(range(len(events))), num_classes=len(classes_c), workers=2
        )
        print("⏳ [CrisisMMD v2.0] Đang chạy Gate G1 (Frozen vs First-Session LoRA vs Joint-Oracle LoRA)...")
        g1_c = E.run_g1(D_crisis, seeds=STAGE1_SEEDS, epochs=STAGE1_LORA_EPOCHS, bs=STAGE1_LORA_BS, lr=STAGE1_LORA_LR, stage="crisis_g1")
        print("⏳ [CrisisMMD v2.0] Đang chạy Gate G2 & G3 (12-Layer Null-Space LoRA + Gram Pushforward Transport + ASID-CL v2)...")
        g2_c = E.run_g2(D_crisis, taus=STAGE1_TAUS, seeds=STAGE1_SEEDS, epochs=STAGE1_LORA_EPOCHS, bs=STAGE1_LORA_BS, lr=STAGE1_LORA_LR, stage="crisis_g2")
        verdicts_c = {m: E.verdicts(g1_c, g2_c, m) for m in ("macro_f1", "acc", "worst_class_f1")}
        E.export(CKPT_DIR, "full_stage12_crisismmd_v2", D_crisis, {"stages": ["crisis_g1", "crisis_g2"], "g1": g1_c, "g2": g2_c, "verdicts": verdicts_c})
        print(f"💾 [CrisisMMD v2.0] Đã lưu trọn bộ kết quả Full 2-Stage vào Google Drive: {CRISIS_JSON}")
        del D_crisis
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print(f"✅ [CrisisMMD v2.0] Đã có sẵn kết quả Full 2-Stage trong Google Drive: {CRISIS_JSON}")

    # --------------------------------------------------------------------------
    # 2B. UPMC-Food101 (10 Tasks x 10 Classes — 2-Stage Null-Space LoRA + ASID-CL v2)
    # --------------------------------------------------------------------------
    FOOD_STAGE12_JSON = CKPT_DIR / "full_stage12_food101_results.json"
    if not FOOD_STAGE12_JSON.exists():
        food_root = pathlib.Path("/content/food101")
        csv_files = list(food_root.rglob("*.csv"))
        tr_csv = [p for p in csv_files if "train" in p.name.lower()][0]
        df_f = pd.read_csv(tr_csv, header=None, names=["image", "text", "label"], on_bad_lines="skip")
        if str(df_f.iloc[0]["label"]).lower() in {"label", "category", "class"}:
            df_f = df_f.iloc[1:].reset_index(drop=True)
        img_index = {pathlib.Path(p).name: p for p in glob.glob(str(food_root / "**/train/**/*.jpg"), recursive=True)}
        df_f["path"] = df_f["image"].map(lambda n: img_index.get(pathlib.Path(str(n)).name))
        df_f = df_f.dropna(subset=["path"]).reset_index(drop=True)
        df_f = df_f[df_f["label"] != df_f["label"].value_counts().idxmin()].reset_index(drop=True)
        classes_f = sorted(df_f["label"].unique())[:100]
        df_f = df_f[df_f["label"].isin(classes_f)].reset_index(drop=True)
        df_f["y"] = df_f["label"].map({c: i for i, c in enumerate(classes_f)})

        # Memory-mapped T4 RAM-safe subset for 12-layer backprop (12,000 raw 224x224 images = 120/class)
        rng = np.random.RandomState(0)
        keep_rows = [rng.permutation(np.array(idx))[:120] for _, idx in df_f.groupby("y").groups.items()]
        df_f = df_f.loc[np.concatenate(keep_rows)].reset_index(drop=True)
        df_f["split"] = "train"
        for _, idx in df_f.groupby("y").groups.items():
            idx = np.array(idx)
            n_va = max(1, len(idx) // 10)
            df_f.loc[idx[:n_va], "split"] = "val"
            df_f.loc[idx[n_va : 2 * n_va], "split"] = "test"
        corder = np.random.RandomState(2026).permutation(100)
        df_f["task"] = df_f["y"].map({int(c): k // 10 for k, c in enumerate(corder)})

        print(f"⏳ [UPMC-Food101 2-Stage] Đang giải mã {len(df_f):,} ảnh 224x224 để huấn luyện 12-Layer Null-Space LoRA...")
        D_food_2stage = E.make_data(
            df_f, path_col="path", text_col="text", label_col="y",
            task_col="task", task_order=list(range(10)), num_classes=100, workers=2
        )
        g1_f = E.run_g1(D_food_2stage, seeds=[STAGE1_SEEDS[0]], epochs=2, bs=STAGE1_LORA_BS, lr=STAGE1_LORA_LR, stage="food_g1")
        g2_f = E.run_g2(D_food_2stage, taus=STAGE1_TAUS, seeds=[STAGE1_SEEDS[0]], epochs=2, bs=STAGE1_LORA_BS, lr=STAGE1_LORA_LR, stage="food_g2")
        verdicts_f = {m: E.verdicts(g1_f, g2_f, m) for m in ("macro_f1", "acc", "worst_class_f1")}
        E.export(CKPT_DIR, "full_stage12_food101", D_food_2stage, {"stages": ["food_g1", "food_g2"], "g1": g1_f, "g2": g2_f, "verdicts": verdicts_f})
        print(f"💾 [UPMC-Food101 2-Stage] Đã lưu trọn bộ kết quả vào Google Drive: {FOOD_STAGE12_JSON}")
    else:
        print(f"✅ [UPMC-Food101 2-Stage] Đã có sẵn kết quả trong Google Drive: {FOOD_STAGE12_JSON}")


## 8. Đóng gói `.zip` Toàn bộ Kết quả & Tải về Máy Cá nhân


In [ ]:
import shutil
zip_base = pathlib.Path("/content/ASID_CL_v2_Full_Colab_Results")
shutil.make_archive(str(zip_base), "zip", root_dir=str(SAVE_ROOT))
print(f"✅ Đã đóng gói toàn bộ kết quả tại: {zip_base}.zip (và đã đồng bộ trên Google Drive tại {SAVE_ROOT})")

try:
    from google.colab import files
    files.download(str(zip_base) + ".zip")
except Exception as e:
    print(f"ℹ️ Bạn có thể tải trực tiếp tệp từ Google Drive ({SAVE_ROOT}) hoặc thanh Files bên trái của Colab ({e}).")
